<a href="https://colab.research.google.com/github/Dayan77uu/chatboot/blob/main/Chatbot_Informatica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chatbot Informática-UNSAAC — POO + Motor de Inferencia
## Objetivo

Implementar el chatbot de la EPIIS–UNSAAC aplicando el **paradigma de Programación
Orientada a Objetos**, integrando los componentes de un sistema de gestión del
conocimiento:

1. **Corpus** —  preguntas etiquetadas en 15 intenciones (conocimiento
   conversacional).
2. **Modelo de clasificación** — pipeline **TF-IDF (palabras 1-2 + caracteres 3-5) →
   Random Forest (300 árboles)** para identificar la intención de cada pregunta.
3. **Repositorio de conocimiento** — base de datos relacional embebida (10 tablas:
   cursos, docentes, horarios, trámites, titulación, servicios, tutorías, etc.),
   cargada en SQLite y accedida a través de la clase `BaseDatos`.
4. **Motor de inferencia** — componente TELL · ASK · INFER · EXPLAIN que combina
   hechos independientes para derivar conclusiones que no existen como un dato único
   en la base de datos (elegibilidad para prácticas, bachillerato, modalidad de
   titulación y regularidad académica).

El sistema se organiza en 12 módulos de dominio: 10 consultan la base de datos
directamente (ASK puro), y `GestorElegibilidad` es el único que razona combinando
varios hechos mediante el motor de inferencia.



### Decisión

Se sigue estrictamente el criterio:

> Si la respuesta es **un solo hecho que ya existe en una fila** → ASK puro.
> Si requiere **combinar dos o más hechos independientes** que nunca están juntos en una
> fila para derivar una conclusión nueva → inferencia.

Bajo este criterio, **solo `GestorElegibilidad` consume el motor**. Los demás módulos
son ASK puro contra la BD. No se sobre-ingenia.

### Pipeline de respuesta

Para entender fácil cómo funciona el bot de principio a fin: toda pregunta pasa por
los mismos 4 primeros pasos (limpiar texto → convertirlo en números → adivinar el tema
→ revisar si está seguro). Lo que cambia es el último paso, según el tema detectado:

```
Pregunta
   ↓
Preprocesamiento NLP (_normalizar: minúsculas, sin tildes, sin puntuación)
   ↓
TF-IDF (palabras 1-2 + caracteres 3-5)
   ↓
Random Forest (300 árboles) → intención + confianza
   ↓
¿intención = saludo/despedida? → respuesta fija
¿confianza < umbral (0.20)? → pide reformular
   ↓
módulo POO correspondiente (tabla de despacho)
   ↓
   ├── 10 módulos (Información General, Docentes, Cursos, Horarios,
   │    Trámites, Tesis, Titulación, Ingresantes, Tutorías, Servicios)
   │    → consulta SQL al repositorio (SQLite) → Respuesta
   │
   └── GestorElegibilidad (prácticas, bachillerato, titulación, regularidad)
        → Motor de Inferencia (TELL · ASK · INFER · EXPLAIN)
        → Respuesta + traza explicativa
```


## 1. Instalación de dependencias e importaciones

**En simple:** aquí solo se traen las "herramientas" que se van a usar más adelante, como cuando antes de cocinar sacas las ollas y los ingredientes. No se hace nada todavía, solo se prepara el terreno.

- `re`, `os`, `sqlite3`, `textwrap` → herramientas básicas de Python para manejar texto y bases de datos.
- `nltk` → trae una lista de "palabras vacías" en español (como "el", "la", "de") que no aportan significado y se descartan al analizar las preguntas.
- `sklearn` (scikit-learn) → la librería que convierte texto en números (TF-IDF) y que entrena el modelo que adivina la intención (Random Forest).
- `train_test_split`, `accuracy_score` → sirven para separar los datos en "para entrenar" y "para comprobar qué tan bien aprendió" el modelo.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 1: IMPORTACIONES Y CONFIGURACIÓN INICIAL
# ─────────────────────────────────────────────────────────────────────
# Importa las librerías necesarias para el proyecto:
#   - re, os, sqlite3, textwrap     → manejo de texto, base de datos, utilidades
#   - nltk                          → stopwords en español (filtrado de palabras vacías)
#   - sklearn (TfidfVectorizer,     → vectorización TF-IDF (convierte texto a números)
#     FeatureUnion, RandomForest)   → clasificación con 300 árboles de decisión
#   - train_test_split, accuracy    → partición de datos y métricas de evaluación
# También descarga el recurso de stopwords de NLTK si no está disponible.
# ═══════════════════════════════════════════════════════════════════════
# En Google Colab descomente la siguiente línea la primera vez:
!pip install scikit-learn nltk pandas --quiet

import re, os, sqlite3, textwrap
from typing import Optional, List, Tuple

import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

try:
    nltk.download("stopwords", quiet=True)
except Exception as e:
    print("Aviso NLTK:", e)

print("Dependencias listas.")


Dependencias listas.


## 2. Repositorio de conocimiento (BD)

El volcado SQL embebido contiene las tablas originales más las tres nuevas. Se ejecuta
sobre SQLite (in-memory) para que el notebook corra en Colab sin servidor; el diseño es
portable a MySQL gracias a la clase `BaseDatos`.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 2: VOLCADO SQL EMBEBIDO (REPOSITORIO DE CONOCIMIENTO)
# ─────────────────────────────────────────────────────────────────────
# Contiene el volcado SQL completo de la base de datos de la Escuela
# Profesional, embebido como una cadena de texto dentro del notebook.
# Incluye 10 tablas:
#   - cursos (plan curricular 2017, con prereq_creditos corregido)
#   - docentes (nombre y dedicación)
#   - horarios_cursos (día, hora, aula, docente por sección)
#   - informacion_general (misión, visión, historia, directora, etc.)
#   - ingresantes (estadísticas de admisión 2025)
#   - temas_tesis (áreas de investigación)
#   - tramites (42 trámites con costos TUPA exactos)
#   - titulacion_modalidades (bachiller, tesis, suficiencia, servicios)
#   - servicios_universitarios (salud, becas, deportes, movilidad, etc.)
#   - tutorias_info (proceso, tipos, horarios, tutor, problemas)
# El volcado se usa en la celda siguiente para crear la BD en memoria.
# ═══════════════════════════════════════════════════════════════════════
SQL_DUMP = r"""
-- =====================================================================
-- escuela_informatica_v2.sql
-- Versión ampliada con la información del PDF Base_Conocimiento + enlaces
-- =====================================================================

-- ---------- TABLAS QUE SE MANTIENEN (datos originales) ----------

CREATE TABLE cursos (
  codigo VARCHAR(20) PRIMARY KEY,
  nombre VARCHAR(255) NOT NULL,
  creditos INT,
  categoria VARCHAR(10),
  semestre INT,
  curricula VARCHAR(255),
  prerrequisitos VARCHAR(255),
  prereq_creditos INT DEFAULT 0
);

-- ============================
-- PRIMER SEMESTRE (Estudios Generales)
-- ============================
INSERT INTO cursos VALUES
('LC901','REDACCIÓN DE TEXTOS',4,'EGT',1,'PLAN CURRICULAR 2017','',0),
('ME901','MATEMATICA I',4,'EG',1,'PLAN CURRICULAR 2017','',0),
('ED901','ESTRATEGIAS DE APRENDIZAJE AUTÓNOMO',4,'EG',1,'PLAN CURRICULAR 2017','',0),
('FP901','FILOSOFÍA Y ÉTICA',3,'EG',1,'PLAN CURRICULAR 2017','',0),
('AS901','SOCIEDAD Y CULTURA',3,'EG',1,'PLAN CURRICULAR 2017','',0),
('DR901','CONSTITUCIÓN POLITICA Y DERECHOS HUMANOS',3,'EG',1,'PLAN CURRICULAR 2017','',0);

-- ============================
-- SEGUNDO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF468','FUNDAMENTOS DE PROGRAMACION',4,'OEFE',2,'PLAN CURRICULAR 2017','',0),
('ME307','MATEMATICAS DISCRETAS I',4,'OEFE',2,'PLAN CURRICULAR 2017','',0),
('ME903','CALCULO I',4,'EGT',2,'PLAN CURRICULAR 2017','ME901',0),
('FI902','FISICA I',4,'EGT',2,'PLAN CURRICULAR 2017','',0),
('IF902','TECNOLOGIA DE LA INFORMACION Y COMUNICACIÓN',3,'EG',2,'PLAN CURRICULAR 2017','',15),
('FP902','LIDERAZGO Y HABILIDADES SOCIALES',3,'EG',2,'PLAN CURRICULAR 2017','',0);

-- ============================
-- TERCER SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF450','ABSTRACCION DE DATOS Y OBJETOS',4,'OEFE',3,'PLAN CURRICULAR 2017','IF468',0),
('FI370','FISICA APLICADA A LA INFORMATICA',4,'OEFE',3,'PLAN CURRICULAR 2017','FI902',0),
('ME350','CALCULO II',4,'OEFE',3,'PLAN CURRICULAR 2017','ME903',0),
('ME351','ALGEBRA LINEAL COMPUTACIONAL',4,'OEFE',3,'PLAN CURRICULAR 2017','ME307 y IF468',0),
('ME352','PROBABILIDADES Y ESTADISTICA',4,'OEFE',3,'PLAN CURRICULAR 2017','ME903',0),
('IF451','PROGRAMACION I',2,'OEFE',3,'PLAN CURRICULAR 2017','IF468',0);

-- ============================
-- CUARTO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF452','ALGORITMOS Y ESTRUCTURA DE DATOS',4,'OEFE',4,'PLAN CURRICULAR 2017','IF450',0),
('LI371','ELECTRONICA Y DISEÑO DIGITAL',3,'OEFE',4,'PLAN CURRICULAR 2017','FI370',0),
('IF480','ADMINISTRACIÓN DE TECNOLOGIAS DE INFORMACION',3,'OEFE',4,'PLAN CURRICULAR 2017','ME352',0),
('ME354','INVESTIGACION OPERATIVA',4,'OEFE',4,'PLAN CURRICULAR 2017','ME351',0),
('ME355','MATEMATICAS DISCRETAS II',4,'OEFE',4,'PLAN CURRICULAR 2017','ME307',0),
('IF453','PROGRAMACION II',2,'OEFE',4,'PLAN CURRICULAR 2017','IF451',0),
('IF060','MUSICA',2,'EC',4,'PLAN CURRICULAR 2017','',60),
('IF061','CORO',2,'EC',4,'PLAN CURRICULAR 2017','',60),
('IF062','DANZA MODERNA',2,'EC',4,'PLAN CURRICULAR 2017','',100),
('IF063','QUECHUA',2,'EC',4,'PLAN CURRICULAR 2017','',100);

-- ============================
-- QUINTO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF650','MODELOS PROBABILISTICOS',4,'OEFE',5,'PLAN CURRICULAR 2017','ME352',80),
('IF550','ORGANIZACIÓN Y ARQUITECTURA DEL COMPUTADOR',4,'OEFE',5,'PLAN CURRICULAR 2017','LI371',0),
('IF454','TEORIA DE LA COMPUTACION',3,'OEFE',5,'PLAN CURRICULAR 2017','IF452 y ME355',0),
('IF610','ANALISIS Y DISEÑO DE SISTEMAS DE INFORMACIÓN',4,'OEFE',5,'PLAN CURRICULAR 2017','IF450 y IF480',0),
('IF481','INGENIERIA ECONOMICA',3,'OEFE',5,'PLAN CURRICULAR 2017','IF480',0),
('ME356','ECUACIONES DIFERENCIALES',4,'OEFE',5,'PLAN CURRICULAR 2017','ME350',0);

-- ============================
-- SEXTO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF455','ALGORITMOS PARALELOS Y DISTRIBUIDOS',4,'OEFE',6,'PLAN CURRICULAR 2017','IF452',0),
('IF611','METODOLOGIA DE DESARROLLO DE SOFTWARE',3,'OEFE',6,'PLAN CURRICULAR 2017','IF610',0),
('IF612','FUNDAMENTOS Y DISEÑO DE BASES DE DATOS',4,'OEFE',6,'PLAN CURRICULAR 2017','IF610',0),
('IF551','SISTEMAS OPERATIVOS',4,'OEFE',6,'PLAN CURRICULAR 2017','IF550',0),
('IF457','METODOS NUMERICOS',3,'OEFE',6,'PLAN CURRICULAR 2017','ME356',0),
('ELEC-06-1','ELECTIVA DE ESPECIALIDAD',4,'EEFE',6,'PLAN CURRICULAR 2017','',0);

-- ============================
-- SÉPTIMO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF456','ALGORITMOS AVANZADOS',4,'OEFE',7,'PLAN CURRICULAR 2017','IF454',0),
('IF651','INTELIGENCIA ARTIFICIAL',3,'OEFE',7,'PLAN CURRICULAR 2017','IF650 y IF612',0),
('IF613','DESARROLLO DE SOFTWARE I',2,'OEFE',7,'PLAN CURRICULAR 2017','IF612',0),
('IF552','REDES DE COMPUTADORAS I',4,'OEFE',7,'PLAN CURRICULAR 2017','IF551',0),
('ELEC-07-1','ELECTIVA DE ESPECIALIDAD',3,'EEFE',7,'PLAN CURRICULAR 2017','',0),
('ELEC-07-2','ELECTIVA DE ESPECIALIDAD',3,'EEFE',7,'PLAN CURRICULAR 2017','',0),
('IF-EC-07','ACTIVIDADES EXTRACURRICULARES',2,'EC',7,'PLAN CURRICULAR 2017','',170);

-- ============================
-- OCTAVO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF710','SEMINARIO DE INVESTIGACION I',3,'OEFE',8,'PLAN CURRICULAR 2017','',150),
('IF652','APRENDIZAJE AUTOMATICO',4,'OEFE',8,'PLAN CURRICULAR 2017','IF651',0),
('IF614','INGENIERIA DE SOFTWARE I',4,'OEFE',8,'PLAN CURRICULAR 2017','IF611 y IF613',0),
('IF482','PLANEAMIENTO Y DIRECCIÓN ESTRATEGICA DE TECNOLOGIA DE INFORMACIÓN',3,'OEFE',8,'PLAN CURRICULAR 2017','',140),
('ELEC-08-1','ELECTIVA DE ESPECIALIDAD',4,'EEFE',8,'PLAN CURRICULAR 2017','',0),
('ELEC-08-2','ELECTIVA DE ESPECIALIDAD',4,'EEFE',8,'PLAN CURRICULAR 2017','',0);

-- ============================
-- NOVENO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF483','FORMULACION DE PROYECTOS DE TECNOLOGIA DE INFORMACIÓN',3,'OEFE',9,'PLAN CURRICULAR 2017','IF482',0),
('ELEC-09-1','ELECTIVA DE ESPECIALIDAD',3,'EEFE',9,'PLAN CURRICULAR 2017','',0),
('ELEC-09-2','ELECTIVA DE ESPECIALIDAD',3,'EEFE',9,'PLAN CURRICULAR 2017','',0),
('ELEC-09-3','ELECTIVA DE ESPECIALIDAD',4,'EEFE',9,'PLAN CURRICULAR 2017','',0),
('ELEC-09-4','ELECTIVA DE ESPECIALIDAD',3,'EEFE',9,'PLAN CURRICULAR 2017','',0),
('ELEC-09-5','ELECTIVA DE ESPECIALIDAD',4,'EEFE',9,'PLAN CURRICULAR 2017','',0),
('IF-EC-09','ACTIVIDADES EXTRACURRICULARES',2,'EC',9,'PLAN CURRICULAR 2017','',0);

-- ============================
-- DÉCIMO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IF711','SEMINARIO DE INVESTIGACION II',3,'OEFE',10,'PLAN CURRICULAR 2017','IF710',0),
('ELEC-10-1','ELECTIVA DE ESPECIALIDAD',4,'EEFE',10,'PLAN CURRICULAR 2017','',0),
('ELEC-10-2','ELECTIVA DE ESPECIALIDAD',3,'EEFE',10,'PLAN CURRICULAR 2017','',0),
('ELEC-10-3','ELECTIVA DE ESPECIALIDAD',3,'EEFE',10,'PLAN CURRICULAR 2017','',0),
('IF020','PRACTICAS PRE-PROFESIONALES',9,'PPP',10,'PLAN CURRICULAR 2017','',200);

-- ============================
-- CURSOS DE ESPECIALIDAD (electivas reales con código, no asignadas a semestre fijo)
-- Disponibles para elegir en los slots EEFE de 6° a 10° semestre
-- ============================
INSERT INTO cursos VALUES
('ME357','MATEMATICAS AVANZADAS PARA INGENIERIA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','ME350',0),
('ME358','ESTADISTICA INFERENCIAL',4,'EEFE',NULL,'PLAN CURRICULAR 2017','ME352',0),
('ME359','ESTADISTICA APLICADA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','ME352',0),
('IF458','COMPUTACIÓN GRÁFICA I',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF452 y IF453',0),
('IF459','COMPUTACIÓN GRÁFICA II',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF458',0),
('IF460','COMPUTACION CUANTICA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF456',0),
('IF464','COMPUTACION SIMBOLICA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF466',0),
('IF465','GEOMETRIA COMPUTACIONAL',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF458',0),
('IF466','COMPILADORES',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF454',0),
('IF467','ANALISIS Y DISEÑO DE ALGORITMOS',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF452 y IF453',0),
('IF484','EMPRENDIMIENTO E INNOVACION',3,'EEFE',NULL,'PLAN CURRICULAR 2017','IF482',0),
('IF485','CONTROL Y AUDITORIA DE SISTEMAS',3,'EEFE',NULL,'PLAN CURRICULAR 2017','IF483',0),
('IF553','LENGUAJE ENSAMBLADOR',3,'EEFE',NULL,'PLAN CURRICULAR 2017','IF550',0),
('IF554','REDES DE COMPUTADORAS II',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF552',0),
('IF555','TOPICOS AVANZADOS EN REDES',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF554',0),
('IF556','SISTEMAS EMBEBIDOS',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF551',0),
('IF557','ARQUITECTURAS DE ALTO RENDIMIENTO',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF554',0),
('IF616','DESARROLLO DE SOFTWARE II',2,'EEFE',NULL,'PLAN CURRICULAR 2017','IF614',0),
('IF617','INGENIERIA DE SOFTWARE II',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF614',0),
('IF618','TOPICOS AVANZADOS EN INGENIERIA DE SOFTWARE',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF614',0),
('IF619','ANALISIS DE DATOS EMPRESARIALES',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF612 y ME352',0),
('IF653','MINERIA DE DATOS',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF652',0),
('IF654','ROBOTICA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF651',0),
('IF655','PROCESAMIENTO DIGITAL DE SEÑALES',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF651',0),
('IF656','PROCESAMIENTO DEL LENGUAJE NATURAL',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF651',0),
('IF657','VISION COMPUTACIONAL',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF651',0),
('IF658','REDES NEURONALES ARTIFICIALES',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF652',0),
('IF659','REDES BAYESIANAS',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF652',0),
('IF661','BUSQUEDA HEURISTICA Y METAHEURISTICA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF651',0),
('IF662','DEEP LEARNING',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF652',0),
('IF663','INTERACCION PERSONA COMPUTADOR',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF651',0),
('IF664','BIOINFORMATICA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF650',0),
('IF665','MODELACION Y SIMULACION DE ESTRUCTURAS BIOMOLECULARES',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF664',0),
('IF666','ESTADISTICA PARA BIOINFORMATICA Y DISEÑO EXPERIMENTAL',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF664',0),
('IF667','ANALISIS DE IMÁGENES BIOMEDICAS EN 2D Y 3D',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF664',0),
('IF668','ANALISIS DE DATOS DE EXPRESION GENICA',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF664',0),
('IF669','MODELADO Y SIMULACION',4,'EEFE',NULL,'PLAN CURRICULAR 2017','IF650',0);

-- ============================
-- ACTIVIDADES EXTRACURRICULARES (catálogo completo, no asignadas a semestre fijo)
-- ============================
INSERT INTO cursos VALUES
('IF064','TALLER DE DEBATE',2,'EC',NULL,'PLAN CURRICULAR 2017','',170),
('IF065','TALLER DE CREATIVIDAD MUSICAL',2,'EC',NULL,'PLAN CURRICULAR 2017','',170),
('IF066','TEATRO',2,'EC',NULL,'PLAN CURRICULAR 2017','',170);

CREATE TABLE docentes (
  id INT PRIMARY KEY,
  nombre VARCHAR(255) NOT NULL,
  dedicacion VARCHAR(40) NOT NULL
);

-- ============================
-- PLAN CURRICULAR 2025 - PRIMER SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('QUG01','QUÍMICA GENERAL',4,'EG',1,'PLAN CURRICULAR 2025','',0),
('MEG01','ALGEBRA Y GEOMETRÍA ANALÍTICA',4,'EG',1,'PLAN CURRICULAR 2025','',0),
('HIG01','HISTORIA CRÍTICA DEL PERÚ E IDENTIDAD NACIONAL',3,'EG',1,'PLAN CURRICULAR 2025','',0),
('FIG01','FÍSICA I',4,'EG',1,'PLAN CURRICULAR 2025','',0),
('CBG01','ECOLOGÍA Y MEDIO AMBIENTE',3,'EG',1,'PLAN CURRICULAR 2025','',0),
('MEG02','CÁLCULO I',4,'EG',1,'PLAN CURRICULAR 2025','',0);

-- ============================
-- SEGUNDO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('LCG01','LINGÜÍSTICA Y COMUNICACIÓN HUMANA',4,'EG',2,'PLAN CURRICULAR 2025','',0),
('MEG03','ESTADÍSTICA GENERAL',4,'EG',2,'PLAN CURRICULAR 2025','',0),
('IFG01','PENSAMIENTO COMPUTACIONAL E INTELIGENCIA ARTIFICIAL',3,'EG',2,'PLAN CURRICULAR 2025','',0),
('MEG04','CÁLCULO II',4,'EG',2,'PLAN CURRICULAR 2025','MEG02',0),
('IFI01','FUNDAMENTOS DE PROGRAMACIÓN',3,'OEFE',2,'PLAN CURRICULAR 2025','',0),
('MEI04','MATEMÁTICAS DISCRETAS I',4,'OEFE',2,'PLAN CURRICULAR 2025','MEG01',0);

-- ============================
-- TERCER SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI02','ALGORITMOS Y ESTRUCTURAS DE DATOS I',4,'OEFE',3,'PLAN CURRICULAR 2025','IFI01 y IFG01',0),
('IFI03','PROGRAMACIÓN I',2,'OEFE',3,'PLAN CURRICULAR 2025','IFI01',0),
('MEI05','ALGEBRA LINEAL COMPUTACIONAL',4,'OEFE',3,'PLAN CURRICULAR 2025','MEG01',0),
('FII02','FÍSICA APLICADA A LA INFORMÁTICA',4,'OEFE',3,'PLAN CURRICULAR 2025','FIG01',0),
('MEI06','MATEMÁTICA DISCRETAS II',4,'OEFE',3,'PLAN CURRICULAR 2025','MEI04',0),
('MEI09','CALCULO III',4,'OEFE',3,'PLAN CURRICULAR 2025','MEG04',0);

-- ============================
-- CUARTO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI04','ALGORITMOS Y ESTRUCTURAS DE DATOS II',4,'OEFE',4,'PLAN CURRICULAR 2025','IFI02 y IFI03',0),
('IFI05','PROGRAMACIÓN II',2,'OEFE',4,'PLAN CURRICULAR 2025','IFI02 y IFI03',0),
('ADI01','ORGANIZACIÓN Y DIRECCIÓN DE EMPRESAS',4,'OEFE',4,'PLAN CURRICULAR 2025','',60),
('ELI01','ELECTRÓNICA Y DISEÑO DIGITAL',4,'OEFE',4,'PLAN CURRICULAR 2025','FII02',0),
('IFI07','MODELOS PROBABILÍSTICOS',4,'OEFE',4,'PLAN CURRICULAR 2025','MEI06 y MEG03',0),
('MEI08','ECUACIONES DIFERENCIALES',4,'OEFE',4,'PLAN CURRICULAR 2025','MEI09',0);

-- ============================
-- QUINTO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI08','ANÁLISIS Y DISEÑO DE ALGORITMOS',4,'OEFE',5,'PLAN CURRICULAR 2025','IFI04',0),
('IFI09','PROGRAMACIÓN III',2,'OEFE',5,'PLAN CURRICULAR 2025','IFI05',0),
('IFI10','ANÁLISIS Y DISEÑO DE SISTEMAS DE INFORMACIÓN',4,'OEFE',5,'PLAN CURRICULAR 2025','IFI04 y ADI01',0),
('IFI11','ORGANIZACIÓN Y ARQUITECTURA DEL COMPUTADOR',4,'OEFE',5,'PLAN CURRICULAR 2025','ELI01',0),
('IFI12','MÉTODOS NUMÉRICOS',4,'OEFE',5,'PLAN CURRICULAR 2025','MEI08',0),
('IFI13','TEORÍA DE LA COMPUTACIÓN',4,'OEFE',5,'PLAN CURRICULAR 2025','IFI04',0);

-- ============================
-- SEXTO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI14','INTELIGENCIA ARTIFICIAL',4,'EEFE',6,'PLAN CURRICULAR 2025','MEI05',0),
('IFI15','FUNDAMENTOS Y DISEÑO DE BASE DE DATOS',4,'EEFE',6,'PLAN CURRICULAR 2025','IFI10',0),
('IFI16','SISTEMAS OPERATIVOS',4,'EEFE',6,'PLAN CURRICULAR 2025','IFI11',0),
('IFI17','MODELADO Y SIMULACIÓN',4,'EEFE',6,'PLAN CURRICULAR 2025','IFI07 y IFI12',0),
('EUI01','DEPORTE',2,'EC',6,'PLAN CURRICULAR 2025','',100),
('ELEC25-06-1','ELECTIVO DE ESPECIALIDAD',4,'EEFE',6,'PLAN CURRICULAR 2025','',0);

-- ============================
-- SÉPTIMO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI19','APRENDIZAJE AUTOMÁTICO',4,'EEFE',7,'PLAN CURRICULAR 2025','IFI14',0),
('IFI20','METODOLOGÍA Y DESARROLLO DE SOFTWARE',5,'EEFE',7,'PLAN CURRICULAR 2025','IFI15',0),
('IFI21','REDES DE COMPUTADORAS I',4,'EEFE',7,'PLAN CURRICULAR 2025','IFI16',0),
('IFI22','CIENCIA DE DATOS',3,'EEFE',7,'PLAN CURRICULAR 2025','IFI14',0),
('IFI23','PROYECCIÓN SOCIAL',2,'EC',7,'PLAN CURRICULAR 2025','',120),
('ELEC25-07-1','ELECTIVO DE ESPECIALIDAD',4,'EEFE',7,'PLAN CURRICULAR 2025','',0);

-- ============================
-- OCTAVO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI24','PROGRAMACIÓN PARALELA Y DISTRIBUIDA',4,'EEFE',8,'PLAN CURRICULAR 2025','IFI21 y IFI08',0),
('IFI25','APRENDIZAJE PROFUNDO',4,'EEFE',8,'PLAN CURRICULAR 2025','IFI19',0),
('IFI26','INGENIERÍA DE SOFTWARE I',4,'EEFE',8,'PLAN CURRICULAR 2025','IFI20',0),
('IFI27','REDES DE COMPUTADORAS II',4,'EEFE',8,'PLAN CURRICULAR 2025','IFI21',0),
('IFI28','FORMULACIÓN DE PROYECTOS DE TECNOLOGÍAS DE LA INFORMACIÓN',3,'EEFE',8,'PLAN CURRICULAR 2025','IFI20',0),
('ELEC25-08-1','ELECTIVO DE ESPECIALIDAD',3,'EEFE',8,'PLAN CURRICULAR 2025','',0);

-- ============================
-- NOVENO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI29','SEMINARIO DE INVESTIGACIÓN',3,'EEFE',9,'PLAN CURRICULAR 2025','',170),
('IFI30','INGENIERÍA DE SOFTWARE II',4,'EEFE',9,'PLAN CURRICULAR 2025','IFI26',0),
('IFI31','SISTEMAS INTERNET DE LA COSAS',4,'EEFE',9,'PLAN CURRICULAR 2025','IFI27 y IFI19',0),
('IFI32','CIBERSEGURIDAD',4,'EEFE',9,'PLAN CURRICULAR 2025','IFI27',0),
('ELEC25-09-1','ELECTIVO DE ESPECIALIDAD',4,'EEFE',9,'PLAN CURRICULAR 2025','',0),
('ELEC25-09-2','ELECTIVO DE ESPECIALIDAD',3,'EEFE',9,'PLAN CURRICULAR 2025','',0);

-- ============================
-- DÉCIMO SEMESTRE
-- ============================
INSERT INTO cursos VALUES
('IFI33','TRABAJO DE INVESTIGACIÓN',4,'EEFE',10,'PLAN CURRICULAR 2025','IFI29',0),
('IFI34','DESARROLLO DE PROYECTOS DE INGENIERÍA DE SOFTWARE',4,'EEFE',10,'PLAN CURRICULAR 2025','IFI28 y IFI30',0),
('IFI35','ADMINISTRACIÓN DE TECNOLOGÍAS DE LA INFORMACIÓN',4,'EEFE',10,'PLAN CURRICULAR 2025','IFI32',0),
('IFI18','DEBATE',2,'EC',10,'PLAN CURRICULAR 2025','',180),
('IFI66','PRÁCTICA PRE PROFESIONALES',8,'PPP',10,'PLAN CURRICULAR 2025','',180);

INSERT INTO `docentes` (`id`, `nombre`, `dedicacion`) VALUES
(1, 'Mgt. Lino Prisciliano Flores Pacheco', 'Tiempo Completo'),
(2, 'Dr. Edwin Carrasco Poblete', 'Tiempo Completo'),
(3, 'Dr. Emilio Palomino Olivera', 'Tiempo Completo'),
(4, 'Dr. Rony Villafuerte Serna', 'Tiempo Completo'),
(5, 'Dr. Luis Beltrán Palma Ttito', 'Tiempo Completo'),
(6, 'Dr. Dennis Iván Candia Oviedo', 'Tiempo Completo'),
(7, 'Dr. Lauro Enciso Rodas', 'Dedicación Exclusiva'),
(8, 'Mgt. Julio César Carbajal Luna', 'Dedicación Exclusiva'),
(9, 'Dr. Javier Arturo Rozas Huacho', 'Dedicación Exclusiva'),
(10, 'Dra. Nila Zonia Acurio Usca', 'Dedicación Exclusiva'),
(11, 'Dr. Robert Wilbert Alzamora Paredes', 'Dedicación Exclusiva'),
(12, 'Dra. Yeshica Isela Ormeño Ayala', 'Dedicación Exclusiva'),
(13, 'Mgt. Waldo Elio Ibarra Zambrano', 'Tiempo Completo'),
(14, 'Mgt. Vanessa Maribel Choque Soto', 'Tiempo Completo'),
(15, 'Mgt. Karelia Medina Miranda', 'Tiempo Completo'),
(16, 'Mgt. Javier David Chávez Centeno', 'Dedicación Exclusiva'),
(17, 'Mgt. Guzmán Ticona Pari', 'Dedicación Exclusiva'),
(18, 'Mgt. Iván César Medrano Valencia', 'Tiempo Completo'),
(19, 'Ing. Lino Aquiles Baca Cárdenas', 'Tiempo Completo'),
(20, 'Lic. Esther Pacheco Vásquez', 'Tiempo Completo'),
(21, 'Mgt. José Mauro Pillco Quispe', 'Tiempo Completo'),
(22, 'Mgt. Efraina Gladys Cutipa Arapa', 'Tiempo Completo'),
(23, 'Mgt. Maritza Katherine Irpanocca Cusimayta', 'Tiempo Completo'),
(24, 'Mgt. Harley Vera Olivera', 'Tiempo Completo'),
(25, 'Mgt. Willian Zamalloa Paro', 'Tiempo Completo'),
(26, 'Mgt. Doris Sabina Aguirre Carbajal', 'Tiempo Completo'),
(27, 'Dr. Dario Francisco Dueñas Bustinza', 'Tiempo Completo'),
(28, 'Mgt. Tany Villalba Villalba', 'Tiempo Completo'),
(29, 'Mgt. Jisbaj Gamarra Salas', 'Tiempo Completo'),
(30, 'Mgt. Henry Samuel Dueñas De La Cruz', 'Tiempo Completo'),
(31, 'Ing. Manuel Aurelio Peñaloza Figueroa', 'Tiempo Parcial 20 hrs'),
(32, 'Ing. Marcio Fernando Merma Quispe', 'Jefe de Prácticas - Tiempo Completo'),
(33, 'Ing. Elmer Cama Cáceres', 'Jefe de Prácticas - Tiempo Completo');

CREATE TABLE horarios_cursos (
  id INT PRIMARY KEY,
  codigo_curso VARCHAR(15),
  seccion VARCHAR(2),
  docente VARCHAR(255),
  dia VARCHAR(2),
  hora_inicio TIME,
  hora_fin TIME,
  tipo VARCHAR(2),
  grupo VARCHAR(2),
  aula VARCHAR(20),
  horas INT,
  FOREIGN KEY (codigo_curso) REFERENCES cursos(codigo)
);

INSERT INTO horarios_cursos VALUES
(1,'MEG01','A','HERRERA-VARGAS-MARCO ANTONIO','MA','09:00:00','11:00:00','T','A','IN101',2),
(2,'MEG01','A','HERRERA-VARGAS-MARCO ANTONIO','JU','09:00:00','11:00:00','P','A','IN101',2),
(3,'MEG01','A','HERRERA-VARGAS-MARCO ANTONIO','VI','10:00:00','11:00:00','T','A','IN101',1),
(4,'MEG01','B','CJUNO-ROQUE-TIMOTEO','MA','09:00:00','11:00:00','T','B','IN102',2),
(5,'MEG01','B','CJUNO-ROQUE-TIMOTEO','JU','09:00:00','11:00:00','P','B','IN102',2),
(6,'MEG01','B','CJUNO-ROQUE-TIMOTEO','VI','10:00:00','11:00:00','T','B','IN102',1),
(7,'MEG02','A','VILLAVICENCIO-SUNA-PERCY MARCO','LU','11:00:00','13:00:00','T','A','IN101',2),
(8,'MEG02','A','VILLAVICENCIO-SUNA-PERCY MARCO','MI','11:00:00','13:00:00','P','A','IN101',2),
(9,'MEG02','A','VILLAVICENCIO-SUNA-PERCY MARCO','VI','11:00:00','12:00:00','T','A','IN101',1),
(10,'MEG02','B','VIZCARDO-TORRES-TALIA','LU','11:00:00','13:00:00','T','B','IN102',2),
(11,'MEG02','B','VIZCARDO-TORRES-TALIA','VI','11:00:00','12:00:00','T','B','IN102',1),
(12,'MEG02','B','TAPARA-CACERES-MARIA SOLEDAD','MI','11:00:00','13:00:00','P','B','IN102',2),
(13,'CBG01','A','CANO-CORDOVA-YOVANA','LU','16:00:00','18:00:00','L','B','LAB CB 02',2),
(14,'CBG01','A','CANO-CORDOVA-YOVANA','MA','11:00:00','13:00:00','T','A','IN101',2),
(15,'CBG01','A','CANO-CORDOVA-YOVANA','JU','11:00:00','13:00:00','P','A','IN101',2),
(16,'CBG01','B','TACO-PALMA-PERCY','MA','11:00:00','13:00:00','T','B','IN102',2),
(17,'CBG01','B','RIVERA-GARCÍA-ANTONY ANDREE','MA','14:00:00','16:00:00','L','B','LAB CB 03',2),
(18,'CBG01','B','VARGAS-SERRANO-KARINA','MI','15:00:00','17:00:00','L','B','LAB CB 02',2),
(19,'FIG01','A','ARTEAGA-CURIE-ROCIO','LU','09:00:00','11:00:00','T','A','IN101',2),
(20,'FIG01','A','ARTEAGA-CURIE-ROCIO','MI','09:00:00','11:00:00','P','A','IN101',2),
(21,'FIG01','A','ARTEAGA-CURIE-ROCIO','VI','09:00:00','10:00:00','T','A','IN101',1),
(22,'FIG01','A','CARHUARUPAY-MOLLEDA-IVANICH IGOR','MA','17:00:00','19:00:00','L','A','Lab. F1',2),
(23,'FIG01','A','CAMAN-ALTAMIRANO-JORGE','MI','13:00:00','15:00:00','L','A','Lab. F1',2),
(24,'FIG01','A','CAMAN-ALTAMIRANO-JORGE','JU','13:00:00','15:00:00','L','A','Lab. F1',2),
(25,'FIG01','A','QUISPITUPA-YUPA-MARITZA','VI','15:00:00','17:00:00','L','A','Lab. F1',2),
(26,'FIG01','B','ATAU-ENRIQUEZ-EDILBERTO','LU','09:00:00','11:00:00','T','B','IN102',2),
(27,'FIG01','B','ATAU-ENRIQUEZ-EDILBERTO','MI','09:00:00','11:00:00','P','B','IN102',2),
(28,'FIG01','B','ATAU-ENRIQUEZ-EDILBERTO','VI','09:00:00','10:00:00','T','B','IN102',1),
(29,'HIG01','A','PEÑA-FLORES-EDWAR','LU','07:00:00','09:00:00','T','A','IN101',2),
(30,'HIG01','A','PEÑA-FLORES-EDWAR','MI','07:00:00','09:00:00','P','A','IN101',2),
(31,'HIG01','B','CUSICUNA-VILCA-LISBETH JUDIT','LU','07:00:00','09:00:00','T','B','IN102',2),
(32,'HIG01','B','CUSICUNA-VILCA-LISBETH JUDIT','MI','07:00:00','09:00:00','P','B','IN102',2),
(33,'QUG01','A','OLARTE-PEREZ-AMANDA','MA','07:00:00','09:00:00','T','A','IN101',2),
(34,'QUG01','A','OLARTE-PEREZ-AMANDA','JU','07:00:00','09:00:00','P','A','IN101',2),
(35,'QUG01','A','OLARTE-PEREZ-AMANDA','VI','08:00:00','09:00:00','T','A','IN101',1),
(36,'QUG01','B','LA TORRE-RIVEROS-LYDA','MA','07:00:00','09:00:00','T','B','IN102',2),
(37,'QUG01','B','LA TORRE-RIVEROS-LYDA','JU','07:00:00','09:00:00','P','B','IN102',2),
(38,'QUG01','B','LA TORRE-RIVEROS-LYDA','VI','08:00:00','09:00:00','T','B','IN102',1),
(39,'MEG04','A','VERA-MALDONADO-LEOPOLDO','LU','14:00:00','16:00:00','T','A','IN101',2),
(40,'MEG04','A','VERA-MALDONADO-LEOPOLDO','MI','14:00:00','16:00:00','P','A','IN101',2),
(41,'MEG04','A','VERA-MALDONADO-LEOPOLDO','VI','14:00:00','15:00:00','T','A','IN101',1),
(42,'MEG04','B','MAGUIÑA-RONDAN-SIMEONA ESTILISTA','LU','14:00:00','16:00:00','T','B','IN102',2),
(43,'MEG04','B','MAGUIÑA-RONDAN-SIMEONA ESTILISTA','MI','14:00:00','16:00:00','P','B','IN102',2),
(44,'MEG04','B','MAGUIÑA-RONDAN-SIMEONA ESTILISTA','VI','14:00:00','15:00:00','T','B','IN102',1),
(45,'MEG03','A','SOCOALAYA-MONTALVO-VANESSA','LU','18:00:00','20:00:00','T','A','IN101',2),
(46,'MEG03','A','SOCOALAYA-MONTALVO-VANESSA','MI','18:00:00','20:00:00','P','A','IN101',2),
(47,'MEG03','A','SOCOALAYA-MONTALVO-VANESSA','VI','18:00:00','19:00:00','T','A','IN101',1),
(48,'MEG03','B','TINTAYA-QUISPE-JOSE PERCY','LU','18:00:00','20:00:00','T','B','IN102',2),
(49,'MEG03','B','TINTAYA-QUISPE-JOSE PERCY','MI','18:00:00','20:00:00','P','B','IN102',2),
(50,'MEG03','B','TINTAYA-QUISPE-JOSE PERCY','VI','18:00:00','19:00:00','T','B','IN102',1),
(51,'IFI01','A','MEDINA-MIRANDA-KARELIA','LU','11:00:00','13:00:00','L','A','IN304',2),
(52,'IFI01','A','MEDINA-MIRANDA-KARELIA','MI','11:00:00','13:00:00','T','A','IN202',2),
(53,'IFI01','A','ZAMALLOA-PARO-WILLIAN','LU','11:00:00','13:00:00','L','A','IN312',2),
(54,'IFI01','B','IBARRA-ZANBRANO-WALDO ELIO','LU','11:00:00','13:00:00','L','A','IN305',2),
(55,'IFI01','B','IBARRA-ZANBRANO-WALDO ELIO','MI','11:00:00','13:00:00','T','B','IN203',2),
(56,'LCG01','A','LOPEZ-HUARANCCA-ISABEL','MA','18:00:00','20:00:00','T','A','IN101',2),
(57,'LCG01','A','LOPEZ-HUARANCCA-ISABEL','JU','18:00:00','20:00:00','P','A','IN101',2),
(58,'LCG01','A','LOPEZ-HUARANCCA-ISABEL','VI','19:00:00','20:00:00','T','A','IN101',1),
(59,'LCG01','B','CARAZAS-GAMARRA-WASHINGTON','MA','18:00:00','20:00:00','T','B','IN102',2),
(60,'LCG01','B','CARAZAS-GAMARRA-WASHINGTON','JU','18:00:00','20:00:00','P','B','IN102',2),
(61,'LCG01','B','CARAZAS-GAMARRA-WASHINGTON','VI','19:00:00','20:00:00','T','B','IN102',1),
(62,'MEI04','A','CAZORLA-MEDINA-EDWIN','LU','16:00:00','18:00:00','T','A','IN208',2),
(63,'MEI04','A','CAZORLA-MEDINA-EDWIN','MI','16:00:00','18:00:00','P','A','IN208',2),
(64,'MEI04','A','CAZORLA-MEDINA-EDWIN','VI','17:00:00','18:00:00','T','A','IN208',1),
(65,'MEI04','B','LOAIZA-HUAMAN-PAUL HERBERT','LU','16:00:00','18:00:00','T','B','IN101',2),
(66,'MEI04','B','LOAIZA-HUAMAN-PAUL HERBERT','MI','16:00:00','18:00:00','P','B','IN101',2),
(67,'MEI04','B','LOAIZA-HUAMAN-PAUL HERBERT','VI','17:00:00','18:00:00','T','B','IN101',1),
(68,'IFG01','A','CHAVEZ-CENTENO-JAVIER DAVID','MA','16:00:00','18:00:00','T','A','IN101',2),
(69,'IFG01','A','CHAVEZ-CENTENO-JAVIER DAVID','JU','16:00:00','18:00:00','L','A','IN312',2),
(70,'IFG01','A','HUAMANI-HUAYHUA-AGUEDO','JU','16:00:00','18:00:00','L','B','IN304',2),
(71,'IFG01','B','SOSA-JAUREGUI-VICTOR DARIO','MA','16:00:00','18:00:00','T','B','IN102',2),
(72,'IFG01','B','SOSA-JAUREGUI-VICTOR DARIO','JU','09:00:00','11:00:00','L','B','IN309',2),
(73,'MEI05','A','ESPINOZA-YABAR-MARITZA','MA','07:00:00','09:00:00','T','A','IN103',2),
(74,'MEI05','A','ESPINOZA-YABAR-MARITZA','JU','07:00:00','09:00:00','P','A','IN103',2),
(75,'MEI05','A','ESPINOZA-YABAR-MARITZA','VI','07:00:00','08:00:00','T','A','IN103',1),
(76,'MEI05','B','RIVERO-CUSIMAYTA-EDITH','MA','14:00:00','16:00:00','T','B','IN103',2),
(77,'MEI05','B','RIVERO-CUSIMAYTA-EDITH','VI','15:00:00','16:00:00','T','B','IN103',1),
(78,'MEI05','B','QUINTANA-YAURIS-MIGUEL ANGEL','JU','14:00:00','16:00:00','P','B','IN103',2),
(79,'IFI02','A','CARBAJAL-LUNA-JULIO CESAR','MA','09:00:00','11:00:00','L','A','IN305',2),
(80,'IFI02','A','CARBAJAL-LUNA-JULIO CESAR','JU','09:00:00','11:00:00','T','A','IN203',2),
(81,'IFI02','A','CARBAJAL-LUNA-JULIO CESAR','VI','10:00:00','11:00:00','T','A','IN203',1),
(82,'IFI02','B','MONTOYA-CUBAS-CARLOS FERNANDO','MA','09:00:00','11:00:00','P','B','IN108',2),
(83,'IFI02','B','ACURIO-USCA-NILA ZONIA','MA','09:00:00','11:00:00','L','A','IN306',2),
(84,'IFI02','B','ACURIO-USCA-NILA ZONIA','JU','09:00:00','11:00:00','T','B','IN103',2),
(85,'IFI02','B','ACURIO-USCA-NILA ZONIA','VI','10:00:00','11:00:00','T','B','IN107',1),
(86,'ME350','A','MAGUIÑA-RONDAN-SIMEONA ESTILISTA','LU','14:00:00','16:00:00','T','A','IN102',2),
(87,'ME350','A','MAGUIÑA-RONDAN-SIMEONA ESTILISTA','MI','14:00:00','16:00:00','P','A','IN102',2),
(88,'ME350','A','MAGUIÑA-RONDAN-SIMEONA ESTILISTA','VI','14:00:00','15:00:00','T','A','IN102',1),
(89,'MEI09','A','GUZMAN-HUAMAN-JUAN WALTER','MA','11:00:00','13:00:00','T','A','IN103',2),
(90,'MEI09','A','GUZMAN-HUAMAN-JUAN WALTER','JU','11:00:00','13:00:00','P','A','IN103',2),
(91,'MEI09','A','GUZMAN-HUAMAN-JUAN WALTER','VI','12:00:00','13:00:00','T','A','IN103',1),
(92,'MEI09','B','CHOQUE-HUAMAN-PATRICIO','LU','16:00:00','18:00:00','T','B','IN103',2),
(93,'MEI09','B','CHOQUE-HUAMAN-PATRICIO','MI','16:00:00','18:00:00','P','B','IN103',2),
(94,'MEI09','B','CHOQUE-HUAMAN-PATRICIO','VI','17:00:00','18:00:00','T','B','IN103',1),
(95,'FI370','A','NINACHI-CONDORI-EDGAR','LU','13:00:00','15:00:00','L','A','Lab. F3',2),
(96,'FI370','A','NINACHI-CONDORI-EDGAR','MA','09:00:00','11:00:00','L','A','Lab. F3',2),
(97,'FI370','A','NINACHI-CONDORI-EDGAR','MA','13:00:00','15:00:00','L','A','Lab. F3',2),
(98,'FI370','A','BACA-FLORES-JESUS ANGEL','LU','11:00:00','13:00:00','T','A','IN103',2),
(99,'FI370','A','BACA-FLORES-JESUS ANGEL','MI','11:00:00','13:00:00','P','A','IN103',2),
(100,'FI370','A','BACA-FLORES-JESUS ANGEL','VI','11:00:00','12:00:00','T','A','IN103',1),
(101,'FI370','A','BAZAN-NUÑEZ-YURI','MI','13:00:00','15:00:00','L','A','Lab. F3',2),
(102,'FII02','A','BACA-FLORES-JESUS ANGEL','LU','11:00:00','13:00:00','T','A','IN103',2),
(103,'FII02','A','BACA-FLORES-JESUS ANGEL','MI','11:00:00','13:00:00','P','A','IN103',2),
(104,'FII02','A','BACA-FLORES-JESUS ANGEL','VI','11:00:00','12:00:00','T','A','IN103',1),
(105,'FII02','A','NINACHI-CONDORI-EDGAR','LU','13:00:00','15:00:00','L','A','Lab. F3',2),
(106,'FII02','A','NINACHI-CONDORI-EDGAR','MA','13:00:00','15:00:00','L','A','Lab. F3',2),
(107,'FII02','A','NINACHI-CONDORI-EDGAR','MA','09:00:00','11:00:00','L','A','Lab. F3',2),
(108,'FII02','A','BAZAN-NUÑEZ-YURI','MI','13:00:00','15:00:00','L','A','Lab. F3',2),
(109,'FII02','B','JANQUI-ESQUIVEL-LEONIDAS GUSTAVO','MA','16:00:00','18:00:00','T','B','IN103',2),
(110,'FII02','B','JANQUI-ESQUIVEL-LEONIDAS GUSTAVO','JU','16:00:00','18:00:00','P','B','IN103',2),
(111,'FII02','B','JANQUI-ESQUIVEL-LEONIDAS GUSTAVO','VI','16:00:00','17:00:00','T','B','IN103',1),
(112,'MEI06','A','CAZORLA-MEDINA-EDWIN','LU','07:00:00','09:00:00','T','A','IN103',2),
(113,'MEI06','A','CAZORLA-MEDINA-EDWIN','MI','07:00:00','09:00:00','P','A','IN103',2),
(114,'MEI06','A','CAZORLA-MEDINA-EDWIN','VI','08:00:00','09:00:00','T','A','IN103',1),
(115,'MEI06','B','CCOA-CHALLCO-MANRRIQUE DAVID','LU','14:00:00','16:00:00','T','B','IN103',2),
(116,'MEI06','B','CCOA-CHALLCO-MANRRIQUE DAVID','MI','14:00:00','16:00:00','P','B','IN103',2),
(117,'MEI06','B','CCOA-CHALLCO-MANRRIQUE DAVID','VI','14:00:00','15:00:00','T','B','IN103',1),
(118,'IFI03','A','DUEÑAS-BUSTINZA-DARIO FRANCISCO','LU','09:00:00','11:00:00','L','A','IN306',2),
(119,'IFI03','A','DUEÑAS-BUSTINZA-DARIO FRANCISCO','MI','09:00:00','11:00:00','L','A','IN306',2),
(120,'IFI03','B','ZAMALLOA-PARO-WILLIAN','LU','09:00:00','11:00:00','L','A','IN206',2),
(121,'IFI03','B','ZAMALLOA-PARO-WILLIAN','MI','09:00:00','11:00:00','L','A','IN206',2),
(122,'IFI03','C','ZUÑIGA-ROJAS-GABRIELA','LU','14:00:00','16:00:00','L','C','IN302',2),
(123,'IFI03','C','ZUÑIGA-ROJAS-GABRIELA','MI','14:00:00','16:00:00','L','C','IN302',2),
(124,'IFI04','A','ACURIO-USCA-NILA ZONIA','LU','09:00:00','11:00:00','T','A','IN107',2),
(125,'IFI04','A','ACURIO-USCA-NILA ZONIA','MI','09:00:00','11:00:00','L','A','IN305',2),
(126,'IFI04','A','ACURIO-USCA-NILA ZONIA','VI','09:00:00','10:00:00','T','A','IN107',1),
(127,'IFI04','B','CHOQUE-SOTO-VANESSA MARIBEL','LU','09:00:00','11:00:00','L','B','IN312',2),
(128,'IFI04','B','CHOQUE-SOTO-VANESSA MARIBEL','MI','09:00:00','11:00:00','T','B','IN107',2),
(129,'IFI04','B','CHOQUE-SOTO-VANESSA MARIBEL','VI','09:00:00','10:00:00','T','B','IN306',1),
(130,'MEI08','A','LAVILLA-CONDORI- MARI LUZ','MA','11:00:00','13:00:00','T','A','IN107',2),
(131,'MEI08','A','LAVILLA-CONDORI- MARI LUZ','JU','11:00:00','13:00:00','P','A','IN107',2),
(132,'MEI08','A','LAVILLA-CONDORI- MARI LUZ','VI','12:00:00','13:00:00','T','A','IN107',1),
(133,'EL371','A','PALOMINO-LOPEZ-ALEXANDER','LU','11:00:00','13:00:00','T','A','IN107',2),
(134,'EL371','A','PALOMINO-LOPEZ-ALEXANDER','MI','11:00:00','13:00:00','P','A','IN107',2),
(135,'EL371','A','PALOMINO-LOPEZ-ALEXANDER','VI','11:00:00','12:00:00','T','A','IN107',1),
(136,'ELI01','A','PALOMINO-LOPEZ-ALEXANDER','LU','11:00:00','13:00:00','T','A','IN107',2),
(137,'ELI01','A','PALOMINO-LOPEZ-ALEXANDER','MI','11:00:00','13:00:00','P','A','IN107',2),
(138,'ELI01','A','PALOMINO-LOPEZ-ALEXANDER','VI','11:00:00','12:00:00','T','A','IN107',1),
(139,'ELI01','B','GUILLEN-LOAYZA-MARGARITA LUZ','LU','11:00:00','13:00:00','T','B','VIRT 1 IN',2),
(140,'ELI01','B','GUILLEN-LOAYZA-MARGARITA LUZ','MI','11:00:00','13:00:00','P','B','VIRT 1 IN',2),
(141,'ELI01','B','GUILLEN-LOAYZA-MARGARITA LUZ','VI','11:00:00','12:00:00','T','B','VIRT 1 IN',1),
(142,'IFI07','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','MA','18:00:00','20:00:00','T','A','IN107',2),
(143,'IFI07','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','JU','18:00:00','20:00:00','L','A','IN107',2),
(144,'IFI07','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','VI','08:00:00','09:00:00','T','A','IN107',1),
(145,'IFI07','B','QUISPE-TORRES-GERAR FRANCIS','MA','14:00:00','16:00:00','P','B','IN202',2),
(146,'IFI07','B','QUISPE-TORRES-GERAR FRANCIS','JU','14:00:00','16:00:00','T','B','IN202',2),
(147,'IFI07','B','QUISPE-TORRES-GERAR FRANCIS','VI','10:00:00','11:00:00','T','B','IN306',1),
(148,'ADI01','A','PANDO-DIAZ-WALDO ALEX','MA','07:00:00','09:00:00','T','A','IN107',2),
(149,'ADI01','A','PANDO-DIAZ-WALDO ALEX','JU','07:00:00','09:00:00','P','A','IN107',2),
(150,'ADI01','A','PANDO-DIAZ-WALDO ALEX','VI','07:00:00','08:00:00','T','A','IN107',1),
(151,'IFI05','A','AGUIRRE-CARBAJAL-DORIS SABINA','MA','09:00:00','11:00:00','L','A','IN303',2),
(152,'IFI05','A','AGUIRRE-CARBAJAL-DORIS SABINA','JU','09:00:00','11:00:00','L','A','IN303',2),
(153,'IFI05','B','VILLALBA-VILLALBA-TANY','MA','09:00:00','11:00:00','L','A','IN304',2),
(154,'IFI05','B','VILLALBA-VILLALBA-TANY','JU','09:00:00','11:00:00','L','A','IN304',2),
(155,'IFI08','A','ROZAS-HUACHO-JAVIER ARTURO','LU','07:00:00','09:00:00','T','A','IN108',2),
(156,'IFI08','A','ROZAS-HUACHO-JAVIER ARTURO','VI','07:00:00','08:00:00','T','A','IN108',1),
(157,'IFI08','A','QUISPE-SOTA-JULIO VLADIMIR','MI','07:00:00','09:00:00','L','A','IN305',2),
(158,'IF610','A','CANDIA-OVIEDO-DENNIS IVAN','LU','09:00:00','11:00:00','T','A','IN103',2),
(159,'IF610','A','CANDIA-OVIEDO-DENNIS IVAN','MI','09:00:00','11:00:00','L','A','IN301',2),
(160,'IF610','A','CANDIA-OVIEDO-DENNIS IVAN','VI','08:00:00','09:00:00','T','A','IN301',1),
(161,'IF610','A','SOSA-JAUREGUI-VICTOR DARIO','MI','09:00:00','11:00:00','P','A','IN302',2),
(162,'IFI10','A','CANDIA-OVIEDO-DENNIS IVAN','LU','09:00:00','11:00:00','T','A','IN103',2),
(163,'IFI10','A','CANDIA-OVIEDO-DENNIS IVAN','MI','09:00:00','11:00:00','L','A','IN301',2),
(164,'IFI10','A','CANDIA-OVIEDO-DENNIS IVAN','VI','08:00:00','09:00:00','T','A','IN301',1),
(165,'IFI10','A','SOSA-JAUREGUI-VICTOR DARIO','MI','09:00:00','11:00:00','P','A','IN302',2),
(166,'ME356','A','LAVILLA-CONDORI- MARI LUZ','MA','11:00:00','13:00:00','T','A','IN107',2),
(167,'ME356','A','LAVILLA-CONDORI- MARI LUZ','JU','11:00:00','13:00:00','P','A','IN107',2),
(168,'ME356','A','LAVILLA-CONDORI- MARI LUZ','VI','12:00:00','13:00:00','T','A','IN107',1),
(169,'IF481','A','PALOMINO-OLIVERA-EMILIO','LU','07:00:00','09:00:00','L','A','IN305',2),
(170,'IF481','A','PALOMINO-OLIVERA-EMILIO','MI','07:00:00','09:00:00','T','A','IN108',2),
(171,'IFI12','A','ESPEJO-ALVAREZ-BENJAMIN','MA','16:00:00','18:00:00','P','A','IN309',2),
(172,'IFI12','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','JU','16:00:00','18:00:00','T','A','IN107',2),
(173,'IFI12','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','VI','17:00:00','18:00:00','T','A','IN107',1),
(174,'IF650','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','MA','18:00:00','20:00:00','T','A','IN107',2),
(175,'IF650','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','JU','18:00:00','20:00:00','L','A','IN107',2),
(176,'IF650','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','VI','08:00:00','09:00:00','T','A','IN107',1),
(177,'IF550','A','PEÑALOZA-FIGUEROA-MANUEL AURELIO','LU','16:00:00','18:00:00','L','A','IN301',2),
(178,'IF550','A','PEÑALOZA-FIGUEROA-MANUEL AURELIO','MI','16:00:00','18:00:00','T','A','IN108',2),
(179,'IF550','A','PEÑALOZA-FIGUEROA-MANUEL AURELIO','VI','16:00:00','17:00:00','T','A','IN108',1),
(180,'IF550','A','LAVILLA-ALVAREZ-VANESA','JU','09:00:00','11:00:00','P','A','IN302',2),
(181,'IFI11','A','CARRASCO-POBLETE-EDWIN','MA','09:00:00','11:00:00','T','A','IN103',2),
(182,'IFI11','A','CARRASCO-POBLETE-EDWIN','JU','09:00:00','11:00:00','L','A','IN301',2),
(183,'IFI11','A','CARRASCO-POBLETE-EDWIN','VI','09:00:00','10:00:00','T','A','IN103',1),
(184,'IFI11','B','PEÑALOZA-FIGUEROA-MANUEL AURELIO','LU','16:00:00','18:00:00','L','A','IN301',2),
(185,'IFI11','B','PEÑALOZA-FIGUEROA-MANUEL AURELIO','MI','16:00:00','18:00:00','T','B','IN108',2),
(186,'IFI11','B','PEÑALOZA-FIGUEROA-MANUEL AURELIO','VI','16:00:00','17:00:00','T','B','IN108',1),
(187,'IFI11','B','LAVILLA-ALVAREZ-VANESA','JU','09:00:00','11:00:00','P','B','IN302',2),
(188,'IFI09','A','UGARTE-ROJAS-HECTOR EDUARDO','MA','07:00:00','09:00:00','L','A','IN306',2),
(189,'IFI09','A','UGARTE-ROJAS-HECTOR EDUARDO','JU','07:00:00','09:00:00','L','A','IN306',2),
(190,'IFI09','B','CHULLO-LLAVE-BORIS','MA','07:00:00','09:00:00','L','B','IN206',2),
(191,'IFI09','B','CHULLO-LLAVE-BORIS','JU','07:00:00','09:00:00','L','B','IN206',2),
(192,'IF454','A','HUILLCA-HUAMAN-ALEX FERNANDO','LU','14:00:00','16:00:00','T','A','IN202',2),
(193,'IF454','A','HUILLCA-HUAMAN-ALEX FERNANDO','VI','14:00:00','16:00:00','P','A','IN309',2),
(194,'IFI13','A','SONCCO-ALVAREZ-JOSE LUIS','LU','11:00:00','13:00:00','T','A','IN206',2),
(195,'IFI13','A','SONCCO-ALVAREZ-JOSE LUIS','MI','11:00:00','13:00:00','P','A','IN309',2),
(196,'IFI13','A','SONCCO-ALVAREZ-JOSE LUIS','VI','11:00:00','12:00:00','T','A','IN206',1),
(197,'IF455','A','MEDRANO-VALENCIA-IVAN CESAR','LU','11:00:00','13:00:00','L','A','IN301',2),
(198,'IF455','A','MEDRANO-VALENCIA-IVAN CESAR','MI','11:00:00','13:00:00','T','A','IN201',2),
(199,'IF455','A','MEDRANO-VALENCIA-IVAN CESAR','VI','11:00:00','12:00:00','T','A','IN201',1),
(200,'IF455','A','MONTOYA-CUBAS-CARLOS FERNANDO','LU','11:00:00','13:00:00','L','B','IN302',2),
(201,'IF455','B','MONTOYA-CUBAS-CARLOS FERNANDO','MI','11:00:00','13:00:00','T','B','V. EG IN 1',2),
(202,'IF455','B','MONTOYA-CUBAS-CARLOS FERNANDO','VI','11:00:00','12:00:00','T','B','V. EG IN 1',1),
(203,'IF458','A','CHULLO-LLAVE-BORIS','LU','07:00:00','09:00:00','L','A','IN301',2),
(204,'IF458','A','CHULLO-LLAVE-BORIS','MI','07:00:00','09:00:00','T','A','IN201',2),
(205,'IF458','A','CHULLO-LLAVE-BORIS','VI','08:00:00','09:00:00','T','A','IN201',1),
(206,'IF458','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','LU','07:00:00','09:00:00','L','B','IN302',2),
(207,'IF612','A','ROZAS-HUACHO-JAVIER ARTURO','MA','09:00:00','11:00:00','L','A','IN301',2),
(208,'IF612','A','ROZAS-HUACHO-JAVIER ARTURO','JU','09:00:00','11:00:00','T','A','IN108',2),
(209,'IF612','A','ROZAS-HUACHO-JAVIER ARTURO','VI','09:00:00','10:00:00','T','A','IN108',1),
(210,'IF612','B','ZAMALLOA-PARO-WILLIAN','MA','09:00:00','11:00:00','L','B','IN302',2),
(211,'IF612','B','ZAMALLOA-PARO-WILLIAN','JU','09:00:00','11:00:00','T','B','IN201',2),
(212,'IF612','B','ZAMALLOA-PARO-WILLIAN','VI','09:00:00','10:00:00','T','B','IN201',1),
(213,'IF611','A','IBARRA-ZANBRANO-WALDO ELIO','MA','07:00:00','09:00:00','L','A','IN301',2),
(214,'IF611','A','IBARRA-ZANBRANO-WALDO ELIO','JU','07:00:00','09:00:00','T','A','IN201',2),
(215,'IF611','B','QUISPE-ONOFRE-CARLOS RAMON','MA','07:00:00','09:00:00','L','A','IN302',2),
(216,'IF611','B','QUISPE-ONOFRE-CARLOS RAMON','JU','07:00:00','09:00:00','T','B','IN202',2),
(217,'IF457','A','BACA-CARDENAS-LINO AQUILES','MA','16:00:00','18:00:00','L','A','IN302',2),
(218,'IF457','A','QUISPE-SURCO-VITTALI','MA','16:00:00','18:00:00','P','A','IN305',2),
(219,'IF457','A','MEDINA-MIRANDA-KARELIA','JU','16:00:00','18:00:00','T','A','IN208',2),
(220,'IF551','A','CARRASCO-POBLETE-EDWIN','LU','09:00:00','11:00:00','L','A','IN301',2),
(221,'IF551','A','CARRASCO-POBLETE-EDWIN','MI','09:00:00','11:00:00','T','A','IN201',2),
(222,'IF551','A','CARRASCO-POBLETE-EDWIN','VI','10:00:00','11:00:00','T','A','IN201',1),
(223,'IF551','A','PALOMINO-OLIVERA-EMILIO','LU','09:00:00','11:00:00','L','B','IN302',2),
(224,'IF456','A','HUILLCA-HUALLPARIMACHI-RAUL','LU','16:00:00','18:00:00','T','A','IN108',2),
(225,'IF456','A','HUILLCA-HUALLPARIMACHI-RAUL','MI','16:00:00','18:00:00','L','A','IN301',2),
(226,'IF456','A','HUILLCA-HUALLPARIMACHI-RAUL','VI','17:00:00','18:00:00','T','A','IN108',1),
(227,'IF456','A','LAVILLA-ALVAREZ-VANESA','MI','16:00:00','18:00:00','P','A','IN310',2),
(228,'IF456','B','HOLGUIN-GARATE-JOHN WALTER','LU','16:00:00','18:00:00','T','B','IN202',2),
(229,'IF456','B','HOLGUIN-GARATE-JOHN WALTER','MI','16:00:00','18:00:00','L','B','IN302',2),
(230,'IF456','B','HOLGUIN-GARATE-JOHN WALTER','VI','17:00:00','18:00:00','T','B','IN202',1),
(231,'IF467','A','DUEÑAS-JIMENEZ-RAY','LU','07:00:00','09:00:00','T','A','IN203',2),
(232,'IF467','A','DUEÑAS-JIMENEZ-RAY','MI','07:00:00','09:00:00','L','A','IN306',2),
(233,'IF467','A','MONTOYA-CUBAS-CARLOS FERNANDO','MI','07:00:00','09:00:00','P','A','IN311',2),
(234,'IF466','A','FLORES-PACHECO-LINO PRISCILIANO','LU','11:00:00','13:00:00','T','A','IN201',2),
(235,'IF466','A','FLORES-PACHECO-LINO PRISCILIANO','MI','11:00:00','13:00:00','L','A','IN301',2),
(236,'IF466','A','SOSA-JAUREGUI-VICTOR DARIO','MI','11:00:00','13:00:00','L','B','IN302',2),
(237,'IF613','A','MONZON-CONDORI-LUIS ALVARO','MA','07:00:00','09:00:00','L','A','IN303',2),
(238,'IF613','A','MONZON-CONDORI-LUIS ALVARO','JU','07:00:00','09:00:00','L','A','IN303',2),
(239,'IF613','B','QUISPE-SOTA-JULIO VLADIMIR','MA','07:00:00','09:00:00','L','A','IN304',2),
(240,'IF613','B','QUISPE-SOTA-JULIO VLADIMIR','JU','07:00:00','09:00:00','L','A','IN304',2),
(241,'IF651','A','IRPANOCCA-CUSIMAYTA-MARITZA','LU','09:00:00','11:00:00','L','A','IN303',2),
(242,'IF651','A','IRPANOCCA-CUSIMAYTA-MARITZA','MI','09:00:00','11:00:00','T','A','IN108',2),
(243,'IF651','A','IRPANOCCA-CUSIMAYTA-MARITZA','VI','10:00:00','11:00:00','T','A','IN108',1),
(244,'IF651','A','VILLAFUERTE-SERNA-RONY','LU','09:00:00','11:00:00','L','B','IN304',2),
(245,'IF651','B','GAMARRA-SALAS-JISBAJ','MI','09:00:00','11:00:00','T','B','IN207',2),
(246,'IF651','B','GAMARRA-SALAS-JISBAJ','VI','10:00:00','11:00:00','T','B','IN207',1),
(247,'IF063','A','PACHECO-VASQUEZ-ESTHER CRISTINA','MA','16:00:00','18:00:00','P','A','IN108',2),
(248,'IF063','A','PACHECO-VASQUEZ-ESTHER CRISTINA','JU','16:00:00','18:00:00','P','A','IN108',2),
(249,'IF552','A','VILLAFUERTE-SERNA-RONY','MA','09:00:00','11:00:00','T','A','IN203',2),
(250,'IF552','A','VILLAFUERTE-SERNA-RONY','JU','09:00:00','11:00:00','L','A','IN305',2),
(251,'IF552','A','VILLAFUERTE-SERNA-RONY','VI','09:00:00','10:00:00','T','A','IN203',1),
(252,'IF552','B','DUEÑAS-BUSTINZA-DARIO FRANCISCO','MA','09:00:00','11:00:00','T','B','IN207',2),
(253,'IF552','B','DUEÑAS-BUSTINZA-DARIO FRANCISCO','JU','09:00:00','11:00:00','L','A','IN306',2),
(254,'IF552','B','DUEÑAS-BUSTINZA-DARIO FRANCISCO','VI','09:00:00','10:00:00','T','B','IN207',1),
(255,'IF652','A','ROZAS-HUACHO-JAVIER ARTURO','LU','09:00:00','11:00:00','T','A','IN202',2),
(256,'IF652','A','ROZAS-HUACHO-JAVIER ARTURO','VI','10:00:00','11:00:00','T','A','IN202',1),
(257,'IF652','A','MONTOYA-CUBAS-CARLOS FERNANDO','MI','09:00:00','11:00:00','L','A','IN303',2),
(258,'IF652','A','ALZAMORA-PAREDES-ROBERT','MI','09:00:00','11:00:00','L','A','IN304',2),
(259,'IF664','A','DUEÑAS-DE LA CRUZ-HENRY SAMUEL','LU','11:00:00','13:00:00','T','A','IN202',2),
(260,'IF664','A','DUEÑAS-DE LA CRUZ-HENRY SAMUEL','MI','11:00:00','13:00:00','L','A','IN303',2),
(261,'IF664','A','DUEÑAS-DE LA CRUZ-HENRY SAMUEL','VI','11:00:00','12:00:00','T','A','IN202',1),
(262,'IF614','A','GAMARRA-SALAS-JISBAJ','MA','07:00:00','09:00:00','T','A','IN202',2),
(263,'IF614','A','GAMARRA-SALAS-JISBAJ','VI','07:00:00','08:00:00','T','A','IN202',1),
(264,'IF614','A','CAMA-CACERES-ELMER','JU','07:00:00','09:00:00','L','A','IN301',2),
(265,'IF614','B','COSIO-LOAIZA-STEPHAN JHOEL','MA','14:00:00','16:00:00','T','B','IN101',2),
(266,'IF614','B','COSIO-LOAIZA-STEPHAN JHOEL','JU','14:00:00','16:00:00','L','B','IN303',2),
(267,'IF614','B','COSIO-LOAIZA-STEPHAN JHOEL','VI','14:00:00','15:00:00','T','B','IN107',1),
(268,'IF614','C','TICONA-FELIX-LUZ INDIRA','MA','07:00:00','09:00:00','T','C','IN201',2),
(269,'IF614','C','TICONA-FELIX-LUZ INDIRA','VI','07:00:00','08:00:00','T','C','IN309',1),
(270,'IF614','C','ABARCA-MORA-RAIMAR','JU','07:00:00','09:00:00','P','C','IN309',2),
(271,'IF669','A','PALMA-TTITO-LUIS BELTRAN','LU','07:00:00','09:00:00','T','A','IN202',2),
(272,'IF669','A','PALMA-TTITO-LUIS BELTRAN','MI','07:00:00','09:00:00','L','A','IN301',2),
(273,'IF669','A','PALMA-TTITO-LUIS BELTRAN','VI','08:00:00','09:00:00','T','A','IN202',1),
(274,'IF669','A','FALCON-HUALLPA-ELIDA','MI','07:00:00','09:00:00','L','B','IN302',2),
(275,'IF482','A','TICONA-PARI-GUZMAN','MA','09:00:00','11:00:00','T','A','IN202',2),
(276,'IF482','A','TICONA-PARI-GUZMAN','JU','09:00:00','11:00:00','P','A','IN202',2),
(277,'IF710','A','ORMEÑO-AYALA-YESHICA ISELA','LU','16:00:00','18:00:00','T','A','IN201',2),
(278,'IF710','A','ORMEÑO-AYALA-YESHICA ISELA','MI','16:00:00','18:00:00','P','A','IN201',2),
(279,'IF710','B','VILLAFUERTE-SERNA-RONY','LU','11:00:00','13:00:00','T','B','IN207',2),
(280,'IF710','B','VILLAFUERTE-SERNA-RONY','MI','11:00:00','13:00:00','P','B','IN207',2),
(281,'IF710','C','ZUÑIGA-ROJAS-GABRIELA','LU','11:00:00','13:00:00','T','C','IN306',2),
(282,'IF710','C','ZUÑIGA-ROJAS-GABRIELA','MI','11:00:00','13:00:00','P','C','IN306',2),
(283,'IF459','A','QUISPE-ONOFRE-CARLOS RAMON','LU','07:00:00','09:00:00','T','A','IN201',2),
(284,'IF459','A','QUISPE-ONOFRE-CARLOS RAMON','MI','07:00:00','09:00:00','L','A','IN304',2),
(285,'IF459','A','QUISPE-ONOFRE-CARLOS RAMON','VI','07:00:00','08:00:00','T','A','IN201',1),
(286,'IF062','A','GAMARRA-SALAS-JISBAJ','LU','14:00:00','16:00:00','P','A','IN108',2),
(287,'IF062','A','GAMARRA-SALAS-JISBAJ','MI','14:00:00','16:00:00','P','A','IN108',2),
(288,'IF483','A','PALOMINO-OLIVERA-EMILIO','MA','09:00:00','11:00:00','T','A','IN107',2),
(289,'IF483','A','PALOMINO-OLIVERA-EMILIO','JU','09:00:00','11:00:00','P','A','IN107',2),
(290,'IF483','B','CHOQUE-SOTO-VANESSA MARIBEL','MA','09:00:00','11:00:00','P','B','IN201',2),
(291,'IF483','B','CHOQUE-SOTO-VANESSA MARIBEL','JU','09:00:00','11:00:00','T','B','IN207',2),
(292,'IF617','A','VENEGAS-VERGARA-MARIA DEL PILAR','MA','07:00:00','09:00:00','L','A','IN305',2),
(293,'IF617','A','VENEGAS-VERGARA-MARIA DEL PILAR','JU','07:00:00','09:00:00','T','A','IN207',2),
(294,'IF617','A','VENEGAS-VERGARA-MARIA DEL PILAR','VI','07:00:00','08:00:00','T','A','IN207',1),
(295,'IF617','A','VILLENA-LEON-OLMER CLAUDIO','MA','07:00:00','09:00:00','P','A','IN312',2),
(296,'IF656','A','ALZAMORA-PAREDES-ROBERT','LU','07:00:00','09:00:00','L','A','IN303',2),
(297,'IF656','A','ALZAMORA-PAREDES-ROBERT','MI','07:00:00','09:00:00','T','A','IN203',2),
(298,'IF656','A','ALZAMORA-PAREDES-ROBERT','VI','08:00:00','09:00:00','T','A','IN203',1),
(299,'IF656','A','ZUÑIGA-ROJAS-GABRIELA','LU','07:00:00','09:00:00','L','B','IN304',2),
(300,'IF654','A','PILLCO-QUISPE-JOSE MAURO','LU','11:00:00','13:00:00','T','A','IN203',2),
(301,'IF654','A','PILLCO-QUISPE-JOSE MAURO','VI','11:00:00','12:00:00','T','A','IN203',1),
(302,'IF654','A','COSIO-LOAIZA-STEPHAN JHOEL','MI','11:00:00','13:00:00','L','A','IN304',2),
(303,'IF657','A','MEDRANO-VALENCIA-IVAN CESAR','LU','09:00:00','11:00:00','L','A','IN305',2),
(304,'IF657','A','MEDRANO-VALENCIA-IVAN CESAR','MI','09:00:00','11:00:00','T','A','IN202',2),
(305,'IF657','A','MEDRANO-VALENCIA-IVAN CESAR','VI','09:00:00','10:00:00','T','A','IN202',1),
(306,'IF657','A','MONTOYA-CUBAS-CARLOS FERNANDO','LU','09:00:00','11:00:00','P','A','IN309',2),
(307,'IF657','B','MONTOYA-CUBAS-CARLOS FERNANDO','LU','07:00:00','09:00:00','T','B','IN208',2),
(308,'IF657','B','MONTOYA-CUBAS-CARLOS FERNANDO','VI','08:00:00','09:00:00','T','B','IN208',1),
(309,'IF657','B','ZUÑIGA-ROJAS-GABRIELA','MI','07:00:00','09:00:00','L','B','IN303',2),
(310,'IF711','A','FLORES-PACHECO-LINO PRISCILIANO','LU','14:00:00','16:00:00','T','A','IN201',2),
(311,'IF711','A','FLORES-PACHECO-LINO PRISCILIANO','MI','14:00:00','16:00:00','P','A','IN201',2),
(312,'IF619','A','PILLCO-QUISPE-JOSE MAURO','MA','16:00:00','18:00:00','L','A','IN304',2),
(313,'IF619','A','PILLCO-QUISPE-JOSE MAURO','JU','16:00:00','18:00:00','T','A','IN102',2),
(314,'IF619','A','MEDINA-MIRANDA-KARELIA','VI','16:00:00','18:00:00','L','B','IN304',2),
(315,'IF460','A','HUILLCA-HUALLPARIMACHI-RAUL','MA','14:00:00','16:00:00','T','A','IN201',2),
(316,'IF460','A','HUILLCA-HUALLPARIMACHI-RAUL','JU','14:00:00','16:00:00','L','A','IN301',2),
(317,'IF460','A','HUILLCA-HUALLPARIMACHI-RAUL','VI','14:00:00','15:00:00','T','A','IN201',1),
(318,'IF061','A','DIAZ-CACERES-LISHA SABAH','MA','14:00:00','16:00:00','P','A','IN102',2),
(319,'IF061','A','DIAZ-CACERES-LISHA SABAH','JU','14:00:00','16:00:00','P','A','IN102',2),
(320,'IF662','A','ALZAMORA-PAREDES-ROBERT','MA','07:00:00','09:00:00','T','A','IN203',2),
(321,'IF662','A','ALZAMORA-PAREDES-ROBERT','JU','07:00:00','09:00:00','L','A','IN305',2),
(322,'IF662','A','ALZAMORA-PAREDES-ROBERT','VI','07:00:00','08:00:00','T','A','IN203',1),
(323,'IF662','A','CCACYAHUILLCA-BEJAR-HANS HARLEY','JU','07:00:00','09:00:00','L','B','IN302',2),
(324,'IF616','A','SEGUNDO-CARPIO-LISETH URPY','LU','16:00:00','18:00:00','L','A','IN303',2),
(325,'IF616','A','SEGUNDO-CARPIO-LISETH URPY','MI','16:00:00','18:00:00','L','A','IN303',2),
(326,'IF616','B','CCACYAHUILLCA-BEJAR-HANS HARLEY','LU','16:00:00','18:00:00','L','B','IN304',2),
(327,'IF616','B','CCACYAHUILLCA-BEJAR-HANS HARLEY','MI','16:00:00','18:00:00','L','B','IN304',2),
(328,'IF484','A','VERA-OLIVERA-HARLEY','MA','14:00:00','16:00:00','T','A','IN108',2),
(329,'IF484','A','VERA-OLIVERA-HARLEY','JU','14:00:00','16:00:00','P','A','IN108',2),
(330,'MEI65','A','FARFAN-ESCALANTE-GUIDO','LU','07:00:00','09:00:00','T','A','IN107',2),
(331,'MEI65','A','FARFAN-ESCALANTE-GUIDO','MI','07:00:00','09:00:00','P','A','IN107',2),
(332,'MEI65','A','FARFAN-ESCALANTE-GUIDO','VI','07:00:00','08:00:00','T','A','IN102',1),
(333,'IF653','A','QUISPE-SOTA-JULIO VLADIMIR','MA','16:00:00','18:00:00','L','A','IN303',2),
(334,'IF653','A','QUISPE-SOTA-JULIO VLADIMIR','JU','16:00:00','18:00:00','T','A','IN201',2),
(335,'IF653','A','QUISPE-SOTA-JULIO VLADIMIR','VI','16:00:00','17:00:00','T','A','IN201',1),
(336,'IF554','A','PEÑALOZA-FIGUEROA-MANUEL AURELIO','MA','16:00:00','18:00:00','T','A','IN201',2),
(337,'IF554','A','PEÑALOZA-FIGUEROA-MANUEL AURELIO','JU','16:00:00','18:00:00','L','A','IN301',2),
(338,'IF554','A','PEÑALOZA-FIGUEROA-MANUEL AURELIO','VI','17:00:00','18:00:00','T','A','IN201',1),
(339,'IF554','B','COSIO-LOAIZA-STEPHAN JHOEL','LU','14:00:00','16:00:00','T','B','IN203',2),
(340,'IF554','B','COSIO-LOAIZA-STEPHAN JHOEL','MI','14:00:00','16:00:00','P','B','IN306',2),
(341,'IF554','B','COSIO-LOAIZA-STEPHAN JHOEL','VI','15:00:00','16:00:00','T','B','IN203',1),
(342,'IF556','A','PILLCO-QUISPE-JOSE MAURO','MA','14:00:00','16:00:00','L','A','IN301',2),
(343,'IF556','A','PILLCO-QUISPE-JOSE MAURO','JU','14:00:00','16:00:00','T','A','IN201',2),
(344,'IF556','A','PILLCO-QUISPE-JOSE MAURO','VI','15:00:00','16:00:00','T','A','IN201',1),
(345,'IF064','A','CUTIPA-ARAPA-EFRAINA GLADYS','MA','14:00:00','16:00:00','P','A','IN107',2),
(346,'IF064','A','CUTIPA-ARAPA-EFRAINA GLADYS','JU','14:00:00','16:00:00','P','A','IN107',2);


-- ---------- INFORMACION GENERAL ampliada ----------

CREATE TABLE informacion_general (
  id INT PRIMARY KEY,
  titulo VARCHAR(255) NOT NULL,
  contenido TEXT NOT NULL
);

INSERT INTO informacion_general VALUES
(1,'Historia','Fue creada el 13 de Diciembre de 1971, dentro del Plan de Estructuración del Programa Académico de Ciencias Físico-Matemáticas (Res. CG-110-71). Mediante Res. CU-056-91 se aprueba el Proyecto para la Creación del Departamento Académico de Informática. Es reaperturada el 22 de Enero de 1993 (Res. CU-009-93). El 28 de noviembre de 2013 se aprueba el Proyecto de Acreditación (Res. R-2193-2013-UNSAAC). Con la Ley Universitaria 30220, se convierte en parte de la Facultad de Ingeniería Eléctrica, Electrónica, Informática y Mecánica (FIEEIM) con el nombre de Escuela Profesional de Ingeniería Informática y de Sistemas.'),
(2,'Misión','La Escuela Profesional de Ingeniería Informática y de Sistemas realiza y fomenta la investigación mediante cursos, conferencias y coordinación permanente con docentes y estudiantes, promoviendo el desarrollo y conclusión de tesis y el aporte tecnológico a la región.'),
(3,'Visión','Ser un referente en el desarrollo de la informática en la región Cusco, formando profesionales competentes que contribuyan al desarrollo social, productivo y tecnológico, y fomenten la investigación y el desarrollo de software.'),
(4,'Directora','Yeshica Isela Ormeño Ayala'),
(5,'Departamento Académico','Lino Prisciliano Flores Pacheco (Jefe del Departamento Académico de Ingeniería Informática)'),
(6,'Teléfono','084-232398, anexo 1402'),
(7,'Ubicación','Facultad de Ingeniería Eléctrica, Electrónica, Informática y Mecánica (FIEEIM), Ciudad Universitaria de Perayoc, Cusco'),
(8,'Duración','5 años (10 semestres académicos), modalidad presencial'),
(9,'Acreditación','La Escuela ha desarrollado procesos de acreditación; el seguimiento institucional se realiza con referencia a ICACIT. La vigencia debe verificarse en el portal oficial in.unsaac.edu.pe antes de citarla en documentos formales.'),
(10,'Perfil del egresado','El egresado de Ingeniería Informática y de Sistemas posee competencias en: análisis y diseño de sistemas, desarrollo de software, ingeniería de datos e inteligencia artificial, redes y sistemas operativos, gestión de proyectos de tecnologías de la información, e investigación aplicada. Domina el inglés a nivel básico-intermedio y el quechua en el contexto cultural andino.'),
(11,'Campo laboral','El egresado puede desempeñarse como ingeniero de software, científico de datos, analista de sistemas, administrador de bases de datos o redes, consultor TI, líder de proyectos tecnológicos, investigador y docente universitario. Sectores: banca, gobierno, salud, educación, telecomunicaciones, retail, consultoría y startups.'),
(12,'Sitio web oficial','in.unsaac.edu.pe — sitio de la Escuela Profesional; daii.unsaac.edu.pe — Departamento Académico; ccomputo.unsaac.edu.pe — Centro de Cómputo (catálogo de cursos).');

-- ---------- INGRESANTES (igual) ----------

CREATE TABLE ingresantes (
  id INT PRIMARY KEY,
  oportunidad VARCHAR(50) NOT NULL,
  modalidad VARCHAR(50) NOT NULL,
  mayor_nota DECIMAL(5,2) NOT NULL,
  anio INT NOT NULL,
  cantidad INT NOT NULL
);

INSERT INTO ingresantes VALUES
(1,'Primera oportunidad','CEPRU',18.25,2025,12),
(2,'Primera oportunidad','Examen Admisión',16.00,2025,24),
(3,'Ordinario','CEPRU',16.25,2025,15),
(4,'Ordinario','Examen Admisión',18.00,2025,29);

-- ---------- TEMAS DE TESIS (igual) ----------

CREATE TABLE temas_tesis (
  id INT PRIMARY KEY,
  tema VARCHAR(255) NOT NULL
);

INSERT INTO temas_tesis VALUES
(1,'Inteligencia Artificial'),
(2,'Ciberseguridad'),
(3,'Sistemas Distribuidos'),
(4,'Big Data');

-- ---------- TRAMITES (ampliada — 10 trámites del PDF) ----------

CREATE TABLE tramites (
  id INT PRIMARY KEY,
  descripcion VARCHAR(60) NOT NULL,
  detalle TEXT NOT NULL,
  requisitos TEXT,
  plazo VARCHAR(80),
  costo VARCHAR(40),
  enlace VARCHAR(200)
);

INSERT INTO tramites VALUES
(1,'Matrícula de alumnos regulares',
 'Acto formal y voluntario que confiere la calidad de estudiante. El estudiante selecciona las asignaturas del Plan de Estudios que le corresponde llevar en el semestre.',
 '1. Figurar en el sistema como alumno regular. 2. No ser deudor de ninguna dependencia de la UNSAAC. 3. Pago según escala.',
 'Según calendario académico',
 'Escala A (invictos): S/ 20.00; Escala B: S/ 2.50 a S/ 8.50 por crédito según asignaturas desaprobadas; Escala C: S/ 12.50 por crédito',
 'ccomputo.unsaac.edu.pe'),

(2,'Matrícula de ingresante',
 'El ingresado admitido a la UNSAAC realiza su matrícula al grupo correspondiente de Estudios Generales.',
 '1. Registrar matrícula vía internet (Centro de Cómputo). 2. Pago por derechos de examen médico, carné universitario, carné de biblioteca y matrícula.',
 'Según calendario académico',
 'S/ 113.00 (incluye examen médico S/ 70, carné S/ 17, biblioteca S/ 6, matrícula S/ 20)',
 'ccomputo.unsaac.edu.pe'),

(3,'Matrícula condicionada por bajo rendimiento',
 'El estudiante solicita matrícula con pleno conocimiento de condicionamiento a superar el bajo récord académico.',
 '1. Solicitud dirigida al Rector. 2. Pago por derechos de trámite.',
 '5 días hábiles',
 'S/ 5.00',
 'tramite.unsaac.edu.pe'),

(4,'Matrícula en cursos dirigidos',
 'Facilidad para completar créditos cuando la asignatura no se ofrece en el ciclo. Solo para estudiantes por egresar, máximo 2 cursos.',
 '1. Solicitud dirigida al Rector. 2. Ficha de seguimiento académico. 3. Formato de matrícula firmado por el Director de Escuela acreditando situación por egresar. 4. Pago por asignatura.',
 '2 días hábiles',
 'S/ 25.00 por asignatura',
 'tramite.unsaac.edu.pe'),

(5,'Convalidación de asignaturas',
 'Reconocimiento de asignaturas aprobadas en otra Facultad, universidad peruana o extranjera, Fuerzas Armadas, PNP o institución con rango universitario.',
 '1. Solicitud dirigida al Rector. 2. Certificado de estudios originales de la universidad de origen (apostillados si son extranjeros). 3. Sílabos certificados de cada asignatura aprobada. 4. Plan de estudios autenticado de la institución de procedencia. 5. Pago por convalidación.',
 '10 días hábiles',
 'S/ 5.00 por asignatura',
 'tramite.unsaac.edu.pe'),

(6,'Homologación de asignaturas',
 'Reconocimiento en créditos de asignaturas aprobadas por otra con diferente denominación vigente en el Plan de Estudios, sea por traslado interno o cambio de currículo.',
 '1. Solicitud dirigida al Rector con relación de asignaturas (nombre, código, creditaje, categoría, fecha). 2. Ficha de seguimiento académico. 3. Pago por cuadro de homologación.',
 '5 días hábiles',
 'S/ 15.00',
 'tramite.unsaac.edu.pe'),

(7,'Reserva de matrícula',
 'Suspensión temporal de estudios por motivos de salud, trabajo u otros. Se tramita hasta 65 días antes de finalizado el semestre, por un máximo de 6 semestres consecutivos o alternos.',
 '1. Solicitud dirigida al Rector. 2. Ficha de seguimiento académico. 3. Documentos sustentatorios (certificado médico, de trabajo u otros). 4. Pago por derechos.',
 '5 días hábiles',
 'S/ 15.00',
 'tramite.unsaac.edu.pe'),

(8,'Reinicio de estudios',
 'Para el estudiante que dejó de matricularse por más de 2 semestres (hasta un límite de 6) y desea recuperar la condición de alumno regular.',
 '1. Solicitud dirigida al Rector. 2. Constancia de notas del último semestre cursado. 3. Pago por reinicio. 4. Si aplica, constancia de homologación o convalidación.',
 'Según calendario',
 'S/ 22.00 por semestre',
 'tramite.unsaac.edu.pe'),

(9,'Traslado interno (misma Facultad)',
 'Cambio de Escuela Profesional a otra dentro de la misma Facultad. Solo se realiza en el 2do semestre académico.',
 '1. Solicitud dirigida al Rector. 2. Ficha de seguimiento académico que acredite mínimo 72 créditos aprobados. 3. No estar en bajo rendimiento académico. 4. Pago por traslado.',
 '5 días hábiles',
 'S/ 110.00',
 'tramite.unsaac.edu.pe'),

(10,'Traslado interno (otra Facultad)',
 'Cambio de Escuela Profesional a otra de distinta Facultad. Solo se realiza en el 2do semestre académico.',
 '1. Solicitud dirigida al Rector. 2. Ficha de seguimiento académico que acredite mínimo 72 créditos aprobados. 3. No estar en bajo rendimiento académico. 4. Pago por traslado.',
 '5 días hábiles',
 'S/ 141.00',
 'tramite.unsaac.edu.pe'),

(11,'Traslado externo',
 'Ingreso desde otra universidad del país o del extranjero para realizar estudios en una Escuela Profesional de la UNSAAC.',
 '1. Solicitud al Rector. 2. DNI. 3. Certificado de estudios originales que acredite mínimo 4 semestres o 72 créditos (apostillado si es extranjero). 4. Certificado electrónico de no tener antecedentes penales ni judiciales. 5. Sílabos de asignaturas aprobadas. 6. Certificado de no ser deudor de la universidad de procedencia. 7. Constancia de no haber sido separado por disciplina o bajo rendimiento. 8. Pago.',
 '30 días hábiles',
 'S/ 450.00',
 'tramite.unsaac.edu.pe'),

(12,'Admisión de titulados o graduados',
 'Graduados o titulados de universidades del país o del extranjero solicitan plaza para obtener una segunda profesión en la UNSAAC.',
 '1. Solicitud al Rector. 2. DNI. 3. Copia de diploma de grado o título (inscrito en SUNEDU). 4. Certificados de estudios de pregrado. 5. Pago por inscripción.',
 '20 días hábiles',
 'S/ 453.00',
 'tramite.unsaac.edu.pe'),

(13,'Subsanación de asignaturas (por egresar)',
 'El estudiante por egresar puede subsanar hasta 2 asignaturas desaprobadas con promedio mínimo de 10 puntos para completar créditos del plan de estudios.',
 '1. Solicitud al Rector. 2. Tener condición de egresante con máximo 2 asignaturas desaprobadas. 3. Ficha de seguimiento académico. 4. Pago por subsanación.',
 '5 días hábiles',
 'S/ 9.00',
 'tramite.unsaac.edu.pe'),

(14,'Constancia de buena conducta',
 'Documento expedido por el Decano que acredita el buen comportamiento del estudiante regular en su labor estudiantil.',
 '1. Solicitud al Rector. 2. Ficha de seguimiento académico. 3. Pago por derechos.',
 '1 día hábil',
 'S/ 15.00',
 'Decanato de Facultad'),

(15,'Rectificación de nombre',
 'Corrección del nombre del estudiante por mandato judicial, notarial o administrativo.',
 '1. Solicitud al Rector con documentación que acredite el error: resolución judicial, documento notarial o evidencia de error administrativo. 2. Número de DNI. 3. Pago.',
 '5 días hábiles',
 'S/ 96.00',
 'tramite.unsaac.edu.pe'),

-- ======================== SERVICIOS EXCLUSIVOS (Sección 2) ========================

(16,'Certificado de estudios',
 'Documento oficial con promedios finales de las asignaturas del Plan de Estudios cursadas por el estudiante, expedido por semestre académico.',
 '1. Solicitud al Rector. 2. Fotografías tamaño carné a color según número de hojas. 3. Ficha de seguimiento académico. 4. Pago por certificado.',
 '5 a 7 días hábiles',
 'S/ 9.00 por semestre (desde 1985); S/ 21.00 (1974–1984); S/ 76.00 por año (antes de 1974)',
 'pladdes.unsaac.edu.pe'),

(17,'Constancia de estudios',
 'Documento que acredita la condición de estudiante matriculado.',
 '1. Solicitud al Rector. 2. Constancia de matrícula. 3. Pago por derechos.',
 '1 día hábil',
 'S/ 15.00',
 'pladdes.unsaac.edu.pe'),

(18,'Constancia de créditos acumulados o de egresado',
 'Documento que acredita la situación académica del estudiante o su condición de egresado.',
 '1. Solicitud al Rector. 2. Ficha de seguimiento académico. 3. Pago por constancia.',
 '1 día hábil',
 'S/ 12.00',
 'pladdes.unsaac.edu.pe'),

(19,'Constancia de tercio, quinto y décimo superior',
 'Certifica el orden de mérito basado en el promedio aritmético del semestre académico concluido.',
 '1. Solicitud al Rector. 2. Informe del Centro de Cómputo del cuadro de promedio aritmético. 3. Pago por constancia.',
 '1 día hábil',
 'S/ 15.00',
 'Decanato de Facultad'),

(20,'Constancia de no ser deudor',
 'Acredita que el estudiante no tiene deudas ni bienes pendientes de devolución con la UNSAAC.',
 '1. Solicitud al Rector. 2. Pago por constancia.',
 '1 día hábil',
 'S/ 15.00',
 'pladdes.unsaac.edu.pe'),

(21,'Constancia de ingreso',
 'Documento que acredita que el postulante alcanzó vacante en una Escuela Profesional.',
 '1. Solicitud al Rector. 2. Pago por constancia.',
 '1 día hábil',
 'S/ 15.00',
 'Oficina de Admisión'),

(22,'Ficha de seguimiento académico',
 'Reporte con código, categoría de asignaturas y notas cursadas por semestre, según el Plan de Estudios. Emitido por el Centro de Cómputo.',
 '1. Pago por derechos de ficha.',
 '1 día hábil',
 'S/ 4.00',
 'ccomputo.unsaac.edu.pe'),

(23,'Carné universitario',
 'Documento que acredita la condición de estudiante de la UNSAAC. Emitido por Registro y Servicios Académicos.',
 '1. Tener matrícula vigente. 2. Imagen digitalizada a color (fondo blanco, 240×288px, 300dpi, .jpg). 3. Enviar imagen y recibo de pago desde correo institucional a servicios.academicos@unsaac.edu.pe. 4. Pago.',
 '30 días hábiles',
 'S/ 12.60',
 'servicios.academicos@unsaac.edu.pe'),

(24,'Duplicado de carné universitario',
 'Duplicado por pérdida o sustracción del carné universitario.',
 '1. Solicitud al Rector. 2. Declaración jurada por pérdida o sustracción. 3. Pago por duplicado.',
 '30 días hábiles',
 'S/ 12.60',
 'servicios.academicos@unsaac.edu.pe'),

(25,'Carné de biblioteca',
 'Carné de lector de biblioteca para alumnos regulares de la UNSAAC.',
 '1. Pago por derechos de carné.',
 '1 día hábil',
 'S/ 6.00',
 'Biblioteca Central'),

(26,'Duplicado de carné de biblioteca',
 'Duplicado por pérdida o deterioro del carné de lector.',
 '1. Formato de Biblioteca Central llenado. 2. Declaración jurada de pérdida. 3. Pago.',
 '1 día hábil',
 'S/ 8.00',
 'Biblioteca Central'),

(27,'Copia visada de sílabos',
 'Expedición de sílabo visado para fines de convalidación u otros trámites.',
 '1. Solicitud al Rector. 2. Pago por sílabos.',
 '1 día hábil',
 'S/ 5.00',
 'pladdes.unsaac.edu.pe'),

(28,'Duplicado de constancia de matrícula',
 'Duplicado del documento que acredita la condición de estudiante matriculado.',
 '1. Solicitud al Rector. 2. Pago por duplicado.',
 '1 día hábil',
 'S/ 8.00',
 'ccomputo.unsaac.edu.pe'),

(29,'Duplicado de constancia de notas',
 'Duplicado del documento con calificaciones obtenidas.',
 '1. Solicitud al Rector. 2. Pago por duplicado.',
 '1 día hábil',
 'S/ 8.00',
 'ccomputo.unsaac.edu.pe'),

(30,'Carta de presentación del Decano',
 'Carta de presentación para realizar prácticas preprofesionales, expedida por el Decano de la Facultad.',
 '1. Solicitud al Rector. 2. Ficha de seguimiento académico. 3. Pago por derechos.',
 '1 día hábil',
 'S/ 10.00',
 'Decanato de Facultad'),

(31,'Rectificación de nombre en base de datos del Centro de Cómputo',
 'Corrección de nombre errado en la base de datos del Centro de Cómputo, de acuerdo al DNI.',
 '1. Solicitud al Rector. 2. Número de DNI. 3. Pago.',
 '5 días hábiles',
 'S/ 12.00',
 'ccomputo.unsaac.edu.pe'),

-- ======================== GRADOS Y TÍTULOS ========================

(32,'Calificación de expediente para Bachiller',
 'El egresado solicita ser declarado Apto para optar el Grado de Bachiller otorgado por la UNSAAC a nombre de la Nación.',
 '1. Solicitud al Rector. 2. Copia simple de certificado de idioma extranjero y de computación básica. 3. Ficha de seguimiento académico con conformidad (haber cumplido todos los créditos del Plan Curricular). 4. Declaración jurada de homologación/convalidación (si aplica). 5. Dos fotografías a color tamaño carné 4×3cm (terno oscuro, camisa blanca, fondo blanco) + fotos para certificados. 6. Pago por bachillerato y rotulado.',
 '20 días hábiles',
 'S/ 415.00',
 'tramite.unsaac.edu.pe'),

(33,'Título Profesional — Modalidad Sustentación de Tesis',
 'Calificación de expediente para optar al título mediante sustentación de tesis (investigación propia de la especialidad en acto público).',
 '1. Solicitud al Rector. 2. Copia simple del diploma de Bachiller. 3. Declaración jurada de no tener antecedentes penales ni judiciales. 4. Pago por derechos.',
 '10 días hábiles',
 'S/ 434.00',
 'tramite.unsaac.edu.pe'),

(34,'Título Profesional — Modalidad Suficiencia Profesional',
 'Calificación de expediente para optar al título mediante documento escrito y presentación oral o evaluación de materias de formación profesional en acto público.',
 '1. Solicitud al Rector. 2. Copia simple del diploma de Bachiller. 3. Declaración jurada de no tener antecedentes penales ni judiciales. 4. Pago por derechos.',
 '10 días hábiles',
 'S/ 540.00',
 'tramite.unsaac.edu.pe'),

(35,'Título Profesional — Modalidad Servicios a Nivel Profesional',
 'Solo para quienes ingresaron antes del 10 de junio de 2014. Requiere mínimo 3 años de experiencia profesional en labores de la especialidad tras egresar.',
 '1. Solicitud al Rector. 2. Copia simple del diploma de Bachiller. 3. Informe técnico de labor realizada (2 ejemplares). 4. Tres años consecutivos de experiencia (boletas de pago). 5. Declaración jurada de no tener antecedentes penales ni judiciales. 6. Pago.',
 '30 días hábiles',
 'S/ 626.00',
 'tramite.unsaac.edu.pe'),

(36,'Modificación del plan de tesis',
 'Para modificar el plan de tesis cuando durante el desarrollo de la investigación surgen situaciones que exigen cambios.',
 '1. Solicitud al Rector. 2. Informe del asesor de tesis. 3. Dos ejemplares del plan modificado. 4. Copia de la resolución anterior. 5. Pago.',
 '5 días hábiles',
 'S/ 30.00',
 'tramite.unsaac.edu.pe'),

(37,'Aprobación de dictamen de tesis',
 'Declarar suficiente la tesis o trabajo de investigación para su exposición oral.',
 '1. Solicitud al Rector. 2. Dictamen final de tesis. 3. Pago.',
 '5 días hábiles',
 'S/ 25.00',
 'tramite.unsaac.edu.pe'),

(38,'Inscripción de tema de tesis y nombramiento de asesor',
 'El aspirante al título inscribe su Plan de Tesis y solicita nombramiento de asesor (profesor especialista). Para estudiantes con más del 80% de créditos aprobados del Plan Curricular.',
 '1. Solicitud al Rector. 2. Dos ejemplares del plan de tesis refrendado por el asesor. 3. Carta de aceptación del asesor. 4. Ficha de seguimiento académico. 5. Pago.',
 '5 días hábiles',
 'S/ 30.00',
 'tramite.unsaac.edu.pe'),

(39,'Nombramiento de dictaminadores de tesis',
 'La tesis es evaluada por dictaminadores nombrados por Resolución del Decano, quienes emiten informe sobre la suficiencia del trabajo.',
 '1. Solicitud al Rector. 2. Informe del asesor. 3. Resolución de aprobación de expediente de título. 4. Dos ejemplares de tesis anillado. 5. Certificado de originalidad firmado por el asesor (sistema antiplagio). 6. Pago.',
 '5 días hábiles',
 'S/ 28.00',
 'tramite.unsaac.edu.pe'),

(40,'Fecha, hora y lugar de sustentación',
 'Determinación de fecha, hora y lugar para sustentación de tesis, examen de suficiencia profesional o examen por servicios a nivel profesional.',
 '1. Solicitud al Rector. 2. Copia de la resolución de aprobación de dictamen. 3. Cinco ejemplares de tesis o dos ejemplares de informe técnico según modalidad. 4. Pago.',
 '5 días hábiles',
 'S/ 30.00',
 'Decanato de Facultad'),

(41,'Rotulado de diploma de título profesional',
 'La UNSAAC emite el diploma de título profesional con código de barras y número correlativo para inscripción en SUNEDU. Requiere que el titulado entregue ejemplares de tesis empastadas.',
 '1. Solicitud al Rector. 2. Dos fotografías a color tamaño carné 4×3cm (terno oscuro, camisa blanca). 3. Dos ejemplares de tesis empastada (biblioteca de la Escuela + repositorio institucional). 4. Constancia de depósito del repositorio institucional (previo CD + tesis empastada). 5. Pago por rotulado.',
 '5 días hábiles',
 'S/ 102.00',
 'tramite.unsaac.edu.pe'),

(42,'Acto de juramentación o promesa de honor',
 'Acto académico público de juramentación, promesa de honor o colación para el nuevo profesional (pregrado y posgrado).',
 '1. Solicitud al Rector. 2. Pago por derechos de medalla, solapera, alquiler de toga y birrete.',
 '1 día hábil',
 'S/ 75.00',
 'tramite.unsaac.edu.pe');

-- ---------- TITULACION_MODALIDADES (NUEVA) ----------

CREATE TABLE titulacion_modalidades (
  id INT PRIMARY KEY,
  modalidad VARCHAR(60) NOT NULL,
  descripcion TEXT NOT NULL,
  costo_soles DECIMAL(7,2),
  plazo_dias_habiles INT,
  requisitos TEXT
);

INSERT INTO titulacion_modalidades VALUES
(1,'Bachiller (Calificación de Expediente)','Calificación de expediente para optar el Grado Académico de Bachiller y rotulado de diploma. Requisito previo: ser egresado del plan de estudios.',415.00,20,'Solicitud al Rector; certificado de Idioma Extranjero y Computación Básica; Ficha de Seguimiento Académico; declaración jurada de convalidaciones si aplica; fotografías; pago de derechos.'),
(2,'Sustentación de Tesis','Trabajo de investigación propio de la especialidad, sustentado en acto público ante jurado.',434.00,10,'Bachiller; plan de tesis aprobado; dictamen aprobado; pago de derechos.'),
(3,'Trabajo de Suficiencia Profesional','Documento escrito y sustentación oral sobre análisis o propuesta de mejora/innovación basada en aprendizajes adquiridos.',540.00,10,'Bachiller; declaración jurada de no tener antecedentes penales ni judiciales; pago de derechos.'),
(4,'Servicios a Nivel Profesional','Modalidad por experiencia profesional. Solo aplica a quienes ingresaron antes del 10 de junio de 2014.',626.00,30,'Bachiller; ingreso antes del 10 de junio de 2014; mínimo 3 años de experiencia profesional tras egresar; informe técnico; sustentación oral.'),
(5,'Aprobación de Dictamen de Tesis','Trámite de aprobación del dictamen de la tesis emitido por los dictaminadores.',25.00,5,'Plan de tesis aprobado; informe de dictaminadores.'),
(6,'Modificación del Plan de Tesis','Trámite para modificar un plan de tesis ya aprobado.',30.00,5,'Solicitud justificada; visto bueno del director de tesis.');

-- ---------- SERVICIOS_UNIVERSITARIOS (NUEVA — absorbe recursos) ----------

CREATE TABLE servicios_universitarios (
  id INT PRIMARY KEY,
  categoria VARCHAR(30) NOT NULL,
  nombre VARCHAR(120) NOT NULL,
  descripcion TEXT NOT NULL,
  ubicacion VARCHAR(200),
  enlace VARCHAR(300)
);

INSERT INTO servicios_universitarios VALUES
-- salud
(1,'salud','Centro Universitario de Salud','Atención médica general, odontológica y primeros auxilios gratuitos para estudiantes matriculados.','Ciudad Universitaria de Perayoc','https://www.unsaac.edu.pe/centro-universitario-de-salud/'),
(2,'salud','Atención Psicológica','Consultas individuales y talleres grupales gratuitos; gestión a través de Bienestar Universitario.','Bienestar Universitario','https://www.unsaac.edu.pe/'),
-- comedor
(3,'comedor','Comedor Universitario','Servicio de alimentación subsidiada para estudiantes con carné universitario. Becas de alimentación disponibles vía Bienestar.','Campus Universitario, zona de servicios','https://web.facebook.com/p/Comité-de-Comensales-UNSAAC-2024-2025-100069221791444/'),
-- becas
(4,'becas','Becas UNSAAC','Información general sobre becas y convocatorias vigentes en la universidad.','Bienestar Universitario','https://www.unsaac.edu.pe/becas/'),
(5,'becas','Becas de cooperación internacional','Becas gestionadas por la Oficina de Cooperación Técnica Internacional (OCTI).','OCTI','https://octi.unsaac.edu.pe/redes-de-cooperacion/becas/'),
(6,'becas','Beca posgrado UNISC Brasil','Convocatoria de becas de posgrado en la Universidade de Santa Cruz do Sul (Brasil) gestionada por OCTI.','OCTI','https://octi.unsaac.edu.pe/urgente-convocatoria-de-becas-de-posgrado-unisc-brasil/'),
(7,'becas','PRONABEC (Beca 18 y otros)','Becas del Estado peruano: Beca 18, Beca Permanencia y otras modalidades.','PRONABEC / Bienestar','https://www.pronabec.gob.pe'),
-- deportes
(8,'deportes','Actividades Deportivas','Talleres y disciplinas: fútbol, vóleibol, natación, atletismo, artes marciales, entre otros.','Complejo deportivo UNSAAC','https://web.facebook.com/UNSAACDEPORTES/'),
(9,'deportes','Ingreso por deportista calificado','Modalidad de admisión para deportistas calificados; vacantes específicas en el proceso de admisión.','Oficina de Admisión','https://admision.unsaac.edu.pe/modalidad-deportistas-calificados/'),
-- vivienda
(10,'vivienda','Vivienda estudiantil','Información sobre alojamiento y apoyo a estudiantes; canal informativo institucional.','UNSAAC (canal informativo)','https://web.facebook.com/unsaac.edu.pe/'),
-- movilidad
(11,'movilidad','Programa de Movilidad RPU','Programa de movilidad académica entre universidades peruanas. La UNSAAC participa como miembro de la Red Peruana de Universidades.','Oficina de Relaciones Internacionales','https://rpu.edu.pe/programa_movilidad/universidad-nacional-san-antonio-abad-del-cusco/'),
-- convenios
(12,'convenios','Convenios institucionales (OCTI)','Convenios de cooperación con instituciones nacionales e internacionales gestionados por OCTI.','OCTI','https://octi.unsaac.edu.pe/redes-de-cooperacion/'),
-- recursos (absorbidos)
(13,'recursos','Laboratorios de Cómputo','Laboratorios especializados para programación, redes, bases de datos e inteligencia artificial.','Facultad FIEEIM','in.unsaac.edu.pe'),
(14,'recursos','Licencias de Software académico','Herramientas y software académico disponible para estudiantes de la EPIIS.','EPIIS / Centro de Cómputo','ccomputo.unsaac.edu.pe'),
(15,'recursos','Biblioteca Virtual','Acceso a libros digitales, tesis y artículos científicos para la comunidad universitaria.','UNSAAC','https://www.unsaac.edu.pe/'),
(16,'recursos','Plataformas académicas','Sistema académico, correo institucional y plataformas virtuales de la UNSAAC.','UNSAAC','ccomputo.unsaac.edu.pe'),
(17,'recursos','Repositorio Digital Institucional','Repositorio de tesis y trabajos de investigación de acceso abierto.','UNSAAC','https://repositorio.unsaac.edu.pe/handle/20.500.12918/85'),
(18,'recursos','Internet y correo institucional','Servicios de conectividad del campus y correo institucional como medio oficial de comunicación.','Campus','ccomputo.unsaac.edu.pe'),
-- bienestar y defensoría
(19,'bienestar','Bienestar Universitario','Servicios de salud, psicología, asistencia social y apoyo socioeconómico.','Edificio Central, 1er piso','https://www.unsaac.edu.pe/'),
(20,'bienestar','Defensoría Universitaria','Atiende denuncias de acoso, discriminación y vulneración de derechos. Confidencialidad garantizada.','Rectorado, 2do piso','https://www.unsaac.edu.pe/');

-- ---------- TUTORIAS_INFO (NUEVA — pasa el texto fijo a BD) ----------

CREATE TABLE tutorias_info (
  id INT PRIMARY KEY,
  subtema VARCHAR(30) NOT NULL,
  contenido TEXT NOT NULL
);

INSERT INTO tutorias_info VALUES
(1,'proceso','La tutoría es un proceso permanente de acompañamiento al estudiante en sus dimensiones académica, psicosocial, cognitiva y afectiva, brindado por un docente tutor. Es obligatoria para estudiantes con matrícula condicionada. Cada tutor elabora un expediente del tutorado con diagnóstico inicial, estrategias de atención, seguimiento del rendimiento y un informe semestral al Director de Escuela.'),
(2,'tipos','El reglamento contempla tutoría personalizada (individual) y tutoría grupal. Las modalidades se realizan mediante entrevistas programadas, comentarios de trabajos u otras actividades definidas por el tutor.'),
(3,'horarios','La frecuencia mínima la define el Comité Tutorial de la Escuela. Se garantizan al menos tres momentos clave por semestre: al inicio del curso, después del primer parcial y una semana antes de finalizar el semestre. El tutor puede convocar más reuniones y el estudiante también puede solicitarlas.'),
(4,'tutor_asignado','Se asigna un tutor desde el primer semestre hasta la culminación de la carrera. Cada tutor atiende un máximo de 25 estudiantes (puede variar por Escuela). El estudiante puede solicitar cambio de tutor de forma razonada ante el Comité Tutorial; resuelve el Comité y en última instancia el Vicerrectorado Académico.'),
(5,'problemas','El reglamento no define un procedimiento formal de reprogramación. Si el estudiante falta reiteradamente a clases, el tutor debe contactarlo para indagar las razones. Para temas de mayor gravedad se deriva a la Unidad de Bienestar Universitario.');

"""

print("Volcado SQL embebido:", len(SQL_DUMP), "caracteres")

Volcado SQL embebido: 79427 caracteres


### 2.1 Carga del volcado en SQLite


- `_limpiar_sql()` → quita comentarios y líneas vacías del texto SQL.
- `_dividir_sentencias()` → corta el texto en instrucciones individuales (cada una termina en `;`), con cuidado de no cortar mal si hay un `;` dentro de un texto entre comillas.
- `construir_base_datos()` → crea la base de datos SQLite **en memoria** (no se guarda en disco, vive solo mientras corre el notebook) y ejecuta todas las instrucciones para crear las 10 tablas y llenarlas con datos.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 3: FUNCIONES DE CARGA Y CONSTRUCCIÓN DE LA BASE DE DATOS
# ─────────────────────────────────────────────────────────────────────
# Define tres funciones auxiliares:
#   1. _limpiar_sql()         → elimina comentarios y líneas vacías del dump
#   2. _dividir_sentencias()  → separa sentencias SQL respetando comillas
#                               (para no cortar textos que contienen ';')
#   3. construir_base_datos() → crea una BD SQLite en memoria, ejecuta los
#                               CREATE TABLE e INSERT del volcado embebido.
#                               Usa SQLite en vez de MySQL para que funcione
#                               en Colab sin servidor; el diseño es portable.
# Al final verifica que las  tablas estén pobladas correctamente.
# ═══════════════════════════════════════════════════════════════════════
def _limpiar_sql(sql: str) -> str:
    lineas = []
    for linea in sql.splitlines():
        s = linea.strip()
        if not s or s.startswith("--") or s.startswith("/*"):
            continue
        lineas.append(linea)
    return "\n".join(lineas)


def _dividir_sentencias(sql: str) -> List[str]:
    sentencias, buffer, en_cadena = [], [], False
    for ch in sql:
        buffer.append(ch)
        if ch == "'":
            en_cadena = not en_cadena
        elif ch == ";" and not en_cadena:
            sentencias.append("".join(buffer).strip()); buffer = []
    resto = "".join(buffer).strip()
    if resto:
        sentencias.append(resto)
    return sentencias


def construir_base_datos() -> sqlite3.Connection:
    sql = _limpiar_sql(SQL_DUMP).replace('\\"', '"')
    conn = sqlite3.connect(":memory:")
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    n_create = n_insert = 0
    for sent in _dividir_sentencias(sql):
        head = sent.lstrip().upper()
        if head.startswith("CREATE TABLE"):
            cur.execute(sent); n_create += 1
        elif head.startswith("INSERT INTO"):
            cur.execute(sent); n_insert += 1
    conn.commit()
    print(f"Repositorio: {n_create} tablas creadas, {n_insert} sentencias INSERT.")
    return conn


_c = construir_base_datos()
for t in ["cursos","docentes","horarios_cursos","informacion_general","ingresantes",
          "temas_tesis","tramites","titulacion_modalidades","servicios_universitarios",
          "tutorias_info"]:
    n = _c.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<28}: {n}")
_c.close()


Repositorio: 10 tablas creadas, 31 sentencias INSERT.
  cursos                      : 164
  docentes                    : 33
  horarios_cursos             : 346
  informacion_general         : 12
  ingresantes                 : 4
  temas_tesis                 : 4
  tramites                    : 42
  titulacion_modalidades      : 6
  servicios_universitarios    : 20
  tutorias_info               : 5


### 2.2 Clase `BaseDatos`



- `consultar(sql, params)` → hace una pregunta tipo SELECT y devuelve **todas** las filas que encontró.
- `consultar_uno(sql, params)` → igual, pero devuelve solo **la primera** fila (o nada si no encontró).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 4: CLASE BaseDatos (PATRÓN DE ACCESO A DATOS)
# ─────────────────────────────────────────────────────────────────────
# Encapsula la conexión SQLite y ofrece métodos genéricos de consulta.
# Es la ÚNICA clase que conoce el motor de base de datos; todos los
# módulos del chatbot dependen de su interfaz, no del motor directamente.
# Métodos:
#   - consultar(sql, params)     → ejecuta SELECT, devuelve lista de filas
#   - consultar_uno(sql, params) → devuelve la primera fila o None
#   - cerrar()                   → cierra la conexión
# Gracias a esta capa, migrar de SQLite a MySQL solo requiere cambiar
# esta clase, sin tocar el resto del sistema.
# ═══════════════════════════════════════════════════════════════════════
class BaseDatos:
    """Capa de acceso al repositorio. Única clase que conoce el motor de BD."""
    def __init__(self, conexion: sqlite3.Connection):
        self._conn = conexion
    def consultar(self, sql: str, params: tuple = ()) -> List[sqlite3.Row]:
        cur = self._conn.cursor(); cur.execute(sql, params); return cur.fetchall()
    def consultar_uno(self, sql: str, params: tuple = ()) -> Optional[sqlite3.Row]:
        f = self.consultar(sql, params); return f[0] if f else None
    def cerrar(self) -> None: self._conn.close()

print("Clase BaseDatos definida.")


Clase BaseDatos definida.


## 3. Corpus de intenciones



In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 5: CORPUS DE INTENCIONES (CONOCIMIENTO CONVERSACIONAL)
# ─────────────────────────────────────────────────────────────────────
#   saludo, informacion_general, docentes, cursos, tramites, tutorias,
#   practicas, temas_tesis, titulacion, ingresantes, servicios,
#   contacto, elegibilidad, derivar_a_escuela, despedida
#
# Cada pregunta es un ejemplo de cómo un estudiante formularía una
# consulta. El clasificador aprende de estos patrones para identificar
# la intención detrás de preguntas nuevas que nunca ha visto.
# La intención "derivar_a_escuela" es especial: agrupa preguntas cuya
# respuesta no se debe inventar (datos no confirmados públicamente).
# ═══════════════════════════════════════════════════════════════════════
CORPUS = {
   "saludo": [
    "hola",
    "buenos días",
    "buenas tardes",
    "buenas noches",
    "qué tal",
    "hola buen día",
    "saludos",
    "hola chatbot",
    "buenas",
    "hola cómo estás",

    # --- Variantes cortas / coloquiales ---
    "hey",
    "ey",
    "holaa",
    "holaaa",
    "hola!",
    "hola que tal",
    "qué onda",
    "qué hay",
    "oe",
    "oye",
    "ola",
    "holis",
    "buenass",
    "buen día",
    "buenas noche",

    # --- Con nombre del bot / contexto académico ---
    "hola asistente",
    "hola bot",
    "hola epiis bot",
    "hola, asistente virtual",
    "buenas, necesito ayuda",
    "hola, ¿me puedes ayudar?",
    "hola, tengo una consulta",
    "buenas, quisiera preguntarte algo",

    # --- Formales ---
    "muy buenos días",
    "muy buenas tardes",
    "buenos días, ¿cómo estás?",
    "buenas tardes, disculpa la molestia",
    "estimado asistente, buenos días",
    "un gusto saludarte",
    "reciba un cordial saludo",

    # --- Con pregunta de cortesía incluida ---
    "hola, ¿qué tal estás?",
    "hola, ¿todo bien?",
    "buenas, ¿cómo va todo?",
    "hola, ¿cómo has estado?",
    "qué tal, ¿todo en orden?",
    "hola, ¿qué cuentas?",

    # --- Según hora del día (variantes) ---
    "buen día",
    "muy buen día",
    "feliz tarde",
    "buenas noches, disculpa la hora",
    "hola, perdón por escribir tan tarde",
    "buenos días, espero estés bien",

    # --- Con errores ortográficos comunes ---
    "ola q tal",
    "buenoss dias",
    "wenas",
    "k tal",
    "ola komo estas",
    "buenass tardes",

    # --- Con emojis ---
    "hola 👋",
    "buenos días ☀️",
    "qué tal 😊",
    "hola!! 🙌",

    # --- Saludo + presentación ---
    "hola, soy estudiante de la escuela",
    "hola, soy nuevo aquí",
    "buenas, soy alumno de informática",
    "hola, recién estoy conociendo el chatbot",

    # --- Despedida usada como saludo informal (regional) ---
    "qué más",
    "qué hubo",
    "todo bien por aquí",

    # --- Muy breves / interjecciones ---
    "hi",
    "hello",
    "hey bot",
],
   "informacion_general": [
    # --- Misión / Visión ---
    "¿Cuál es la misión de la Escuela?",
    "misión de la EPIIS",
    "¿para qué existe la Escuela Profesional?",
    "propósito institucional de la carrera",
    "¿cuál es la razón de ser de la Escuela?",
    "¿hacia dónde quiere llegar la Escuela?",
    "¿qué visión a futuro tiene la EPIIS?",
    "visión institucional de la Escuela",
    "¿qué aspira a ser la Escuela en el futuro?",
    # --- más variantes de misión / visión / propósito institucional ---

    # --- Misión ---
    "¿qué busca lograr la Escuela con su misión?",
    "¿en qué consiste la misión institucional de la EPIIS?",
    "explícame la misión de la Escuela Profesional",
    "¿cuál es el compromiso de la Escuela con la sociedad?",
    "¿qué papel cumple la Escuela según su misión?",
    "¿qué dice la misión sobre la formación de profesionales?",
    "¿la misión menciona algo sobre investigación?",
    "¿cómo describe la Escuela su rol formativo?",
    "fundamento de la misión de la carrera",
    "¿a quién está dirigida la misión institucional?",

    # --- Visión ---
    "¿qué quiere lograr la Escuela a largo plazo?",
    "¿cómo se proyecta la EPIIS en los próximos años?",
    "metas futuras de la Escuela Profesional",
    "¿la visión habla de ser líder en la región?",
    "¿qué tan ambiciosa es la visión de la Escuela?",
    "¿la visión incluye estándares internacionales?",
    "¿hacia qué horizonte apunta la visión institucional?",
    "¿qué imagen futura busca proyectar la Escuela?",
    "describe la visión de la EPIIS",
    "¿la visión está alineada con la acreditación?",

    # --- Propósito / razón de ser (variantes adicionales) ---
    "¿por qué existe esta carrera en la UNSAAC?",
    "¿cuál es el sentido de la Escuela Profesional?",
    "¿qué necesidad busca cubrir la EPIIS?",
    "objetivo fundamental de la Escuela",
    "¿qué problema social resuelve esta carrera?",
    "finalidad de la Escuela de Ingeniería Informática",

    # --- Comparativas misión vs visión ---
    "¿cuál es la diferencia entre misión y visión de la Escuela?",
    "compara la misión con la visión institucional",
    "¿la misión y la visión están relacionadas?",
    "explícame la diferencia entre propósito actual y proyección futura de la Escuela",

    # --- Coloquial / informal ---
    "cual es la mision de la escuela",
    "para que sirve esta escuela",
    "que onda con la mision de la epiis",
    "cual es la meta a futuro de la escuela",
    "que busca la escuela como institucion",

    # --- Con errores ortográficos ---
    "mision dela escuela",
    "vision dela epis",
    "kual es la mision",
    "porpocito dela carrera",
    "vicion institucional",

    # --- Muy cortas ---
    "misión epiis",
    "visión epiis",
    "propósito escuela",
    "razón de ser escuela",
    "meta futura escuela",

    # --- Con saludo ---
    "hola, ¿cuál es la misión institucional?",
    "buenos días, quisiera saber la visión de la Escuela",
    "disculpa, ¿cuál es el propósito de la carrera?",


    # --- Historia / fundación ---
    "¿en qué año se fundó la Escuela?",
    "origen de la Escuela de Informática",
    "¿cuándo se creó la carrera de Ingeniería Informática?",
    "antecedentes históricos de la EPIIS",
    "¿cómo nació la Escuela Profesional?",
    "evolución histórica de la carrera",
    "¿la Escuela existe desde hace mucho tiempo?",
    "trayectoria de la Escuela de Informática en el Cusco",
   # --- más variantes sobre historia / fundación de la Escuela ---

    "¿quién fundó la Escuela de Informática?",
    "¿en qué fecha exacta se creó la EPIIS?",
    "¿cuántos años tiene la Escuela funcionando?",
    "historia de la creación de la carrera",
    "¿la Escuela fue creada por resolución del Consejo Universitario?",
    "¿bajo qué facultad nació originalmente la Escuela?",
    "¿la carrera siempre se llamó Ingeniería Informática y de Sistemas?",
    "¿la Escuela cambió de nombre con el tiempo?",
    "¿cómo ha evolucionado el plan de estudios desde su creación?",
    "primeros años de la Escuela Profesional",
    "¿quiénes fueron los primeros docentes de la Escuela?",
    "¿cuál fue la primera promoción de egresados?",
    "hechos importantes en la historia de la EPIIS",
    "¿la Escuela surgió de otra carrera ya existente?",
    "¿la informática se enseñaba antes en otra facultad de la UNSAAC?",
    "reseña histórica de la carrera",
    "línea de tiempo de la Escuela de Informática",
    "¿cuántas décadas lleva la EPIIS formando profesionales?",
    "¿la Escuela ha tenido varias sedes a lo largo de su historia?",
    "contexto en el que se fundó la Escuela",
    "¿por qué se creó esta carrera en el Cusco?",
    "antigüedad de la Escuela Profesional de Informática",
    "¿es una de las escuelas más antiguas de la UNSAAC?",
    "hitos importantes de la EPIIS",

    # --- Coloquial / informal ---
    "desde cuando existe esta escuela",
    "hace cuantos años se creo la carrera",
    "como empezo todo con la escuela de informatica",
    "quien armo esta escuela",
    "cuanto tiempo tiene la epiis funcionando",

    # --- Con errores ortográficos ---
    "desde kuando existe la escuela",
    "kien fundo la epiis",
    "istoria dela escuela",
    "haze cuantos años se creo",
    "orijen dela carrera",

    # --- Muy cortas ---
    "historia escuela",
    "fundación epiis",
    "origen epiis",
    "año de creación",
    "cuándo se creó",

    # --- Con saludo ---
    "hola, ¿cuándo se fundó la Escuela?",
    "buenos días, quisiera saber la historia de la EPIIS",
    "disculpa, ¿en qué año nació esta carrera?",


    # --- Duración / modalidad ---
    "¿cuántos semestres dura la carrera?",
    "¿la carrera es presencial o virtual?",
    "duración total de los estudios",
    "¿cuántos ciclos tiene el plan de estudios?",
    "tiempo estimado para egresar",
    "¿la modalidad de estudios es semestral o anual?",
   # --- más variantes sobre duración / modalidad de estudios ---

    "¿cuántos años dura la carrera de Informática?",
    "¿en cuánto tiempo puedo terminar la carrera?",
    "¿la carrera dura 5 años?",
    "¿cuántos ciclos académicos tengo que aprobar para egresar?",
    "tiempo mínimo para titularse",
    "¿puedo terminar la carrera en menos de 10 semestres?",
    "¿qué pasa si me demoro más de 10 ciclos?",
    "duración oficial del plan de estudios",
    "¿la carrera tiene 10 semestres en total?",
    "¿cuántos créditos necesito acumular para terminar?",
    "¿la malla está dividida en semestres o trimestres?",
    "¿la Escuela ofrece clases presenciales?",
    "¿hay cursos virtuales en la carrera?",
    "¿la carrera es totalmente presencial?",
    "¿existe una modalidad semipresencial?",
    "¿qué cursos se dictan de forma virtual?",
    "¿las clases son en el campus o a distancia?",
    "¿la modalidad cambió después de la pandemia?",
    "¿se puede estudiar la carrera de forma remota?",
    "¿cuántas horas de estudio se requieren por semestre?",
    "¿el tiempo de carrera incluye las prácticas pre profesionales?",
    "duración de la carrera contando el último ciclo de prácticas",
    "¿cuánto dura un semestre académico en la EPIIS?",
    "¿la carrera funciona con el sistema de créditos semestral?",

    # --- Coloquial / informal ---
    "cuanto dura la carrera",
    "es presencial o virtual la carrera",
    "en cuantos años salgo de la carrera",
    "cuantos ciclos son en total",
    "la carrera es semestral o anual",
    "cuanto se tarda uno en egresar normalmente",

    # --- Con errores ortográficos ---
    "kuanto dura la carrera",
    "es presencial o birtual",
    "duracion total dela carera",
    "kuantos siclos tiene el plan de estudios",
    "modalidad semestral o anual",

    # --- Muy cortas ---
    "duración carrera",
    "modalidad estudios",
    "cuántos semestres",
    "presencial o virtual",
    "tiempo para egresar",

    # --- Con saludo ---
    "hola, ¿cuántos semestres dura la carrera?",
    "buenos días, ¿la carrera es presencial?",
    "disculpa, ¿cuánto tiempo toma egresar?",



    # --- Director/a, autoridades, departamento ---
    "¿quién es la actual directora de Escuela?",
    "nombre de la directora de la EPIIS",
    "¿quién está a cargo de la Escuela Profesional?",
    "¿quién es el director del Departamento Académico de Informática y Sistemas?",
    "responsable del Departamento Académico",
    "¿qué cargo tiene Lino Flores Pacheco?",
    "¿qué función cumple el Departamento Académico?",
    "diferencia entre Escuela Profesional y Departamento Académico",
    "¿quién preside la Escuela actualmente?",
    "datos de contacto de la directora",
    "¿hay un consejo de Escuela?",
    "estructura de autoridades de la EPIIS",
   # --- más variantes sobre director/a, autoridades y departamento académico ---

    "¿quién es la directora de la Escuela Profesional de Informática?",
    "¿cómo se llama la directora actual?",
    "información sobre Yeshica Ormeño Ayala",
    "¿desde cuándo es directora Yeshica Ormeño?",
    "¿cuánto dura el cargo de director de Escuela?",
    "¿cómo se elige al director de la Escuela?",
    "¿la directora también es docente?",
    "¿qué funciones tiene la directora de Escuela?",
    "¿quién toma las decisiones académicas en la EPIIS?",
    "¿quién firma los documentos oficiales de la Escuela?",
    "¿a quién le reporta la directora de Escuela?",
    "¿quién es la máxima autoridad de la EPIIS?",
    "¿quién dirige el Departamento Académico de Informática y de Sistemas?",
    "¿qué cargo ocupa Lino Prisciliano Flores Pacheco?",
    "¿el director del departamento académico es el mismo que el de la Escuela?",
    "¿qué diferencia hay entre el jefe de departamento y el director de Escuela?",
    "¿qué hace exactamente el Departamento Académico?",
    "¿el Departamento Académico administra a los docentes?",
    "¿quién supervisa la carga académica de los profesores?",
    "¿qué autoridades conforman la Escuela Profesional?",
    "¿quiénes integran el consejo de Escuela?",
    "¿el consejo de Escuela toma decisiones sobre la malla curricular?",
    "organigrama de la Escuela Profesional",
    "¿cómo está organizada administrativamente la EPIIS?",
    "¿hay un subdirector o solo una directora?",
    "¿cómo contacto a la dirección de la Escuela?",
    "correo electrónico de la directora",
    "horario de atención de la dirección de Escuela",
    "¿dónde queda la oficina de la dirección?",
    "¿la directora atiende consultas de estudiantes directamente?",

    # --- Coloquial / informal ---
    "quien manda en la escuela ahorita",
    "quien es la directora actual",
    "a quien le toca dirigir la escuela",
    "quien esta a cargo del departamento academico",
    "como contacto a la directora",

    # --- Con errores ortográficos ---
    "kien es la directora",
    "direktora dela escuela",
    "departamento akademico",
    "autoridades dela epiis",
    "kien dirije la escuela",

    # --- Muy cortas ---
    "directora epiis",
    "director departamento académico",
    "autoridades escuela",
    "consejo de escuela",
    "jefe departamento académico",

    # --- Con saludo ---
    "hola, ¿quién es la directora de la Escuela?",
    "buenos días, ¿quién dirige el Departamento Académico?",
    "disculpa, ¿cómo contacto a la dirección de la Escuela?",

    # --- Teléfono / contacto institucional ---
    "¿cuál es el teléfono de la Escuela?",
    "número de contacto de la EPIIS",
    "¿tienen anexo telefónico?",
    "¿a qué número llamo para consultas administrativas?",
    "teléfono de la oficina de la Escuela Profesional",
    "¿cómo me comunico con la Escuela por teléfono?",
   # --- más variantes sobre teléfono / contacto institucional ---

    "¿cuál es el número de la oficina de la EPIIS?",
    "¿tienen línea directa para consultas?",
    "número celular de contacto de la Escuela",
    "¿hay un WhatsApp institucional de la Escuela?",
    "¿puedo llamar por teléfono para hacer una consulta?",
    "¿cuál es el código de anexo de la dirección de Escuela?",
    "teléfono fijo de la Facultad FIEEIM",
    "¿cómo contacto telefónicamente al Departamento Académico?",
    "número para comunicarme con secretaría de la Escuela",
    "¿hay un número de emergencia académica?",
    "contacto telefónico para trámites administrativos",
    "¿el teléfono de la Escuela es el mismo que el de la Facultad?",
    "¿a qué hora puedo llamar a la Escuela?",
    "¿la Escuela responde llamadas los fines de semana?",
    "número de contacto para consultas de matrícula",
    "teléfono de informes de la EPIIS",

    # --- Coloquial / informal ---
    "a que numero llamo a la escuela",
    "tienen telefono o solo correo",
    "como llamo a la escuela",
    "cual es el numero pa preguntas",
    "tienen whatsapp de la escuela",

    # --- Con errores ortográficos ---
    "telefono dela escuela",
    "numero de kontacto",
    "anekso telefonico",
    "komo me komunico con la escuela",
    "telefono dela epiis",

    # --- Muy cortas ---
    "teléfono escuela",
    "número de contacto",
    "anexo telefónico",
    "contacto epiis",
    "teléfono epiis",

    # --- Con saludo ---
    "hola, ¿cuál es el teléfono de la Escuela?",
    "buenos días, necesito el número de contacto",
    "disculpa, ¿cómo llamo a la Escuela?",


    # --- Ubicación ---
    "¿en qué pabellón está la Escuela?",
    "¿la Escuela está en Perayoc?",
    "¿dónde queda la Facultad de Ingeniería Eléctrica, Electrónica, Informática y Mecánica?",
    "¿cómo llego a la Escuela de Informática desde la Plaza de Armas?",
    "ubicación exacta de la EPIIS dentro del campus",
    "¿en qué sede está la carrera de Informática?",
    "dirección física de la Facultad FIEEIM",
   # --- más variantes sobre ubicación ---

    "¿dónde se encuentra ubicada la Escuela Profesional?",
    "¿en qué parte de la Ciudad Universitaria está la EPIIS?",
    "¿cómo llego a la Escuela desde el centro de Cusco?",
    "¿la Escuela está dentro del campus de la UNSAAC?",
    "indicaciones para llegar a la Facultad FIEEIM",
    "¿qué referencia puedo usar para encontrar la Escuela?",
    "¿la EPIIS tiene su propio edificio?",
    "¿en qué piso están las oficinas de la Escuela?",
    "¿cerca de qué facultad queda la Escuela de Informática?",
    "¿cómo se llega caminando desde la entrada principal de la universidad?",
    "¿hay movilidad o combi que llegue cerca de la Escuela?",
    "¿la Ciudad Universitaria queda lejos del centro histórico?",
    "dirección exacta para usar en Google Maps",
    "¿puedo ubicar la Escuela en un mapa?",
    "¿en qué distrito está ubicada la UNSAAC?",
    "¿la Facultad FIEEIM tiene varias sedes?",
    "¿la Escuela comparte edificio con otra carrera?",
    "punto de referencia cercano a la Escuela de Informática",

    # --- Coloquial / informal ---
    "donde keda la escuela exactamente",
    "como llego a la facultad desde el centro",
    "en que parte del campus esta la epiis",
    "la escuela esta en perayoc o no",
    "como hago pa llegar a la facultad",

    # --- Con errores ortográficos ---
    "donde keda la escuela",
    "ubicasion dela epiis",
    "komo llego a la facultad",
    "direccion fisika dela fieeim",
    "en q sede esta la carrera",

    # --- Muy cortas ---
    "ubicación escuela",
    "dirección epiis",
    "dónde queda la facultad",
    "ubicación fieeim",
    "cómo llegar a la escuela",

    # --- Con saludo ---
    "hola, ¿dónde queda la Escuela Profesional?",
    "buenos días, ¿cómo llego a la Facultad FIEEIM?",
    "disculpa, ¿cuál es la ubicación exacta de la EPIIS?",


    # --- Acreditación ---
    "¿qué significa que la carrera esté acreditada?",
    "¿hasta cuándo es válida la acreditación de la EPIIS?",
    "certificación de calidad de la carrera",
    "¿qué entidad acredita a la Escuela de Informática?",
    "beneficios de estudiar en una carrera acreditada",
    "¿la acreditación ICACIT es internacional?",
    "¿la EPIIS ha sido reacreditada?",
    "estándares de calidad de la carrera",
   # --- más variantes sobre acreditación ---

    "¿la carrera de Informática está acreditada?",
    "¿qué es ICACIT?",
    "¿qué garantiza la acreditación ICACIT?",
    "¿la acreditación afecta el valor de mi título?",
    "¿mi título tiene más valor si la carrera está acreditada?",
    "¿qué proceso sigue la Escuela para mantener la acreditación?",
    "¿cuándo fue la última vez que se acreditó la carrera?",
    "¿la acreditación se renueva cada cuánto tiempo?",
    "¿qué pasa si la Escuela pierde la acreditación?",
    "¿la acreditación influye en las oportunidades laborales?",
    "¿la acreditación permite estudiar en el extranjero más fácil?",
    "¿otras escuelas de informática del Perú están acreditadas igual?",
    "¿qué requisitos pide ICACIT para acreditar una carrera?",
    "¿la acreditación es nacional o internacional?",
    "diferencia entre acreditación y licenciamiento de SUNEDU",
    "¿la EPIIS está licenciada por SUNEDU?",
    "¿el licenciamiento es lo mismo que la acreditación?",
    "documento que certifica la acreditación de la carrera",
    "¿puedo ver el certificado de acreditación de la Escuela?",

    # --- Coloquial / informal ---
    "la carrera esta acreditada o no",
    "que es eso de icacit",
    "para que sirve que este acreditada la carrera",
    "vale mas mi titulo si esta acreditada la escuela",
    "hasta cuando dura la acreditacion",

    # --- Con errores ortográficos ---
    "akreditacion dela carrera",
    "esta acreditada la hescuela",
    "ikacit que es",
    "certificasion de calidad",
    "la epiis esta akreditada",

    # --- Muy cortas ---
    "acreditación",
    "acreditación epiis",
    "icacit",
    "licenciamiento sunedu",
    "calidad acreditada",

    # --- Con saludo ---
    "hola, ¿la carrera está acreditada?",
    "buenos días, ¿qué entidad acredita a la EPIIS?",
    "disculpa, ¿hasta cuándo es válida la acreditación?",


    # --- Perfil del egresado ---
    "¿qué tipo de profesional forma la Escuela?",
    "¿qué conocimientos técnicos adquiere el egresado?",
    "¿el egresado puede trabajar en desarrollo de software?",
    "¿el perfil del egresado incluye gestión de proyectos?",
    "¿qué lo diferencia a un egresado de la EPIIS de otro ingeniero?",
    "describe al egresado ideal de la carrera",
    "¿qué valores forma la Escuela en sus egresados?",
    "¿el egresado tiene formación en ética profesional?",
   # --- más variantes sobre perfil del egresado ---

    "¿qué competencias debe tener un egresado de la EPIIS?",
    "¿qué habilidades blandas desarrolla el egresado?",
    "¿el egresado sabe programar en varios lenguajes?",
    "¿el egresado está preparado para trabajar en equipo?",
    "¿qué nivel de inglés tiene un egresado de la carrera?",
    "¿el egresado tiene formación en inteligencia artificial?",
    "¿el perfil del egresado incluye ciberseguridad?",
    "¿el egresado puede liderar equipos de desarrollo?",
    "¿qué capacidad de análisis tiene el egresado de informática?",
    "¿el egresado sabe diseñar bases de datos?",
    "¿el perfil de egreso incluye investigación científica?",
    "¿el egresado tiene pensamiento crítico desarrollado?",
    "¿qué lo hace competitivo a un egresado de la EPIIS en el mercado?",
    "¿el egresado puede adaptarse a nuevas tecnologías fácilmente?",
    "¿la formación del egresado incluye responsabilidad social?",
    "¿qué tan completo es el perfil académico del egresado?",
    "¿el egresado de la EPIIS está capacitado para certificaciones internacionales?",
    "¿qué nivel de autonomía profesional tiene el egresado?",
    "comparación entre el perfil de egreso y el campo laboral real",
    "¿el perfil del egresado menciona innovación tecnológica?",
    "¿el egresado puede trabajar de forma multidisciplinaria?",

    # --- Coloquial / informal ---
    "que sabe hacer alguien que sale de esta carrera",
    "que tan preparado sale un egresado de informatica",
    "el egresado puede hacer apps o solo paginas web",
    "que valores te da la escuela cuando egresas",
    "para que esta preparado un ingeniero de la epiis",

    # --- Con errores ortográficos ---
    "perfil del ehgresado",
    "kompetencias del egresado",
    "abilidades del profesional",
    "perfil profecional dela carrera",
    "ke sabe haser un egresado",

    # --- Muy cortas ---
    "perfil egresado",
    "competencias egresado",
    "habilidades egresado",
    "perfil profesional",
    "valores egresado",

    # --- Con saludo ---
    "hola, ¿qué perfil tiene el egresado de la carrera?",
    "buenos días, ¿qué competencias desarrolla un egresado?",
    "disculpa, ¿qué tipo de profesional forma la Escuela?",


    # --- Campo laboral ---
    "¿qué empresas contratan egresados de la EPIIS?",
    "¿se puede trabajar de forma independiente como egresado?",
    "¿el egresado puede trabajar en el extranjero?",
    "sectores donde puede laborar un ingeniero informático",
    "¿hay demanda laboral para esta carrera?",
    "¿qué puestos puede ocupar un egresado de Informática y Sistemas?",
    "¿se puede emprender con este título?",
    "salidas profesionales de la carrera",
   # --- más variantes sobre campo laboral ---

    "¿en qué tipo de empresas puede trabajar un egresado?",
    "¿el egresado puede trabajar en bancos o entidades financieras?",
    "¿hay oportunidades laborales en el sector público?",
    "¿el ingeniero informático puede trabajar en el gobierno regional?",
    "¿se puede trabajar como freelancer con este título?",
    "¿el egresado puede trabajar remoto para empresas extranjeras?",
    "¿qué tan fácil es conseguir trabajo al egresar?",
    "¿el mercado laboral en Cusco es bueno para esta carrera?",
    "¿hay más oportunidades laborales en Lima o en el extranjero?",
    "¿qué cargo puede tener un recién egresado en su primer empleo?",
    "¿el egresado puede trabajar como analista de sistemas?",
    "¿se puede trabajar como desarrollador de software?",
    "¿el egresado puede dedicarse a la docencia universitaria?",
    "¿hay egresados trabajando en empresas tecnológicas grandes?",
    "¿el título permite trabajar en consultoría de TI?",
    "¿qué tan bien pagado está el campo laboral de informática?",
    "¿se puede migrar al área de ciencia de datos después de egresar?",
    "¿el egresado puede especializarse después en otro país?",
    "¿hay convenios de la Escuela con empresas para contratar egresados?",
    "bolsa de trabajo para egresados de la EPIIS",
    "¿la universidad ayuda a conseguir trabajo después de egresar?",

    # --- Coloquial / informal ---
    "en que puedo trabajar si estudio esto",
    "se consigue chamba facil con esta carrera",
    "se puede trabajar afuera con este titulo",
    "donde trabajan los que salen de aqui",
    "se puede hacer negocio propio con este titulo",

    # --- Con errores ortográficos ---
    "kampo laboral dela carrera",
    "en q puedo trabahar",
    "salidas profecionales",
    "se puede emprender con este titulo",
    "demanda laboral dela carrera",

    # --- Muy cortas ---
    "campo laboral",
    "salidas profesionales",
    "oportunidades laborales",
    "dónde trabajan los egresados",
    "demanda laboral",

    # --- Con saludo ---
    "hola, ¿en qué puede trabajar un egresado de esta carrera?",
    "buenos días, ¿hay demanda laboral para Informática y Sistemas?",
    "disculpa, ¿se puede trabajar en el extranjero con este título?",


    # --- Sitio web / redes oficiales ---
    "¿cuál es la página web oficial de la Escuela?",
    "¿dónde encuentro el portal de la EPIIS?",
    "enlace oficial de la Escuela Profesional",
    "¿tienen redes sociales oficiales?",
    "página de Facebook de la Escuela",
    "¿dónde veo anuncios oficiales de la Escuela?",
   # --- más variantes sobre sitio web / redes oficiales ---

    "¿cuál es la dirección web de la EPIIS?",
    "link de la página oficial de la Escuela",
    "¿dónde está la web institucional de la carrera?",
    "¿la Escuela tiene Instagram?",
    "¿tienen cuenta de Twitter o X oficial?",
    "¿hay un canal de YouTube de la Escuela?",
    "¿dónde sigo las publicaciones oficiales de la EPIIS?",
    "¿cómo me entero de los comunicados de la Escuela?",
    "¿la página web tiene información sobre matrícula?",
    "¿el portal web está actualizado?",
    "¿dónde veo el calendario académico en línea?",
    "¿hay un grupo de Facebook de estudiantes de la EPIIS?",
    "¿dónde se publican las convocatorias de la Escuela?",
    "¿la Escuela tiene página independiente o depende de la UNSAAC?",
    "¿el sitio web de la Escuela está dentro del dominio unsaac.edu.pe?",
    "¿dónde reviso comunicados oficiales de la Facultad FIEEIM?",
    "¿hay alguna app oficial de la Escuela?",

    # --- Coloquial / informal ---
    "tienen pagina web o solo facebook",
    "donde sigo a la escuela en redes",
    "cual es el link de la escuela",
    "tienen instagram de la epiis",
    "donde veo los avisos de la escuela",

    # --- Con errores ortográficos ---
    "pagina web dela escuela",
    "rredes sosiales dela epiis",
    "enlase oficial",
    "facebook dela escuela",
    "portal web dela carrera",

    # --- Muy cortas ---
    "página web epiis",
    "sitio web escuela",
    "redes sociales epiis",
    "facebook escuela",
    "portal epiis",

    # --- Con saludo ---
    "hola, ¿cuál es la página web de la Escuela?",
    "buenos días, ¿tienen redes sociales oficiales?",
    "disculpa, ¿dónde veo los comunicados de la EPIIS?",


    # --- Infraestructura  ---
    "¿la Escuela tiene laboratorios de cómputo?",
    "¿cuántos laboratorios tiene la Escuela?",
    "¿hay biblioteca especializada en la Escuela?",
    "¿la Escuela cuenta con auditorio propio?",
    "¿hay salas de cómputo disponibles para estudiantes?",
    "infraestructura tecnológica de la EPIIS",
    "¿tienen wifi en las aulas?",

    # --- Comparativas / generales ---
    "¿qué hace única a la EPIIS frente a otras universidades?",
    "¿por qué estudiar Informática en la UNSAAC?",
    "ventajas de estudiar en esta Escuela",
    "¿la EPIIS es la única escuela de informática en Cusco?",
    "resumen general sobre la Escuela Profesional",
    "cuéntame todo sobre la EPIIS",
    "información institucional completa de la Escuela",

    # --- Coloquial / informal ---
    "de q trata la escuela de informatica",
    "quien manda en la escuela",
    "donde keda la escuela",
    "hace cuanto existe la carrera",
    "la escuela esta acreditada o no",
    "en q trabaja uno que sale de esta carrera",
    "cuantos años son de carrera",
    "como me comunico con la escuela",

    # --- Con errores ortográficos ---
    "mision de la hescuela",
    "kien es la directora",
    "ubikacion de la escuela",
    "acreditacion ikacit",
    "perfil del egresadoo",

    # --- Muy cortas ---
    "misión",
    "visión",
    "historia de la escuela",
    "directora",
    "ubicación",
    "acreditación",
    "perfil del egresado",
    "campo laboral",
    "sitio web",
    "teléfono",

    # --- Con saludo ---
    "hola, ¿cuál es la misión de la Escuela?",
    "buenos días, ¿quién es la directora actual?",
    "disculpa, ¿dónde queda la Escuela?",
    "hola, quisiera saber sobre la acreditación de la carrera",

    ],

    "docentes": [
    # --- Listado general / directorio ---
    "¿Qué docentes tiene la escuela?",
    "lista de profesores de la escuela",
    "¿Quiénes son los docentes de la carrera?",
    "Quiero saber sobre los profesores",
    "Directorio de docentes",
    "muéstrame la lista de docentes",
    "dame el listado completo de profesores",
    "¿Qué profesores hay en la EPIIS?",
    "necesito el directorio de profesores",
    "quiero ver todos los docentes",
    "lista completa de docentes de la escuela",
    "enséñame los docentes registrados",
    "¿me puedes mostrar todos los profesores?",
    "dame el directorio académico",
    "quiero el listado de plana docente",
    "muéstrame la plana docente",
    "¿cuál es la plana docente de la escuela?",
    "necesito ver el cuerpo docente",
    "dame la nómina de profesores",
    "¿qué docentes conforman la escuela?",
    "quisiera conocer a los docentes",
    "muéstrame el padrón de docentes",
    "lista de catedráticos de la escuela",
    "¿quiénes enseñan en la escuela?",
    "¿quiénes son los catedráticos?",
    "dame todos los nombres de los profesores",
    "quiero los nombres de los docentes",

    # --- Conteo ---
    "¿Cuántos docentes hay en la escuela?",
    "Cantidad de docentes registrados",
    "¿cuántos profesores tiene la carrera?",
    "número total de docentes",
    "¿cuántos catedráticos hay?",
    "dime cuántos docentes son en total",
    "¿cuál es el número de profesores de la escuela?",
    "total de docentes de tiempo completo",
    "total de docentes de dedicación exclusiva",
    "¿cuántos docentes principales hay?",
    "¿cuántos docentes asociados hay?",
    "¿cuántos auxiliares hay registrados?",
    "¿cuántos jefes de práctica hay?",
    "cantidad de profesores por categoría",

    # --- Por tipo de dedicación / categoría ---
    "¿Qué profesores son de tiempo completo?",
    "¿Qué docentes tienen dedicación exclusiva?",
    "Docentes a dedicación exclusiva en la Escuela",
    "¿Quiénes son los docentes principales?",
    "Lista de docentes principales",
    "docentes asociados de la escuela",
    "¿qué profesores son auxiliares?",
    "lista de docentes auxiliares",
    "¿quiénes son jefes de práctica?",
    "docentes de tiempo parcial",
    "¿qué docentes trabajan a tiempo parcial?",
    "profesores principales de tiempo completo",
    "docentes asociados a tiempo completo",
    "¿cuáles son los docentes de dedicación exclusiva?",
    "muéstrame solo a los docentes principales",
    "filtra los docentes por dedicación exclusiva",
    "quiero ver solo a los auxiliares",
    "¿hay docentes a tiempo parcial de 20 horas?",
    "dame los profesores con cargo de jefe de prácticas",
    "categoría de los docentes de la escuela",
    "¿qué tipos de dedicación tienen los docentes?",
    "clasificación de los docentes por dedicación",

    # --- Información sobre un docente específico (nombre completo) ---
    "información sobre el docente Carrasco",
    "¿Quién es el profesor Enciso Rodas?",
    "Datos del docente Rozas Huacho",
    "Información del docente Flores Pacheco",
    "Datos del profesor Palomino Olivera",
    "¿Quién es Nila Acurio?",
    "¿Quién es Edwin Carrasco?",
    "información sobre el profesor Villafuerte",
    "datos del docente Palma Ttito",
    "¿quién es el docente Candia Oviedo?",
    "cuéntame sobre el profesor Carbajal Luna",
    "información del docente Alzamora Paredes",
    "¿quién es la docente Ormeño Ayala?",
    "datos del profesor Ibarra Zambrano",
    "¿quién es la profesora Choque Soto?",
    "información sobre Karelia Medina",
    "¿quién es Javier Chávez Centeno?",
    "datos del docente Ticona Pari",
    "¿quién es Iván Medrano Valencia?",
    "información del profesor Baca Cárdenas",
    "¿quién es Esther Pacheco Vásquez?",
    "datos del docente Pillco Quispe",
    "¿quién es Efraina Cutipa Arapa?",
    "información sobre Maritza Irpanocca",
    "¿quién es el profesor Harley Vera?",
    "datos de Willian Zamalloa",
    "¿quién es Doris Aguirre Carbajal?",
    "información del docente Dueñas Bustinza",
    "¿quién es Tany Villalba?",
    "datos del profesor Jisbaj Gamarra",
    "¿quién es Henry Dueñas De La Cruz?",
    "información sobre Manuel Peñaloza",
    "¿quién es el ingeniero Marcio Merma?",
    "datos del profesor Elmer Cama Cáceres",
    "cuéntame del docente Lino Baca",
    "quiero saber del profesor Guzmán Ticona",

    # --- Información usando solo nombre de pila o apodo común ---
    "¿quién es el profesor Lino?",
    "¿quién es el profesor Edwin?",
    "¿quién es el doctor Emilio Palomino?",
    "información sobre el doctor Rony Villafuerte",
    "datos del magíster Julio Carbajal",
    "¿quién es la doctora Yeshica Ormeño?",
    "información sobre el ingeniero Lino Baca",
    "¿quién es la licenciada Esther Pacheco?",

    # --- Hoja de vida / CV ---
    "¿dónde veo la hoja de vida del profesor Carrasco?",
    "quiero ver el CV del docente Enciso",
    "¿tiene hoja de vida el profesor Rozas?",
    "muéstrame la hoja de vida de Nila Acurio",
    "link de la hoja de vida de Edwin Carrasco",
    "¿cómo veo el currículum del docente Villafuerte?",
    "necesito el CTI Vitae del profesor Palma Ttito",

    # --- Correo / contacto ---
    "¿cuál es el correo del profesor Carrasco?",
    "dame el email del docente Enciso Rodas",
    "¿cómo contacto al profesor Rozas Huacho?",
    "correo electrónico de Nila Acurio",
    "necesito el contacto de Edwin Carrasco",
    "¿cuál es el email de la profesora Choque Soto?",
    "dame el correo de Karelia Medina",
    "¿cómo escribo al docente Pillco Quispe?",
    "correo institucional del profesor Baca Cárdenas",

    # --- Por curso / área que dicta (uso de memoria contextual de la app) ---
    "¿Qué profesor dicta inteligencia artificial?",
    "Profesores del área de software",
    "¿qué docente enseña bases de datos?",
    "¿quién dicta el curso de redes?",
    "¿qué profesor enseña sistemas embebidos?",
    "¿quién dicta análisis de datos empresariales?",
    "docentes del área de sistemas",
    "¿qué docentes investigan en IA?",
    "¿quién enseña programación avanzada?",
    "profesores especializados en machine learning",
    "¿qué docente dicta arquitectura de software?",
    "¿quién dicta el curso de ingeniería de software?",

    # --- Por semestre / ciclo ---
    "Docentes que dictan en sexto semestre",
    "Docentes que dictan en séptimo semestre",
    "¿qué docentes dictan en primer semestre?",
    "¿quiénes enseñan en el ciclo de noveno semestre?",
    "profesores que dictan en quinto ciclo",
    "¿qué docentes tiene el cuarto semestre?",
    "lista de profesores del último ciclo",
    "¿quién dicta en el segundo semestre?",

    # --- Variantes informales / coloquiales / con errores comunes ---
    "profes de la escuela",
    "dime los profes que hay",
    "quien son los profes de sistemas",
    "que profesores hay nomas",
    "dame los profes porfa",
    "oe que docentes hay",
    "necesito saber los profes de la carrera",
    "quien es el profe carrasco",
    "sabes quien es el profe enciso",
    "me puedes decir los docentes",
    "ayudame con la lista de profesores",
    "quiero info de los profesores",
    "dame data de los docentes",
    "tienes info del profesor rozas",
    "que sabes del profesor carrasco",
    "cuentame de los docentes porfavor",

    # --- Variantes formales / institucionales ---
    "Solicito el directorio de docentes de la Escuela Profesional",
    "Requiero la nómina docente actualizada",
    "Por favor, proporcione la lista de docentes principales",
    "Favor de indicar los docentes de dedicación exclusiva",
    "Solicito información curricular del Dr. Carrasco Poblete",
    "Se requiere el listado de plana docente de la EPIIS",
    "Sírvase mostrar el directorio completo de catedráticos",

    # --- Comparativas / agregadas ---
    "¿hay más docentes de tiempo completo o dedicación exclusiva?",
    "¿cuál categoría de docentes es más numerosa?",
    "compara la cantidad de docentes por tipo de dedicación",
    "¿qué porcentaje de docentes son auxiliares?",

    # --- Preguntas sobre grado académico ---
    "¿qué docentes tienen grado de doctor?",
    "lista de docentes con título de magíster",
    "¿cuántos doctores hay entre los docentes?",
    "¿quiénes son los docentes con grado de Dr.?",
    "profesores con grado de Mgt.",
    "¿hay docentes con título de ingeniero?",
    "¿qué docentes son licenciados?",

    # --- Búsqueda parcial / aproximada ---
    "busco un profesor que se llama Javier",
    "hay algún docente apellido Villalba",
    "busco al docente de apellido Medrano",
    "existe un profesor llamado Harley",
    "busco a la profesora Vanessa",
    "hay un docente apellido Gamarra",
    "busco información de alguien apellido Dueñas",

    # --- Preguntas abiertas / exploratorias ---
    "cuéntame sobre el cuerpo docente de la escuela",
    "dame un resumen de los profesores de la carrera",
    "explícame quiénes conforman la plana docente",
    "describe a los docentes de la escuela",
    "resume la información de los docentes",

    # --- Variantes con errores ortográficos típicos ---
    "kien es el profesor carrasco",
    "lista de docentess",
    "cuantos doscentes hay",
    "informacion del docente flores pachecho",
    "quien es el dosente rozas",
    "dame el directorio de profesores xd",
    "necesito sabr quien es el profe enciso",

    # --- Variantes muy cortas (estilo búsqueda) ---
    "docentes",
    "profesores",
    "lista docentes",
    "plana docente",
    "directorio docentes",
    "cuántos docentes",
    "docentes tiempo completo",
    "docentes dedicación exclusiva",
    "docente carrasco",
    "docente rozas",
    "docente enciso",
    "docente acurio",

    # --- Con saludo o cortesía incluida ---
    "hola, ¿me puedes dar la lista de docentes?",
    "buenos días, quisiera saber quiénes son los profesores",
    "disculpa, ¿qué docentes tiene la escuela?",
    "por favor ¿me ayudas con info de los docentes?",
    "hola, necesito el directorio de profesores",
    "buenas, ¿quién es el profesor Carrasco?",
],
    "cursos": [
    # --- Listado general por semestre/ciclo ---
    "¿qué cursos se llevan en el semestre?",
    "¿qué cursos hay en el sexto semestre?",
    "¿qué asignaturas tiene el séptimo ciclo?",
    "¿qué cursos llevo en el ciclo 7?",
    "lista de cursos de la carrera",
    "cursos del plan curricular 2017",
    "¿qué cursos hay en primer semestre?",
    "muéstrame los cursos del segundo ciclo",
    "¿qué materias hay en el tercer semestre?",
    "asignaturas del cuarto semestre",
    "¿qué cursos tiene el quinto ciclo?",
    "cursos del octavo semestre",
    "¿qué se lleva en noveno ciclo?",
    "materias del décimo semestre",
    "dame todos los cursos por ciclo",
    "muéstrame la malla por semestre",
    "¿cuántos cursos hay en cada semestre?",
    "¿qué cursos tiene el primer año?",
    "asignaturas del segundo año de la carrera",
    "cursos que se dictan en el último ciclo",
    # --- más variantes sobre listado de cursos por semestre/ciclo ---

    "¿qué cursos corresponden al sexto ciclo?",
    "muéstrame las asignaturas del séptimo semestre",
    "¿cuáles son los cursos del primer ciclo de la carrera?",
    "dame la lista de materias del segundo semestre",
    "¿qué llevo en el tercer año de la carrera?",
    "cursos correspondientes al cuarto año",
    "¿qué asignaturas tiene el quinto año?",
    "materias que se dictan en el sexto año",
    "¿qué cursos hay en el ciclo 1?",
    "¿qué cursos hay en el ciclo 2?",
    "¿qué cursos hay en el ciclo 3?",
    "¿qué cursos hay en el ciclo 4?",
    "¿qué cursos hay en el ciclo 5?",
    "¿qué cursos hay en el ciclo 8?",
    "¿qué cursos hay en el ciclo 9?",
    "¿qué cursos hay en el ciclo 10?",
    "lista de asignaturas por ciclo académico",
    "muéstrame el detalle de cursos de cada semestre",
    "¿qué materias debo llevar en el ciclo que sigue?",
    "¿cuántas asignaturas tiene el plan por semestre?",
    "dame el resumen de cursos de todos los semestres",
    "¿qué cursos componen el primer ciclo académico?",
    "cursos obligatorios del tercer semestre",
    "¿qué se enseña en el cuarto ciclo de la carrera?",
    "lista completa de materias por año académico",
    "¿qué cursos van en el primer ciclo de estudios generales?",
    "asignaturas correspondientes a estudios específicos por semestre",
    "¿en qué ciclo empiezan los cursos de especialidad?",
    "¿desde qué semestre hay cursos de especialidad?",
    "¿qué cursos tiene cada ciclo según la malla curricular?",

    # --- Sobre rango de semestres ---
    "cursos desde el primer hasta el quinto semestre",
    "materias de los primeros cinco ciclos",
    "cursos de la segunda mitad de la carrera",
    "asignaturas de los últimos cinco semestres",
    "¿qué cursos hay entre sexto y décimo ciclo?",

    # --- Coloquial / informal ---
    "que jalo en sexto ciclo",
    "que cursos tengo en el siguiente semestre",
    "que materias hay en el ciclo que viene",
    "que se ve en el primer año",
    "cuantos cursos jalo por semestre",
    "que cursos toca en septimo",

    # --- Con errores ortográficos ---
    "q cursos hay en el siklo 6",
    "materias del cuatro semestre",
    "q se ve en el kinto siclo",
    "kursos del octabo semestre",
    "cursos del primer siklo",

    # --- Muy cortas ---
    "cursos ciclo 1",
    "cursos ciclo 5",
    "cursos ciclo 8",
    "materias por semestre",
    "cursos por año",

    # --- Con saludo ---
    "hola, ¿qué cursos hay en el sexto semestre?",
    "buenos días, ¿qué asignaturas tiene el primer ciclo?",
    "disculpa, ¿qué materias llevo en cuarto semestre?",


    # --- Por área temática ---
    "¿cuáles son los cursos de inteligencia artificial?",
    "¿qué cursos hay del área de software?",
    "¿qué cursos son de bases de datos?",
    "¿qué materias hay de redes de computadoras?",
    "¿qué cursos hay de algoritmos?",
    "cursos del área de ciencias básicas",
    "asignaturas del área de gestión de TI",
    "cursos del área de ingeniería computacional",
    "¿qué cursos hay sobre sistemas embebidos?",
    "materias relacionadas a machine learning",
    "¿qué cursos hay de bioinformática?",
    "cursos sobre visión computacional",
    "¿hay cursos de procesamiento de lenguaje natural?",
    "asignaturas del área de investigación",
    "¿qué cursos hay de matemáticas?",
    "materias de electrónica y hardware",
    "¿qué cursos tratan sobre robótica?",
    "cursos relacionados a ciberseguridad",
    "¿hay algún curso de deep learning?",
    "materias sobre compiladores",
    "¿qué cursos hay de computación gráfica?",
    "cursos del área de ciencias de la computación",
   # --- más variantes sobre cursos por área temática ---

    "asignaturas relacionadas con inteligencia artificial",
    "¿qué materias tocan el tema de IA?",
    "cursos donde se aprende sobre redes neuronales",
    "¿qué cursos forman parte del área de ingeniería de software?",
    "materias enfocadas en desarrollo de software",
    "¿qué cursos tienen que ver con metodologías de desarrollo?",
    "asignaturas sobre diseño de bases de datos",
    "¿qué materias hay sobre administración de bases de datos?",
    "cursos relacionados a SQL o modelado de datos",
    "¿qué asignaturas tratan sobre redes de computadoras?",
    "materias sobre protocolos de red",
    "cursos donde se estudia TCP/IP",
    "¿qué cursos enseñan sobre estructuras de datos?",
    "asignaturas sobre complejidad algorítmica",
    "cursos que tratan sobre diseño de algoritmos",
    "¿qué materias hay sobre cálculo y álgebra?",
    "cursos del área de física aplicada",
    "asignaturas sobre estadística y probabilidades",
    "¿qué cursos tienen contenido de programación?",
    "materias introductorias a la programación",
    "¿qué cursos hay sobre arquitectura de computadoras?",
    "asignaturas de electrónica digital",
    "cursos sobre diseño de circuitos",
    "¿qué materias tratan temas de gestión empresarial en TI?",
    "asignaturas sobre planeamiento estratégico de tecnologías",
    "cursos relacionados a formulación de proyectos tecnológicos",
    "¿qué materias hay sobre auditoría de sistemas?",
    "cursos sobre control de calidad de software",
    "asignaturas de modelos probabilísticos y simulación",
    "¿qué cursos tratan sobre minería de datos?",
    "materias sobre análisis de datos empresariales",
    "cursos relacionados con big data",
    "¿qué asignaturas hay sobre sistemas operativos?",
    "materias que abordan virtualización o contenedores",
    "cursos sobre lenguajes de programación avanzados",
    "¿qué materias tocan el tema de paralelismo y concurrencia?",
    "asignaturas sobre computación distribuida",
    "cursos relacionados a sistemas en la nube",
    "¿qué materias hay sobre interacción persona-computador?",
    "cursos de diseño de interfaces de usuario",

    # --- Comparativas entre áreas ---
    "¿qué área tiene más cursos, IA o software?",
    "compara los cursos de redes con los de bases de datos",
    "¿cuál área temática tiene más créditos en total?",

    # --- Coloquial / informal ---
    "que cursos hay de ia",
    "que materias tocan temas de redes",
    "hay cursos de ciberseguridad o no",
    "que cursos ven temas de robotica",
    "que materias son de bases de datos",

    # --- Con errores ortográficos ---
    "kursos de intelijencia artificial",
    "materias de basesdedatos",
    "q cursos ay de redes",
    "kursos relacionados a algoritmos",
    "materias sobre maquinas de aprendizaje",

    # --- Muy cortas ---
    "cursos de IA",
    "cursos de software",
    "cursos de redes",
    "cursos de bases de datos",
    "cursos de algoritmos",

    # --- Con saludo ---
    "hola, ¿qué cursos hay de inteligencia artificial?",
    "buenos días, ¿qué materias hay del área de software?",
    "disculpa, ¿qué cursos hay relacionados a redes?",


    # --- Créditos ---
    "¿cuántos créditos tiene sistemas operativos?",
    "¿cuántos créditos hay que llevar por semestre?",
    "¿qué es un crédito académico?",
    "¿cuántas horas equivale 1 crédito?",
    "horas teóricas y horas prácticas por crédito",
    "¿cuántos créditos tiene algoritmos avanzados?",
    "¿cuántos créditos suma toda la carrera?",
    "¿cuántos créditos necesito para egresar?",
    "¿cuántos créditos tiene inteligencia artificial?",
    "créditos totales del plan de estudios",
    "¿cuántos créditos acumulados hay hasta sexto semestre?",
    "¿qué curso tiene más créditos?",
    "¿hay cursos de 2 créditos?",
    "¿cuántos créditos exige el seminario de investigación?",
    "¿cuántos créditos necesito para llevar quechua?",
    "¿qué créditos se requieren para una actividad extracurricular?",

    # --- Prerrequisitos ---
    "¿cuáles son los prerrequisitos de inteligencia artificial?",
    "¿qué necesito para llevar bases de datos?",
    "requisito para el curso de redes I",
    "¿qué curso es prerrequisito de deep learning?",
    "¿puedo llevar IA sin haber llevado modelos probabilísticos?",
    "¿qué necesito aprobar antes de algoritmos avanzados?",
    "prerrequisitos de ingeniería de software I",
    "¿cuál es el requisito de sistemas operativos?",
    "¿qué cursos debo aprobar antes de redes II?",
    "cadena de prerrequisitos hasta deep learning",
    "¿qué necesito para llevar el seminario de investigación II?",
    "requisitos para minería de datos",
    "¿se puede llevar bioinformática sin modelos probabilísticos?",
    "¿cuál es el curso previo a compiladores?",
   # --- más variantes sobre prerrequisitos ---

    "¿qué cursos necesito aprobar antes de sistemas operativos?",
    "¿cuál es el prerrequisito de fundamentos de bases de datos?",
    "¿qué necesito llevar antes de ingeniería de software II?",
    "requisitos previos para análisis y diseño de sistemas de información",
    "¿qué curso debo aprobar antes de redes de computadoras I?",
    "¿cuál es el requisito de algoritmos y estructura de datos?",
    "¿qué necesito para llevar programación II?",
    "prerrequisito de cálculo II",
    "¿qué debo aprobar antes de ecuaciones diferenciales?",
    "¿cuál es el curso previo a métodos numéricos?",
    "requisitos para llevar teoría de la computación",
    "¿qué necesito antes de algoritmos paralelos y distribuidos?",
    "prerrequisito del curso de robótica",
    "¿qué debo llevar antes de redes neuronales artificiales?",
    "¿cuál es el requisito de procesamiento del lenguaje natural?",
    "¿qué cursos necesito aprobar antes de visión computacional?",
    "requisitos previos para aprendizaje automático",
    "¿qué se necesita para llevar análisis de datos empresariales?",
    "prerrequisito de modelado y simulación",
    "¿qué curso es requisito de geometría computacional?",
    "¿cuál es el prerrequisito de computación cuántica?",
    "requisitos para llevar arquitecturas de alto rendimiento",
    "¿qué necesito antes de tópicos avanzados en redes?",
    "prerrequisito de control y auditoría de sistemas",
    "¿qué debo aprobar antes de emprendimiento e innovación?",
    "requisitos del curso de lenguaje ensamblador",
    "¿cuál es el curso previo a formulación de proyectos de TI?",
    "¿qué necesito para llevar planeamiento estratégico de TI?",
    "prerrequisito de administración de tecnologías de información",
    "¿qué cursos debo llevar antes de desarrollo de software II?",

    # --- Verificación de elegibilidad ("¿puedo llevar X sin Y?") ---
    "¿puedo llevar redes II sin haber aprobado redes I?",
    "¿puedo matricularme en compiladores sin teoría de la computación?",
    "¿es posible llevar minería de datos sin aprendizaje automático?",
    "¿puedo llevar ingeniería de software I sin metodologías de desarrollo?",
    "¿se puede llevar estadística inferencial sin probabilidades y estadística?",
    "¿puedo saltarme el prerrequisito de algún curso?",
    "¿qué pasa si llevo un curso sin haber aprobado su requisito?",
    "¿el sistema me deja matricular si no cumplo el prerrequisito?",

    # --- Cadena de prerrequisitos (rutas largas) ---
    "ruta de prerrequisitos hasta redes neuronales artificiales",
    "cadena completa de cursos previos a visión computacional",
    "qué cursos debo llevar en orden hasta llegar a robótica",
    "secuencia de prerrequisitos hasta procesamiento de lenguaje natural",
    "ruta de cursos previos hasta ingeniería de software II",
    "qué cadena de cursos necesito para llegar a ciencia de datos",

    # --- Sobre REQ1 y REQ2 (dos prerrequisitos a la vez) ---
    "¿algoritmos y estructura de datos tiene dos prerrequisitos?",
    "¿qué cursos son los dos requisitos de análisis y diseño de sistemas?",
    "¿hay cursos con más de un prerrequisito?",
    "¿cuáles son los dos cursos previos de teoría de la computación?",

    # --- Coloquial / informal ---
    "que necesito pa llevar ia",
    "que jalo antes de bases de datos",
    "puedo llevar redes 2 sin redes 1",
    "que curso es requisito de compiladores",
    "que necesito aprobar antes de IA",

    # --- Con errores ortográficos ---
    "prerekisitos de inteligencia artificial",
    "q nesesito pa bases de datos",
    "rekisito de redes 1",
    "q kurso es prerekisito de deep learning",
    "puedo yevar ia sin modelos probabilisticos",

    # --- Muy cortas ---
    "prerrequisitos IA",
    "requisito bases de datos",
    "requisito redes I",
    "prerrequisito compiladores",
    "requisito sistemas operativos",

    # --- Con saludo ---
    "hola, ¿cuáles son los prerrequisitos de inteligencia artificial?",
    "buenos días, ¿qué necesito para llevar bases de datos?",
    "disculpa, ¿cuál es el requisito de redes I?",


    # --- Recomendación / planificación ---
    "¿qué cursos puedo llevar este semestre?",
    "recomiéndame cursos de programación",
    "ayúdame a armar mi horario del próximo ciclo",
    "¿qué cursos me convienen llevar si quiero IA?",
    "qué cursos debería priorizar para terminar antes",
    "armame una ruta de cursos para especializarme en software",
    "qué cursos llevar si me interesa la ciberseguridad",
    "plan de cursos para especializarme en IA",
    "¿en qué orden debo llevar los cursos de bases de datos?",
    "ayúdame a planificar mi malla hasta el décimo ciclo",
    "qué cursos puedo jalar este semestre si ya aprobé sistemas operativos",
    "qué cursos electivos me conviene tomar para bioinformática",

    # --- Sobre el sistema curricular en general ---
    "¿cuántas semanas dura el semestre?",
    "¿qué son los períodos lectivos?",
    "¿el currículo es flexible?",
    "duración del semestre académico",
    "malla curricular de la carrera",
    "plan de estudios de Ingeniería Informática",
    "¿dónde consulto la malla curricular?",
    "¿cuántos semestres dura la carrera?",
    "¿cuántos años dura la carrera de Informática?",
    "¿cuál es la duración total del plan de estudios?",
    "¿cuántos cursos tiene en total la malla?",
    "¿qué porcentaje de la malla son estudios específicos?",
    "¿qué porcentaje corresponde a estudios de especialidad?",
    "¿cuántos cursos son obligatorios y cuántos electivos?",
    "objetivos del plan de estudios",
    "objetivo general de la carrera",
   # --- más variantes sobre sistema curricular en general ---

    "¿cuántas semanas tiene cada período lectivo?",
    "¿el semestre se divide en partes más pequeñas?",
    "¿qué son los tres períodos lectivos del semestre?",
    "¿cuántas semanas dura cada período dentro del ciclo?",
    "¿el semestre académico tiene 17 semanas?",
    "duración de un período lectivo",
    "¿cómo se organiza el calendario académico por semestre?",
    "¿el currículo permite cambiar el orden de los cursos?",
    "¿puedo llevar cursos de otro semestre si quiero?",
    "¿la malla es rígida o puedo organizar mis cursos como quiera?",
    "flexibilidad del plan de estudios",
    "¿puedo adelantar cursos de ciclos posteriores?",
    "¿puedo atrasarme y llevar cursos de ciclos anteriores más tarde?",
    "¿dónde descargo la malla curricular en PDF?",
    "¿la malla curricular está disponible en la página de la Escuela?",
    "documento oficial del plan de estudios",
    "¿quién aprueba el plan de estudios de la carrera?",
    "¿cada cuánto se actualiza la malla curricular?",
    "¿el plan de estudios se revisa cada año?",
    "total de semestres considerados en el plan curricular",
    "¿cuántos créditos tiene la carrera en total?",
    "suma total de créditos del plan de estudios",
    "¿cuántas asignaturas conforman toda la malla?",
    "número total de cursos de la carrera",
    "¿qué proporción de cursos son de formación general?",
    "¿qué parte de la malla corresponde a prácticas pre profesionales?",
    "¿qué porcentaje de créditos son de actividades extracurriculares?",
    "¿la malla tiene más cursos obligatorios que electivos?",
    "cantidad de cursos electivos frente a obligatorios",
    "¿cuál es el objetivo general del plan de estudios?",
    "¿qué busca lograr el plan curricular de la carrera?",
    "objetivos específicos del plan de estudios",
    "¿el plan de estudios tiene objetivos de formación básica?",
    "¿qué objetivos de formación profesional tiene la malla?",
    "fines del plan curricular de Ingeniería Informática",

    # --- Coloquial / informal ---
    "cuantas semanas dura el siclo",
    "la malla es flexible o no",
    "donde descargo la malla en pdf",
    "cuantos cursos son en total en la carrera",
    "que porcentaje es de especialidad",

    # --- Con errores ortográficos ---
    "kuantas semanas dura el semestre",
    "periodos lektivos",
    "malla kurricular dela carrera",
    "kuantos kursos tiene la malla",
    "obhetivos del plan de estudios",

    # --- Muy cortas ---
    "duración semestre",
    "períodos lectivos",
    "malla curricular",
    "plan de estudios",
    "objetivos plan estudios",

    # --- Con saludo ---
    "hola, ¿cuántas semanas dura el semestre?",
    "buenos días, ¿dónde consulto la malla curricular?",
    "disculpa, ¿cuál es el objetivo general del plan de estudios?",


    # --- Electivos ---
    "¿hay cursos electivos?",
    "¿qué pasa si desapruebo un curso electivo?",
    "¿puedo reemplazar un electivo por otro?",
    "¿cuántos electivos debo llevar en toda la carrera?",
    "¿en qué semestres hay electivos?",
    "lista de cursos electivos disponibles",
    "¿qué electivos hay para el área de inteligencia artificial?",
    "¿puedo llevar un electivo antes de lo planificado?",
    "¿los electivos tienen prerrequisitos?",
    "¿cuántos créditos suman los electivos?",
    "diferencia entre curso obligatorio y electivo",

    # --- Por código de curso ---
    "¿qué curso es IF612?",
    "información del curso IF651",
    "¿qué asignatura corresponde al código ME356?",
    "dime qué es IF710",
    "¿a qué curso pertenece el código IF482?",
    "datos del curso con código IF454",

    # --- Actividades extracurriculares / prácticas ---
    "¿qué actividades extracurriculares hay?",
    "¿puedo llevar danza moderna como actividad extracurricular?",
    "requisitos para llevar quechua",
    "¿qué créditos necesito para el taller de teatro?",
    "¿cuándo se hacen las prácticas pre profesionales?",
    "¿cuántos créditos necesito para hacer prácticas?",
    "¿cuántas horas son las prácticas pre profesionales?",
    "información sobre el curso de coro",
    "¿qué opciones hay de actividades extracurriculares?",

    # --- Comparativas ---
    "¿qué pesa más en créditos, estudios específicos o de especialidad?",
    "compara los créditos de IA y de bases de datos",
    "¿qué área tiene más cursos, software o computación?",
    "¿cuál área curricular tiene menos créditos?",
    "diferencia de créditos entre programación I y II",

    # --- Negación / exclusión ---
    "cursos que no requieren prerrequisito",
    "asignaturas sin ningún requisito previo",
    "¿qué cursos no son obligatorios?",
    "materias que no pertenecen a ingeniería de software",

    # --- Coloquial / informal ---
    "que cursos hay este ciclo",
    "que jalo este semestre",
    "cuantos creditos tiene IA",
    "que cursos son los mas dificiles",
    "cuales son los cursos facilonchos",
    "que cursos puedo jalar si ya pase bd",
    "oe que cursos hay en septimo",
    "dame los cursos nomas porfa",
    "ayudame con los cursos de este ciclo",
    "que curso jalo despues de sistemas operativos",
    "cuantos cursos son en toda la carrera",
    "cuales son los cursos chancas",

    # --- Variantes con errores ortográficos ---
    "q cursos ay en sexto semestre",
    "q creditos tiene sistemas opertivos",
    "prerequisitos de inteligencia artifical",
    "cursos del area de softwer",
    "cuantos cretidos son en total",
    "q cursos hay de algoritmos avanzdos",

    # --- Muy cortas (estilo búsqueda) ---
    "cursos",
    "malla curricular",
    "plan de estudios",
    "cursos sexto semestre",
    "cursos séptimo semestre",
    "créditos totales",
    "cursos de IA",
    "cursos de bases de datos",
    "prerrequisitos",
    "cursos electivos",
    "cursos obligatorios",

    # --- Formales / institucionales ---
    "Solicito el detalle de los cursos del plan curricular vigente",
    "Requiero información sobre los prerrequisitos de las asignaturas",
    "Sírvase indicar los cursos correspondientes al sexto ciclo académico",
    "Favor proporcionar el listado de cursos electivos disponibles",
    "Se solicita el número de créditos exigidos para egresar",

    # --- Con saludo ---
    "hola, ¿qué cursos hay en sexto semestre?",
    "buenos días, ¿me ayudas con la malla curricular?",
    "disculpa, ¿cuántos créditos tiene IA?",
    "hola, necesito saber los prerrequisitos de redes",
    "buenas, ¿qué cursos puedo llevar este ciclo?",

    # --- Sobre virtualización / modalidad ---
    "¿qué cursos están virtualizados?",
    "¿hay cursos que se dictan de forma virtual?",
    "lista de asignaturas virtualizadas",
    "¿qué cursos se pueden llevar online?",

    # --- Sobre horas (HT/HP) ---
    "¿cuántas horas teóricas tiene programación I?",
    "¿cuántas horas prácticas tiene bases de datos?",
    "horas totales del curso de redes I",
    "¿cuántas horas a la semana es sistemas operativos?",

    # --- Multi-curso / cadena ---
    "qué cursos debo llevar antes de aprendizaje automático",
    "ruta completa de cursos hasta deep learning",
    "secuencia de cursos de matemáticas en la carrera",
    "qué cursos forman parte de la línea de bioinformática",
    "todos los cursos que dependen de fundamentos de programación",

    # --------------------------------------------------------enfoque PLAN CURRICULAR 2025 ---

    # --- Identificar el plan/versión ---
    "¿qué cursos tiene la malla 2025?",
    "cursos del plan curricular 2025",
    "¿cuál es la malla vigente este año?",
    "¿qué plan de estudios me corresponde, 2017 o 2025?",
    "diferencias entre el plan 2017 y el plan 2025",
    "¿qué cambió en la malla nueva respecto a la anterior?",
    "¿el plan 2025 tiene más o menos cursos que el 2017?",
    "¿soy del plan curricular 2025?",
    "¿cómo sé a qué malla pertenezco?",
    "¿puedo cambiarme del plan 2017 al 2025?",
    "¿qué cursos del plan anterior ya no existen en el nuevo?",
    "¿qué cursos son nuevos en la malla 2025?",
    "compara el plan 2017 con el 2025 en créditos totales",

    # --- Cursos nuevos / exclusivos de la malla 2025 ---
    "¿qué es el curso de ciberseguridad?",
    "¿en qué semestre llevo ciberseguridad?",
    "información sobre sistemas internet de las cosas",
    "¿qué es IoT en la malla de informática?",
    "¿qué cursos hay de ciencia de datos?",
    "¿dónde está ubicado el curso de ciencia de datos en la malla?",
    "¿qué prerrequisito tiene aprendizaje profundo?",
    "información sobre el curso de proyección social",
    "¿qué es el curso de debate en la carrera?",
    "¿en qué ciclo se lleva trabajo de investigación?",
    "diferencia entre seminario de investigación y trabajo de investigación",
    "¿qué cursos de IA hay en el plan 2025?",
    "información sobre pensamiento computacional e inteligencia artificial",
    "¿qué curso reemplaza a sistemas embebidos en la malla nueva?",

    # --- Por código específico de la malla 2025 ---
    "¿qué curso es IFI14?",
    "información del curso IFI20",
    "¿a qué asignatura corresponde el código MEI09?",
    "dime qué es IFI31",
    "¿qué curso tiene el código ADI01?",
    "datos del curso ELI01",
    "¿qué asignatura es EUI01?",
    "¿qué es el curso IFI66?",
    "código IFG01 a qué curso pertenece",

    # --- Electivos por semestre (2025) ---
    "¿cuántos electivos de especialidad hay en sexto semestre?",
    "¿en qué semestres hay electivos en la malla 2025?",
    "¿cuántos créditos suman los electivos del plan 2025?",
    "¿noveno semestre tiene dos electivos?",
    "¿cuántos electivos debo llevar en total en el plan 2025?",
    "¿el octavo semestre tiene electivo de especialidad?",
    "¿qué semestre tiene más electivos de especialidad?",

    # --- Requisitos de créditos acumulados (patrón "XXX Cr.") ---
    "¿cuántos créditos necesito para llevar organización y dirección de empresas?",
    "¿cuántos créditos pido para el curso de deporte?",
    "¿cuántos créditos necesito para proyección social?",
    "¿cuántos créditos acumulados pide el seminario de investigación?",
    "¿cuántos créditos necesito para llevar debate?",
    "¿cuántos créditos se requieren para hacer la práctica pre profesional?",
    "¿qué créditos necesito para el curso con código ADI01?",
    "lista de cursos que requieren créditos acumulados en vez de un curso previo",

    # --- Prerrequisitos cruzados (REQ1 y REQ2) ---
    "¿qué cursos necesito antes de modelado y simulación?",
    "¿cuáles son los dos requisitos de algoritmos y estructuras de datos I?",
    "¿qué necesito aprobar para llevar análisis y diseño de sistemas de información?",
    "¿qué prerrequisitos tiene desarrollo de proyectos de ingeniería de software?",
    "¿sistemas internet de las cosas tiene más de un requisito?",
    "requisitos completos de programación paralela y distribuida",
    "¿qué dos cursos necesito antes de administración de tecnologías de la información?",

    # --- Sobre práctica pre-profesional 2025 ---
    "¿cuántos créditos vale la práctica pre profesional en el plan 2025?",
    "¿cuántas horas de práctica son en el plan 2025?",
    "¿en qué semestre se hace la práctica pre profesional?",
    "¿qué requisito de créditos pide la práctica pre profesional?",

    # --- Comparar cargas de HT/HP ---
    "¿qué curso tiene más horas prácticas en la malla 2025?",
    "¿metodología y desarrollo de software tiene más horas que los demás?",
    "¿por qué metodología y desarrollo de software tiene 5 créditos?",
    "¿qué cursos del plan 2025 no tienen horas teóricas?",
    "¿programación I tiene clases teóricas?",

    # --- Curso por nombre exacto (variaciones del plan 2025) ---
    "información sobre algoritmos y estructuras de datos I",
    "¿qué es algebra lineal computacional?",
    "datos del curso métodos numéricos",
    "información sobre análisis y diseño de algoritmos",
    "¿qué es organización y arquitectura del computador?",
    "información del curso fundamentos y diseño de base de datos",
    "¿qué contiene el curso de modelos probabilísticos?",
    "información sobre redes de computadoras II",

    # --- Ruta completa / dependencias en cadena (2025) ---
    "qué cursos debo aprobar antes de llegar a ciberseguridad",
    "ruta completa hasta aprendizaje profundo en la malla 2025",
    "secuencia de cursos de programación en el plan 2025",
    "cadena de requisitos hasta sistemas internet de las cosas",
    "qué cursos dependen de fundamentos de programación en la malla nueva",
    "ruta de cursos para llegar a ingeniería de software II",

    # --- Totales y resumen del plan 2025 ---
    "¿cuántos créditos suma cada semestre en el plan 2025?",
    "¿el plan 2025 también tiene 22 créditos por semestre?",
    "¿cuántos créditos en total tiene la carrera con el plan 2025?",
    "resumen de la malla curricular 2025",
    "¿cuántos cursos obligatorios tiene el plan 2025 sin contar electivos?",

    # --- Coloquial / informal sobre el plan 2025 ---
    "que cursos hay en la malla nueva",
    "cual es el plan que estoy llevando, el viejo o el 2025",
    "que cambio en la malla nueva",
    "cuantos cursos jalo en la malla 2025 antes de ciberseguridad",
    "que es eso de IFI66",
    "para que sirve el curso de iot",
    "que jalo antes de aprendizaje profundo",

    # --- Con errores ortográficos ---
    "q cursos tiene la mlla 2025",
    "cibersegurdad que semestre es",
    "prerequisitos de aprendisaje profundo",
    "q es sistemas internte de las cosas",
    "cuantos creditos pide el seminaro de investigacion",

    # --- Muy cortas ---
    "malla 2025",
    "plan 2025",
    "cursos malla 2025",
    "ciberseguridad",
    "aprendizaje profundo",
    "trabajo de investigación",
    "IoT carrera",
    "ciencia de datos curso",

    "¿qué cursos se llevan en el semestre?",
    "¿qué cursos hay en el sexto semestre?",
    "¿qué asignaturas tiene el séptimo ciclo?",
    "¿cuáles son los cursos de inteligencia artificial?",
    "¿qué cursos hay del área de software?",
    "¿qué cursos son de bases de datos?",
    "¿cuántos créditos tiene sistemas operativos?",
    "¿cuáles son los prerrequisitos de inteligencia artificial?",
    "¿qué cursos puedo llevar este semestre?",
    "lista de cursos de la carrera",
    "¿qué materias hay de redes de computadoras?",
    "cursos del plan curricular 2017",
    "¿qué cursos llevo en el ciclo 7?",
    "recomiéndame cursos de programación",
    "¿qué cursos hay de algoritmos?",
    "¿cuántos créditos hay que llevar por semestre?",
    "¿qué es un crédito académico?",
    "¿cuántas semanas dura el semestre?",
    "¿qué son los períodos lectivos?",
    "¿el currículo es flexible?",
    "duración del semestre académico",
    "malla curricular de la carrera",
    "plan de estudios de Ingeniería Informática",
    "¿dónde consulto la malla curricular?",
    "¿hay cursos electivos?",
    "¿qué pasa si desapruebo un curso electivo?",
    "¿puedo reemplazar un electivo por otro?",
    "¿cuántas horas equivale 1 crédito?",
    "horas teóricas y horas prácticas por crédito",
    "muéstrame los cursos del primer semestre",
    "¿qué llevo en segundo ciclo?",
    "cursos del tercer semestre de la carrera",
    "¿qué materias son del cuarto ciclo?",
    "dame los cursos del quinto semestre",
    "qué hay en el octavo ciclo",
    "cursos del noveno semestre",
    "qué se lleva en el décimo ciclo",
    "ciclo 1 qué cursos tiene",
    "todos los cursos por semestre",
    "muéstrame la malla por ciclos",
    "qué cursos tiene cada semestre de la carrera",
    "lista completa de cursos por ciclo",
    "información sobre el curso de robótica",
    "qué es el curso de visión computacional",
    "dame detalles del curso de minería de datos",
    "qué trata el curso de compiladores",
    "qué es bioinformática en la malla",
    "info del curso IFI651",
    "qué curso es IF652",
    "a qué corresponde el código MEG01",
    "qué curso tiene el código IFI04",
    "busca el curso con código IF612",
    "cómo se llama el curso IFI13",
    "dime el nombre completo de IF710",
    "cuántos créditos tiene cálculo I",
    "créditos de algoritmos y estructuras de datos",
    "cuántos créditos vale programación I",
    "créditos de seminario de investigación",
    "cuántos créditos suma el curso de redes",
    "cuántos créditos necesito para llevar electivos",
    "total de créditos de la carrera",
    "cuántos créditos tiene la carrera en total",
    "créditos mínimos para egresar",
    "cuántos créditos llevo si curso todo el semestre",
    "qué necesito para llevar algoritmos avanzados",
    "requisitos del curso de aprendizaje automático",
    "qué curso necesito aprobar antes de IA",
    "prerrequisito de bases de datos",
    "qué debo aprobar antes de sistemas operativos",
    "requisitos para llevar deep learning",
    "qué cursos previos necesito para redes II",
    "prerrequisitos de ingeniería de software I",
    "puedo llevar visión computacional sin requisitos",
    "qué cursos abren la puerta a inteligencia artificial",
    "cursos relacionados a machine learning",
    "qué cursos hay de ciberseguridad",
    "materias del área de hardware",
    "cursos de electrónica",
    "qué cursos son de matemáticas",
    "materias de estadística y probabilidad",
    "cursos relacionados a sistemas embebidos",
    "qué cursos tocan procesamiento de lenguaje natural",
    "cursos de computación gráfica",
    "materias de gestión y administración de TI",
    "qué cursos son de formación general",
    "cursos de cultura general en la malla",
    "materias de humanidades en la carrera",
    "qué cursos electivos hay disponibles",
    "lista de cursos dirigidos",
    "qué son los cursos dirigidos",
    "diferencia entre curso obligatorio y electivo",
    "cuántos electivos debo llevar en la carrera",
    "qué electivos hay para el noveno semestre",
    "puedo llevar un curso dirigido sin horario publicado",
    "qué hago si un curso dirigido no tiene horario",
    "qué significa la categoría EEEP",
    "qué es un curso OEES",
    "diferencia entre EG y EEG",
    "qué cursos son de estudios generales",
    "qué cursos son obligatorios de especialidad",
    "diferencia entre plan 2017 y plan 2024",
    "qué plan curricular sigo",
    "cursos del plan curricular 2024",
    "mi carrera usa qué plan de estudios",
    "cambios entre el plan antiguo y el nuevo",
    "qué cursos me recomiendas llevar este ciclo",
    "ayúdame a armar mi plan de cursos",
    "qué cursos debería priorizar",
    "en qué orden debo llevar los cursos de programación",
    "qué cursos son más pesados en créditos",
    "qué cursos tienen menos prerrequisitos",
    "cursos fáciles de jalar según los créditos",
    "cuántos semestres dura la carrera",
    "cuántos ciclos tiene ingeniería informática",
    "cuántos cursos tengo que llevar en total",
    "qué necesito para egresar",
    "requisitos para titularme",
    "cuántos créditos necesito para hacer prácticas pre profesionales",
],

    "tramites": [
        # --- continuación intent "tramites" ---

    # --- Matrícula de alumnos regulares / ingresante ---
    "¿qué necesito para matricularme como alumno regular?",
    "¿cuánto cuesta la matrícula si tengo cursos jalados?",
    "¿qué es la escala A, B y C de matrícula?",
    "¿cuánto pago si soy invicto?",
    "¿qué pasa si debo dinero a la universidad y quiero matricularme?",
    "soy ingresante, ¿cómo me matriculo por primera vez?",
    "¿cuánto cuesta la matrícula de un ingresante?",
    "¿qué incluye el pago de matrícula de ingresante?",
    "¿necesito examen médico para matricularme?",
    "¿dónde hago mi matrícula vía internet?",
    "matrícula para estudiantes de primer ciclo",
    "¿qué requisitos tengo si nunca jalé un curso?",

    # --- Matrícula condicionada / cursos dirigidos ---
    "¿qué es la matrícula condicionada?",
    "tengo bajo rendimiento, ¿cómo me matriculo?",
    "¿qué pasa si mi récord académico es bajo?",
    "matrícula por bajo rendimiento académico",
    "¿qué es un curso dirigido?",
    "necesito llevar un curso que ya no se dicta",
    "¿puedo llevar un curso dirigido si no estoy por egresar?",
    "¿cuántos cursos dirigidos puedo llevar como máximo?",
    "costo de un curso dirigido",

    # --- Convalidación ---
    "vengo de otra universidad, ¿cómo convalido mis cursos?",
    "¿qué documentos pide la convalidación de asignaturas?",
    "¿necesito apostillar mis certificados si vengo del extranjero?",
    "¿cuánto cuesta convalidar un curso?",
    "¿cuánto demora la convalidación de asignaturas?",
    "¿qué sílabos necesito para convalidar?",
    "convalidación de cursos de las fuerzas armadas",
    "convalidación de estudios de la PNP",

    # --- Homologación ---
    "¿qué diferencia hay entre convalidación y homologación?",
    "cambié de currículo, ¿necesito homologar mis cursos?",
    "¿cómo homologo una asignatura con nombre distinto?",
    "costo de homologación de asignaturas",
    "tiempo que demora la homologación",
    "¿qué necesito para homologar por cambio de plan curricular?",

    # --- Reserva de matrícula ---
    "quiero pausar mis estudios por motivos de salud",
    "¿hasta cuándo puedo reservar matrícula?",
    "¿cuántos semestres puedo reservar como máximo?",
    "documentos para reserva de matrícula por trabajo",
    "¿la reserva de matrícula puede ser por semestres no consecutivos?",
    "costo de la reserva de matrícula",

    # --- Reinicio de estudios ---
    "dejé de estudiar dos semestres, ¿cómo regreso?",
    "¿cómo recupero mi condición de alumno regular?",
    "¿qué necesito para el reinicio de estudios?",
    "¿hasta cuántos semestres puedo estar inactivo antes de perder la matrícula?",
    "costo del reinicio de estudios por semestre",

    # --- Traslados (interno mismo/otra facultad, externo) ---
    "¿en qué semestre puedo hacer un traslado interno?",
    "¿cuántos créditos necesito para un traslado interno?",
    "diferencia de costo entre traslado a la misma facultad y a otra",
    "¿puedo trasladarme si tengo bajo rendimiento?",
    "vengo de otra universidad del extranjero, ¿cómo me traslado?",
    "¿cuántos créditos mínimos pide el traslado externo?",
    "¿necesito certificado de antecedentes penales para el traslado externo?",
    "costo del traslado externo",
    "tiempo que demora un traslado externo",
    "¿el traslado interno se puede hacer en cualquier semestre?",

    # --- Admisión de titulados/graduados ---
    "ya tengo un título, ¿puedo estudiar otra carrera en la UNSAAC?",
    "¿cómo ingreso si ya soy titulado de otra universidad?",
    "requisitos para segunda profesión",
    "¿necesito que mi título esté en SUNEDU?",
    "costo de admisión para titulados",

    # --- Subsanación ---
    "estoy por egresar y tengo un curso jalado, ¿qué hago?",
    "¿cuántos cursos puedo subsanar como máximo?",
    "¿qué nota mínima necesito para subsanar?",
    "¿la subsanación es solo para egresantes?",
    "costo de la subsanación de un curso",

    # --- Constancias y certificados ---
    "¿cómo saco mi constancia de no ser deudor?",
    "necesito un papel que diga que no debo nada a la universidad",
    "¿cómo saco una constancia de ingreso?",
    "necesito demostrar que ingresé a la universidad",
    "¿cómo saco la constancia de tercio superior?",
    "¿cómo demuestro que estoy en el quinto superior?",
    "¿qué necesito para una constancia de buena conducta?",
    "¿quién expide la constancia de buena conducta?",
    "necesito una constancia que acredite mis créditos acumulados",
    "constancia para egresados",
    "¿cómo saco mi ficha de seguimiento académico?",
    "necesito el reporte de mis notas por semestre",
    "¿cuánto cuesta la ficha de seguimiento académico?",

    # --- Carné universitario / biblioteca ---
    "¿cómo saco mi carné de estudiante?",
    "requisitos para el carné universitario",
    "¿qué tamaño de foto pido para el carné?",
    "perdí mi carné, ¿cómo saco un duplicado?",
    "¿cómo tramito mi carné de biblioteca?",
    "se me deterioró mi carné de biblioteca, ¿qué hago?",
    "costo del duplicado del carné de biblioteca",

    # --- Sílabos / duplicados de documentos ---
    "necesito una copia visada de mi sílabo",
    "¿para qué sirve el sílabo visado?",
    "perdí mi constancia de matrícula, ¿cómo saco otra?",
    "necesito un duplicado de mi constancia de notas",
    "¿cómo corrijo mi nombre en el sistema del centro de cómputo?",
    "mi nombre está mal escrito en mis registros académicos",

    # --- Carta de presentación / prácticas ---
    "necesito una carta para hacer mis prácticas",
    "¿quién emite la carta de presentación para prácticas preprofesionales?",
    "¿cómo solicito la carta del decano para prácticas?",

    # --- Rectificación de nombre legal ---
    "cambié de nombre por trámite legal, ¿cómo actualizo mis documentos?",
    "necesito corregir mi nombre por error notarial",
    "¿qué documentos pide la rectificación de nombre?",

    # --- Bachillerato ---
    "ya terminé mis cursos, ¿cómo saco el bachillerato?",
    "requisitos para ser declarado apto para el grado de bachiller",
    "¿necesito certificado de inglés para el bachillerato?",
    "¿necesito certificado de computación para el bachillerato?",
    "¿cuántas fotos necesito para el trámite de bachiller?",
    "costo del bachillerato",
    "tiempo de demora del trámite de bachiller",

    # --- Título profesional (las 3 modalidades) ---
    "¿cuáles son las modalidades para obtener el título profesional?",
    "diferencia entre sustentación de tesis y suficiencia profesional",
    "¿qué es la modalidad de servicios a nivel profesional?",
    "¿puedo titularme por experiencia laboral?",
    "¿cuántos años de experiencia necesito para titularme por servicios profesionales?",
    "¿desde cuándo ya no se puede titular por servicios a nivel profesional?",
    "costo de titularse por sustentación de tesis",
    "costo de titularse por suficiencia profesional",
    "¿cuál modalidad de título es más barata?",
    "requisitos generales para sacar el título profesional",

    # --- Proceso de tesis (todo el flujo) ---
    "¿cómo inscribo el tema de mi tesis?",
    "¿cómo me asignan un asesor de tesis?",
    "¿qué porcentaje de créditos necesito para inscribir mi plan de tesis?",
    "quiero cambiar mi plan de tesis, ¿qué trámite hago?",
    "¿cómo se aprueba el dictamen de mi tesis?",
    "¿quiénes son los dictaminadores de tesis?",
    "¿cómo se nombran los dictaminadores?",
    "necesito el certificado de originalidad de mi tesis",
    "¿qué es el sistema antiplagio para tesis?",
    "¿cómo programo la fecha de mi sustentación?",
    "¿cuántos ejemplares de tesis necesito para sustentar?",
    "¿cómo obtengo mi diploma de título después de sustentar?",
    "¿qué es el rotulado del diploma?",
    "¿necesito subir mi tesis al repositorio institucional?",
    "¿cuántas tesis empastadas debo entregar?",

    # --- Juramentación ---
    "¿qué es el acto de juramentación?",
    "¿cómo me preparo para mi colación?",
    "costo de la toga y birrete para la juramentación",
    "¿la juramentación aplica también para posgrado?",

    # --- Sobre costos en general ---
    "¿cuál es el trámite más caro de todos?",
    "¿cuál es el trámite más barato?",
    "¿cuánto cuesta en total titularse por tesis incluyendo bachillerato?",
    "lista de trámites con su costo",
    "¿los pagos de trámites varían según el año de ingreso?",

    # --- Sobre tiempos de atención ---
    "¿cuál es el trámite que demora más tiempo?",
    "¿qué trámites se resuelven en un solo día?",
    "tiempo de espera para un certificado de estudios",
    "¿el traslado externo es el que demora más?",

    # --- Sobre dónde se hace cada trámite ---
    "¿en qué página web hago mis trámites?",
    "¿qué trámites se hacen en pladdes.unsaac.edu.pe?",
    "¿qué trámites se hacen en el centro de cómputo?",
    "¿dónde tramito mi carné universitario?",
    "¿qué trámites pasan por decanato y no por el rector?",

    # --- Documentos genéricos recurrentes ---
    "¿qué es la ficha de seguimiento académico y para qué trámites se pide?",
    "¿en cuántos trámites necesito la ficha de seguimiento académico?",
    "¿todos los trámites requieren solicitud al rector?",
    "¿qué trámites no necesitan pago?",

    # --- Coloquial / informal ---
    "como saco mi certificado de estudios",
    "que necesito para sacar mi bachiller",
    "cuanto cuesta el carnet de la u",
    "perdi mi carnet que hago",
    "como jalo un curso dirigido",
    "me quiero trasladar a otra escuela como hago",
    "quiero pausar mis estudios un semestre",
    "vengo de otra uni como convalido mis cursos",
    "cuanto sale titularse por tesis",
    "como saco mi constancia de tercio superior",

    # --- Con errores ortográficos ---
    "como tramito mi vachillerato",
    "q nesesito para la matricula",
    "konvalidacion de cursos de otra universidad",
    "cuanto kuesta el traslado externo",
    "komo saco mi carnet universitario",

    # --- Muy cortas ---
    "matrícula",
    "bachillerato",
    "traslado externo",
    "traslado interno",
    "convalidación",
    "homologación",
    "carné universitario",
    "constancia de estudios",
    "certificado de estudios",
    "reserva de matrícula",
    "título profesional",
    "sustentación de tesis",
    "costo de matrícula",

    # --- Con saludo ---
    "hola, ¿cómo saco mi certificado de estudios?",
    "buenos días, necesito tramitar mi bachillerato",
    "disculpa, ¿cómo me traslado a otra facultad?",
    "hola, ¿qué necesito para convalidar mis cursos?",
    "buenas, quisiera saber cómo se hace la reserva de matrícula",

    ],
    "tutorias": [
        "¿cómo es el proceso de tutorías?",
        "¿qué tipos de tutorías existen?",
        "¿qué es la tutoría individual?",
        "¿quién es mi docente tutor?",
        "¿para qué sirven las tutorías?",
        "¿cómo funciona el sistema de tutorías?",
        "¿dónde se registran las tutorías?",
        "¿qué pasa si necesito apoyo psicológico en tutoría?",
        "información sobre el reglamento de tutorías",
        "quiero saber sobre tutorías",
        "¿cómo me asignan un tutor?",
        "¿qué hace el docente tutor?",
        "¿cada cuánto debo reunirme con mi tutor?",
        "¿con qué frecuencia se hacen las tutorías?",
        "¿cuándo se hacen las reuniones de tutoría?",
        "¿cuántos estudiantes tiene cada tutor?",
        "máximo de estudiantes por tutor",
        "¿puedo solicitar cambio de tutor?",
        "¿cómo cambio de tutor?",
        "¿quién aprueba el cambio de tutor?",
        "¿qué hace el Comité Tutorial?",
        "¿desde cuándo me asignan un tutor?",
        "¿hasta cuándo tengo tutor?",
        "¿qué pasa si falto a las tutorías?",
        "¿cómo reprogramo una tutoría?",
        "qué pasa si no asisto a la tutoría",
        "¿la tutoría es obligatoria?",
        "tutoría para matrícula condicionada",
        "expediente del tutorado",
        "informe semestral del tutor",
        "¿el tutor me deriva a Bienestar?",
    ],
    "practicas": [
        "¿cómo son las prácticas preprofesionales?",
        "¿qué requisitos necesito para las prácticas preprofesionales?",
        "¿cuántas horas de prácticas debo hacer?",
        "¿qué documentos piden para las prácticas?",
        "¿cuánto duran las prácticas preprofesionales?",
        "¿cómo se evalúan las prácticas?",
        "¿dónde puedo hacer mis prácticas?",
        "¿cuántos créditos necesito para prácticas?",
        "información sobre prácticas pre profesionales",
        "¿cómo presento mi informe de prácticas?",
        "modalidades de prácticas preprofesionales",
        "plazo para presentar el informe de prácticas",
        "¿dónde se tramitan las prácticas?",
        "PLADDES y prácticas",
        "convocatorias de prácticas en la UNSAAC",
        "convenios para prácticas preprofesionales",
        "modelos de convenio para prácticas",
        "evaluación e informe de prácticas",
    ],
    "temas_tesis": [
        "¿cuáles son los temas de tesis disponibles?",
        "¿qué temas de tesis hay aprobados?",
        "¿puedo hacer una tesis de inteligencia artificial?",
        "¿qué temas de tesis hay de machine learning?",
        "temas de investigación de la Escuela",
        "¿sobre qué puedo hacer mi tesis?",
        "líneas de investigación disponibles",
        "temas de tesis de ciberseguridad",
        "¿qué temas hay para tesis de datos?",
        "¿cómo registro mi proyecto de tesis?",
        "áreas de investigación de la carrera",
        "¿hay tesis sobre IoT?",
        "¿hay tesis sobre big data?",
        "tesis del repositorio UNSAAC",
        "¿dónde veo las tesis publicadas?",
        "¿cómo accedo al repositorio de tesis?",
        "tesis recientes de la EPIIS",
        "¿qué grupos de investigación hay?",
        "semilleros de investigación",
        "¿hay convocatorias de investigación del VRIN?",
        "investigación en la EPIIS",
        "proyectos de investigación de la Escuela",
        "¿publican tesis docentes de la EPIIS?",
        "subvenciones de investigación",
    ],
    "titulacion": [
        "¿cuáles son las modalidades de titulación?",
        "¿de qué formas me puedo titular?",
        "¿cuántas formas de titulación existen?",
        "modalidades para obtener el título profesional",
        "¿qué es la sustentación de tesis?",
        "¿qué es el trabajo de suficiencia profesional?",
        "¿qué son los servicios a nivel profesional?",
        "diferencia entre tesis y suficiencia profesional",
        "¿qué es un proyecto aplicativo?",
        "diferencia entre tesis y proyecto aplicativo",
        "¿cómo obtengo el grado de bachiller?",
        "¿qué necesito para el bachillerato?",
        "requisitos del bachillerato",
        "¿el bachiller es automático en la UNSAAC?",
        "documentos para tramitar el bachiller",
        "fotos para el bachillerato",
        "pasos del trámite de bachiller",
        "¿cuántos créditos necesito para el bachiller?",
        "¿qué porcentaje de créditos necesito para el trabajo de investigación de bachiller?",
        "¿cómo registro mi plan de tesis?",
        "requisitos del plan de tesis",
        "¿cuánto dura la vigencia del plan de tesis?",
        "¿puedo hacer tesis en grupo?",
        "¿cuántas personas pueden hacer una tesis colectiva?",
        "¿qué porcentaje de créditos necesito para inscribir mi plan de tesis?",
        "vigencia del plan de tesis",
        "contenido del plan de tesis",
        "¿cuánto cuesta titularse por tesis?",
        "¿cuánto cuesta el trabajo de suficiencia profesional?",
        "¿cuánto cuesta la modalidad de servicios profesionales?",
        "costos del bachillerato",
        "tasas TUPA para titulación",
        "¿cuánto cuesta el dictamen de tesis?",
        "¿cuánto demora titularse?",
        "plazo de atención del trámite de bachiller",
        "plazo de la sustentación de tesis",
        "¿quién es el jurado de la tesis?",
        "¿cómo se asigna el jurado de tesis?",
        "¿cuánto dura la sustentación de tesis?",
        "tiempo de exposición en la sustentación",
        "¿qué pasa si me desaprueban en la sustentación?",
        "calificación de la tesis",
        "¿dónde se publican las tesis?",
        "¿cómo subo mi tesis al repositorio?",
        "publicación obligatoria de la tesis",
        "RENATI y la UNSAAC",
    ],
    "ingresantes": [
        "¿cuántos ingresantes hubo este año?",
        "información para ingresantes",
        "¿cuáles son las modalidades de admisión?",
        "¿cuál fue la nota más alta de ingreso?",
        "¿cuántos ingresaron por CEPRU?",
        "estadísticas de ingresantes",
        "¿cuántos ingresaron por examen de admisión?",
        "¿qué modalidades de ingreso existen?",
        "datos de admisión de la Escuela",
        "¿cuántos estudiantes ingresaron en 2025?",
        "información sobre el proceso de admisión",
        "¿qué becas hay para ingresantes?",
        "modalidad de ingreso ordinario",
        "modalidad primera oportunidad",
        "modalidad deportistas calificados",
        "¿hay ingreso por deportista calificado?",
        "ingreso por modalidad deportistas",
        "vacantes para deportistas calificados",
    ],
    "servicios": [
        "¿qué servicios universitarios hay?",
        "información sobre bienestar universitario",
        "¿cómo accedo a movilidad estudiantil?",
        "¿hay atención psicológica para estudiantes?",
        "¿dónde queda el comedor universitario?",
        "¿qué servicios de salud ofrece la universidad?",
        "¿cómo uso la biblioteca?",
        "¿hay actividades deportivas?",
        "información sobre el seguro estudiantil",
        "¿qué becas ofrece la universidad?",
        "servicios para estudiantes de la UNSAAC",
        "¿cómo solicito apoyo socioeconómico?",
        "¿hay centro universitario de salud?",
        "Centro Universitario de Salud",
        "atención médica gratuita",
        "horarios del centro de salud",
        "consultas odontológicas en la UNSAAC",
        "primeros auxilios para estudiantes",
        "horario del comedor universitario",
        "¿cómo accedo al comedor?",
        "¿el comedor es gratis?",
        "becas de alimentación",
        "menú del comedor universitario",
        "¿qué becas tiene la UNSAAC?",
        "¿hay becas de posgrado?",
        "becas de posgrado UNISC Brasil",
        "convocatorias de becas vigentes",
        "becas a través de OCTI",
        "becas PRONABEC en la UNSAAC",
        "¿cómo postulo a Beca 18?",
        "Beca Permanencia",
        "becas de cooperación internacional",
        "actividades deportivas de la UNSAAC",
        "¿qué deportes puedo practicar en la universidad?",
        "talleres deportivos",
        "UNSAAC deportes Facebook",
        "ingreso por deportista calificado",
        "¿hay vivienda estudiantil?",
        "alojamiento para estudiantes",
        "vivienda universitaria",
        "programa de movilidad estudiantil",
        "¿la UNSAAC participa en RPU?",
        "intercambio con otras universidades peruanas",
        "movilidad académica interuniversitaria",
        "¿cómo postulo a movilidad estudiantil?",
        "convocatorias de movilidad",
        "convenios institucionales de la UNSAAC",
        "convenios con empresas para estudiantes",
        "¿qué convenios tiene la universidad?",
        "redes de cooperación de la UNSAAC",
        "Oficina de Cooperación OCTI",
        "¿hay laboratorios de cómputo?",
        "laboratorios de informática",
        "¿qué software académico tiene la universidad?",
        "licencias de software para estudiantes",
        "biblioteca virtual UNSAAC",
        "¿cómo accedo al repositorio digital?",
        "¿cómo activo mi correo institucional?",
        "plataformas académicas de la UNSAAC",
        "Internet en el campus",
        "wifi del campus universitario",
        "salas de estudio en la biblioteca",
        "¿hay defensoría universitaria?",
        "denuncias de acoso en la UNSAAC",
    ],
    "contacto": [
        "¿con quién me puedo contactar?",
        "¿cuál es el correo de la escuela?",
        "¿cuál es el número de teléfono de la escuela?",
        "¿cómo contacto a la secretaría académica?",
        "datos de contacto de la escuela",
        "¿cómo me comunico con la dirección?",
        "¿cuál es el email de la directora?",
        "teléfono de la Escuela Profesional",
        "contacto de la Escuela de Informática",
        "¿a quién le escribo para consultas?",
        "anexo telefónico de la escuela",
        "horario de atención al público",
        "¿cómo contacto al Departamento Académico?",
        "número del Departamento Académico de Informática",
        "correo institucional de la EPIIS",
        "directorio institucional",
        "¿cómo llamo a la escuela?",
        "datos de contacto del director",
        "¿cuál es el anexo de la escuela?",
        "Teléfono 084-232398 anexo",
        "correo de Yeshica Ormeño",
        "correo del jefe de departamento",
    ],
    "elegibilidad": [
        "¿puedo iniciar mis prácticas preprofesionales?",
        "tengo 175 créditos y seguro vigente, ¿puedo iniciar prácticas?",
        "tengo 160 créditos, ¿ya puedo hacer prácticas?",
        "ya aprobé el VIII semestre, ¿puedo hacer prácticas?",
        "¿soy elegible para iniciar las prácticas?",
        "tengo 200 créditos pero no tengo seguro, ¿puedo iniciar prácticas?",
        "tengo 170 créditos y SOAT vigente, ¿puedo empezar prácticas?",
        "¿cumplo los requisitos para mis prácticas?",
        "¿estoy listo para iniciar prácticas preprofesionales?",
        "¿qué necesito para empezar mis prácticas?",
        "¿puedo iniciar las prácticas pre profesionales?",
        "soy del IX semestre, ¿puedo hacer prácticas?",
        "tengo todos los certificados, ¿puedo tramitar bachiller?",
        "tengo idioma extranjero y computación básica, ¿puedo iniciar el bachiller?",
        "tengo plan de estudios completo y trabajo de investigación sustentado, ¿soy elegible para bachiller?",
        "ya terminé el plan, tengo prácticas y certificados, ¿puedo tramitar el bachiller?",
        "¿soy elegible para el bachillerato?",
        "tengo el certificado de idioma pero no el de computación, ¿puedo iniciar el bachiller?",
        "tengo bachiller y plan de tesis avalado, ¿puedo titularme por tesis?",
        "ingresé en 2013, tengo bachiller y 4 años de experiencia, ¿qué modalidad de titulación tengo?",
        "tengo bachiller, ¿puedo hacer el trabajo de suficiencia profesional?",
        "tengo 80% de créditos, ¿puedo inscribir mi plan de tesis?",
        "tengo 70% de créditos, ¿puedo inscribir mi trabajo de investigación?",
        "¿qué modalidad de titulación me corresponde?",
        "¿puedo titularme por servicios profesionales?",
        "ingresé en 2010, tengo bachiller y 5 años trabajando, ¿puedo titularme por experiencia?",
        "estoy llevando 14 créditos, ¿soy estudiante regular?",
        "llevo 10 créditos este semestre, ¿soy regular?",
        "matriculé 18 créditos y mi promedio es 17, ¿hasta cuántos créditos puedo llevar?",
        "tengo promedio 16.5 y llevo 20 créditos, ¿puedo aumentar mi carga?",
        "¿hasta cuántos créditos puedo matricularme?",
        "¿soy estudiante regular si llevo menos de 12 créditos?",
        "matriculé 20 créditos y mi promedio es 16.5, ¿cuál es mi tope de créditos?",
        "matriculé 15 créditos, ¿soy regular?",
        "este semestre llevo 18 créditos con promedio 17, ¿puedo subir mi carga?",
        "matriculé 22 créditos, ¿es mi tope?",
        "con promedio 16, ¿hasta qué tope de créditos llego?",
        "llevo 19 créditos y mi promedio ponderado es 17.5, dame mi tope",
        "tengo promedio 14 y llevo 20 créditos, ¿cuál es mi tope?",
        "matriculé 11 créditos este semestre, ¿soy no regular?",
    ],
    "derivar_a_escuela": [
        "¿cuál es el horario de atención de un docente?",
        "¿qué horario tiene el profesor X para tutorías?",
        "¿cuál es la carga académica del semestre actual?",
        "¿qué cursos dicta el profesor Carrasco este semestre?",
        "¿cuántas horas exactas de prácticas exige la EPIIS?",
        "¿qué convenios tiene EPIIS específicamente para sus estudiantes?",
        "lista actual de semilleros de la Escuela",
        "docentes investigadores actuales de la EPIIS",
        "formato vigente de informe de prácticas",
        "última versión del formato de titulación",
        "¿cuál es el horario de atención del profesor Carrasco?",
        "¿qué horarios tiene cada docente?",
        "horario individual del docente",
        "horario personalizado de un profesor",
        "carga académica vigente del semestre",
        "qué cursos enseña cada profesor este ciclo",
    ],
    "despedida": [
        "adiós",
        "gracias",
        "muchas gracias",
        "hasta luego",
        "chau",
        "nos vemos",
        "eso es todo",
        "gracias por la ayuda",
        "hasta pronto",
        "ya terminé",
        "ok gracias",
        "perfecto gracias",
        "thanks",
        "bye",
        "hasta la próxima",
        "todo claro",
    ],
}

total = sum(len(v) for v in CORPUS.values())
print(f"Corpus: {len(CORPUS)} intenciones, {total} ejemplos.")
for k, v in CORPUS.items():
    print(f"  {k:<22}: {len(v)} ejemplos")

Corpus: 15 intenciones, 1931 ejemplos.
  saludo                : 72 ejemplos
  informacion_general   : 520 ejemplos
  docentes              : 214 ejemplos
  cursos                : 645 ejemplos
  tramites              : 185 ejemplos
  tutorias              : 31 ejemplos
  practicas             : 18 ejemplos
  temas_tesis           : 24 ejemplos
  titulacion            : 46 ejemplos
  ingresantes           : 18 ejemplos
  servicios             : 64 ejemplos
  contacto              : 22 ejemplos
  elegibilidad          : 40 ejemplos
  derivar_a_escuela     : 16 ejemplos
  despedida             : 16 ejemplos


## 4. Clase `ModeloClasificador` — TF-IDF + Random Forest

TF-IDF combinado (palabras 1-2 + caracteres 3-5) y Random Forest de 300 árboles con
`class_weight="balanced"` para compensar el desbalance entre intenciones.

**En simple:** esta clase es el "detector de temas". Recibe el texto que escribe el usuario y adivina de qué intención se trata (por ejemplo: ¿está preguntando por docentes, por trámites, o por elegibilidad para prácticas?). Funciona en dos pasos: primero convierte las palabras en números (TF-IDF), y luego usa esos números para que un modelo de 300 "árboles de decisión" (Random Forest) vote cuál es la intención más probable, junto con qué tan seguro está de esa respuesta (confianza).

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 6: CLASE ModeloClasificador (PIPELINE NLP + ML)
# ─────────────────────────────────────────────────────────────────────
# Encapsula todo el componente de procesamiento de lenguaje natural y
# aprendizaje automático del chatbot. Pipeline completo:
#   1. _normalizar()     → minúsculas, quita tildes y signos de puntuación
#   2. _cargar_stopwords → carga palabras vacías del español (NLTK)
#   3. Vectorización TF-IDF combinada (FeatureUnion):
#      - n-gramas de PALABRAS (1-2) → captura términos y bigramas
#      - n-gramas de CARACTERES (3-5) → robustez ante errores de tipeo
#        y variaciones morfológicas del español
#   4. RandomForestClassifier (300 árboles, class_weight="balanced")
#      → clasifica la intención de la pregunta del usuario
# Métodos principales:
#   - entrenar(corpus) → entrena con 75/25 split, reporta exactitud,
#                         luego reentrena con el 100% del corpus
#   - predecir(texto)  → devuelve (intención, confianza)
#                         Si confianza < umbral (0.20), el bot pide
#                         que reformulen la pregunta
# ═══════════════════════════════════════════════════════════════════════
class ModeloClasificador:
    def __init__(self, n_estimators: int = 300, umbral_confianza: float = 0.20):
        self.umbral = umbral_confianza
        self._stopwords = self._cargar_stopwords()
        self._vectorizador = FeatureUnion([
            ("palabras", TfidfVectorizer(preprocessor=self._normalizar,
                stop_words=self._stopwords, ngram_range=(1, 2), sublinear_tf=True)),
            ("caracteres", TfidfVectorizer(preprocessor=self._normalizar,
                analyzer="char_wb", ngram_range=(3, 5), sublinear_tf=True)),
        ])
        self._modelo = RandomForestClassifier(
            n_estimators=n_estimators, random_state=42, class_weight="balanced")
        self._entrenado = False

    @staticmethod
    def _normalizar(texto: str) -> str:
        texto = str(texto).lower()
        texto = texto.translate(str.maketrans("áéíóúüñ", "aeiouun"))
        texto = re.sub(r"[^a-z0-9\s]", " ", texto)
        return re.sub(r"\s+", " ", texto).strip()

    def _cargar_stopwords(self) -> List[str]:
        try:
            from nltk.corpus import stopwords
            base = stopwords.words("spanish")
        except Exception:
            base = ["de","la","que","el","en","y","a","los","del","se","las","por",
                    "un","para","con","una","su","al","es","lo","como","mas","o"]
        return [w.translate(str.maketrans("áéíóúüñ","aeiouun")) for w in base]

    def entrenar(self, corpus: dict) -> None:
        X, y = [], []
        for intent, ejemplos in corpus.items():
            for q in ejemplos: X.append(q); y.append(intent)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y)
        M = self._vectorizador.fit_transform(X_tr); self._modelo.fit(M, y_tr)
        pred = self._modelo.predict(self._vectorizador.transform(X_te))
        print(f"Exactitud en validación: {accuracy_score(y_te, pred):.3f}")
        M_full = self._vectorizador.fit_transform(X); self._modelo.fit(M_full, y)
        self._entrenado = True
        print(f"Modelo entrenado con {len(X)} ejemplos, {len(self._modelo.classes_)} intenciones.")

    def predecir(self, texto: str) -> Tuple[str, float]:
        if not self._entrenado: raise RuntimeError("modelo no entrenado")
        v = self._vectorizador.transform([texto])
        p = self._modelo.predict_proba(v)[0]
        i = p.argmax()
        return self._modelo.classes_[i], float(p[i])

print("Clase ModeloClasificador definida.")


Clase ModeloClasificador definida.


## 4.1 Motor de Inferencia (TELL · ASK · INFER · EXPLAIN)

 Es el componente que convierte al chatbot en un
*agente basado en conocimiento*: además de recuperar hechos (ASK) puede **derivar
conclusiones nuevas** combinando reglas (forward chaining).



In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 7: CLASE MotorInferencia (FORWARD CHAINING)
# ─────────────────────────────────────────────────────────────────────
# Motor de inferencia por encadenamiento hacia adelante. Es el componente
# que convierte al chatbot en un agente basado en conocimiento: además
# de recuperar hechos puede DERIVAR conclusiones nuevas combinando reglas.
# Operaciones (equivalentes a un agente lógico):
#   - decir(hecho, valor)  → TELL: agrega un axioma a la base de hechos
#   - preguntar(hecho)     → ASK: consulta si un hecho es verdadero
#   - inferir()            → INFER: aplica reglas hasta punto fijo
#                            (dispara reglas cuyas premisas se cumplen,
#                             repite hasta que no hay nuevos hechos)
#   - explicar()           → EXPLAIN: devuelve la traza completa del
#                            razonamiento (axiomas + reglas disparadas)
# Cada regla tiene formato: (nombre, [premisas], conclusión)
# Solo GestorElegibilidad usa este motor; los demás módulos no lo
# necesitan porque sus respuestas son hechos únicos de la BD (ASK puro).
# ═══════════════════════════════════════════════════════════════════════
class MotorInferencia:
    """Motor de inferencia por encadenamiento hacia adelante (forward chaining)."""

    def __init__(self, reglas: List[Tuple[str, List[str], str]]):
        self._reglas = reglas
        self._hechos: dict = {}
        self._traza: List[str] = []

    def decir(self, hecho: str, valor: bool = True) -> None:
        """TELL: agrega un axioma a la base de hechos."""
        self._hechos[hecho] = valor
        self._traza.append(f"AXIOMA: {hecho} = {valor}")

    def preguntar(self, hecho: str) -> bool:
        """ASK: consulta si un hecho es verdadero."""
        return bool(self._hechos.get(hecho, False))

    def inferir(self) -> List[str]:
        """Aplica reglas hasta punto fijo."""
        cambio = True
        while cambio:
            cambio = False
            for nombre, premisas, conclusion in self._reglas:
                if self._hechos.get(conclusion): continue
                if premisas and all(self._hechos.get(p) for p in premisas):
                    self._hechos[conclusion] = True
                    self._traza.append(
                        f"{nombre}: SI {' Y '.join(premisas)} ENTONCES {conclusion}")
                    cambio = True
        return self._traza

    def explicar(self) -> str:
        if not self._traza:
            return "  (sin hechos suficientes para iniciar el razonamiento)"
        return "\n".join(f"  {paso}" for paso in self._traza)

print("Clase MotorInferencia definida.")


Clase MotorInferencia definida.


## 5. Módulos de conocimiento (POO)



In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 8: MÓDULOS DE CONOCIMIENTO (CLASES POO — ASK PURO)
# ─────────────────────────────────────────────────────────────────────
# Cada clase encapsula un dominio temático del chatbot. Todas reciben
# una instancia de BaseDatos (inyección de dependencias) y exponen
# un método responder(texto) que consulta el repositorio y devuelve
# la respuesta Clases definidas aquí:
#
#   1. InformacionGeneral  → misión, visión, historia, directora,
#                            acreditación, perfil de egresado, campo laboral
#   2. DirectorioDocentes  → lista/filtra docentes por dedicación o nombre
#   3. RepositorioCursos   → cursos por semestre, área, créditos, electivos
#   4. GestorHorarios      → horarios de cursos y docentes (JOIN con horarios_cursos)
#   5. GestorTramites      → 42 trámites TUPA con costos, requisitos y plazos
#   6. GestorTesis         → temas de tesis, repositorio, grupos de investigación
#   7. GestorTitulacion    → modalidades de titulación, costos TUPA, jurado
#   8. ModuloIngresantes   → estadísticas de admisión, modalidades, deportistas
#   9. GestorTutorias      → proceso, tipos, horarios, tutor asignado, problemas
#  10. GestorServicios     → salud, comedor, becas, deportes, vivienda, movilidad,
#                            convenios, recursos tecnológicos, bienestar, defensoría
#
# NINGUNO de estos módulos usa el MotorInferencia. Todos son ASK puro:
# cada respuesta es un hecho que ya existe en una fila de la BD.
# La función auxiliar _sin_tildes() normaliza el texto para la búsqueda.
# ═══════════════════════════════════════════════════════════════════════
def _sin_tildes(t: str) -> str:
    return str(t).lower().translate(str.maketrans("áéíóúüñ", "aeiouun"))


# =====================================================================
class InformacionGeneral:
    """Misión, visión, historia, autoridades, acreditación, perfil, campo laboral."""
    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def _por_titulo(self, like: str) -> Optional[str]:
        f = self._bd.consultar_uno(
            "SELECT contenido FROM informacion_general WHERE titulo LIKE ?",
            (f"%{like}%",))
        return f["contenido"] if f else None

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        mapeo = [
            (["mision"], "Misión"),
            (["vision"], "Visión"),
            (["historia","fundo","creo","creacion","cuando fue"], "Historia"),
            (["director","autoridad","quien dirige","quien es yeshica","ormeno"], "Directora"),
            (["jefe","departamento","flores pacheco","daii"], "Departamento Académico"),
            (["telefono","numero","anexo","llamo"], "Teléfono"),
            (["ubicacion","donde queda","donde esta","direccion","perayoc","fieeim"], "Ubicación"),
            (["dura","duracion","anios","semestres"], "Duración"),
            (["acredit","icacit","calidad"], "Acreditación"),
            (["perfil","egresado","competencia","habilidad"], "Perfil del egresado"),
            (["campo laboral","trabajan","en que puede trabajar","oportunidad laboral",
              "donde trabaj"], "Campo laboral"),
            (["sitio web","pagina","portal"], "Sitio web oficial"),
        ]
        for claves, titulo in mapeo:
            if any(k in t for k in claves):
                contenido = self._por_titulo(titulo)
                if contenido: return f"{titulo}:\n{contenido}"
        # default: resumen
        return ("Escuela Profesional de Ingeniería Informática y de Sistemas — UNSAAC.\n"
                f"Directora: {self._por_titulo('Directora')}.\n"
                f"Ubicación: {self._por_titulo('Ubicación')}.\n"
                f"Duración: {self._por_titulo('Duración')}.")


# =====================================================================
class DirectorioDocentes:
    """Cuerpo docente (ASK puro). Actualizado: nuevas dedicaciones (Tiempo Parcial,
    Jefe de Prácticas) y búsqueda por nombre tolerante a tildes/ñ y puntuación."""
    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        t_limpio = re.sub(r"[¿?¡!.,;:]", "", t)

        if "exclusiva" in t:
            r = self._bd.consultar(
                "SELECT nombre FROM docentes WHERE dedicacion LIKE '%Exclusiva%'")
            return "Docentes con Dedicación Exclusiva:\n" + "\n".join(f"  - {x['nombre']}" for x in r)

        if "jefe de practica" in t:
            r = self._bd.consultar(
                "SELECT nombre, dedicacion FROM docentes WHERE dedicacion LIKE '%Jefe de Pr%'")
            return "Jefes de Prácticas:\n" + "\n".join(
                f"  - {x['nombre']} ({x['dedicacion']})" for x in r)

        if "parcial" in t:
            r = self._bd.consultar(
                "SELECT nombre, dedicacion FROM docentes WHERE dedicacion LIKE '%Parcial%'")
            return "Docentes a Tiempo Parcial:\n" + "\n".join(
                f"  - {x['nombre']} ({x['dedicacion']})" for x in r)

        if "tiempo completo" in t or "completo" in t:
            r = self._bd.consultar(
                "SELECT nombre, dedicacion FROM docentes WHERE dedicacion LIKE '%Completo%'")
            return "Docentes a Tiempo Completo:\n" + "\n".join(
                f"  - {x['nombre']} ({x['dedicacion']})" for x in r)

        # Búsqueda por nombre tolerante a tildes/ñ y puntuación (se filtra en Python,
        # no en SQL, porque los nombres en la BD llevan acentos/ñ y el texto del
        # usuario no; además se descartan signos de puntuación pegados a la palabra
        # y palabras genéricas de la propia pregunta que nunca son parte de un nombre).
        todos = self._bd.consultar("SELECT nombre, dedicacion FROM docentes ORDER BY nombre")
        STOPWORDS = {"quien", "donde", "cual", "cuales", "docente", "docentes",
                     "profesor", "profesora", "profesores", "informacion", "datos"}
        for palabra in t_limpio.split():
            if len(palabra) >= 4 and palabra not in STOPWORDS:
                coincidencias = [x for x in todos if palabra in _sin_tildes(x["nombre"])]
                if coincidencias:
                    return "Resultado:\n" + "\n".join(
                        f"  - {x['nombre']} ({x['dedicacion']})" for x in coincidencias)

        return f"La Escuela cuenta con {len(todos)} docentes registrados:\n" + "\n".join(
            f"  - {x['nombre']} ({x['dedicacion']})" for x in todos)


# =====================================================================
class RepositorioCursos:
    """Catálogo de cursos. Resuelve la inconsistencia de prerrequisitos con prereq_creditos.
    Actualizado: la BD ahora contiene DOS planes curriculares (2017 y 2025) que comparten
    los mismos números de semestre, y semestres del 1 al 10 (antes solo se detectaban 6 y 7)."""

    _IA = ["INTELIGENCIA","APRENDIZAJE","REDES NEURONALES","REDES BAYESIANAS",
           "VISION COMPUTACIONAL","ROBOTICA","DEEP LEARNING","PROCESAMIENTO DEL LENGUAJE",
           "BUSQUEDA HEURISTICA","INTERACCION PERSONA COMPUTADOR","CIENCIA DE DATOS"]

    AREAS = {
        "inteligencia artificial": _IA,
        "machine learning": _IA,
        "aprendizaje automatico": _IA,
        "computacion grafica": ["GRÁFICA","GEOMETRIA COMPUTACIONAL","COMPUTACION CUANTICA",
                                  "COMPUTACION SIMBOLICA"],
        "software": ["SOFTWARE","METODOLOGIA","METODOLOGÍA","COMPILADORES",
                      "ANALISIS Y DISEÑO DE SISTEMAS","ANÁLISIS Y DISEÑO DE SISTEMAS"],
        "datos": ["BASE DE DATOS","BASES DE DATOS","DATOS","NUMERICOS","NUMÉRICOS"],
        "redes": ["REDES DE COMPUTADORAS","SISTEMAS OPERATIVOS","TOPICOS AVANZADOS EN REDES",
                   "ARQUITECTURAS DE ALTO RENDIMIENTO","SISTEMAS EMBEBIDOS","LENGUAJE ENSAMBLADOR"],
        "ciberseguridad": ["CIBERSEGURIDAD","AUDITORIA DE SISTEMAS"],
        "internet de las cosas": ["INTERNET DE LA"],
        "algoritmos": ["ALGORITMOS"],
        "bioinformatica": ["BIOINFORMATICA","BIOMOLECULARES","BIOMEDICAS","EXPRESION GENICA"],
    }

    _SEMESTRES = {
        "primero":1,"primer":1,"1ro":1, "segundo":2,"2do":2, "tercero":3,"tercer":3,"3ro":3,
        "cuarto":4,"4to":4, "quinto":5,"5to":5, "sexto":6,"6to":6,
        "septimo":7,"setimo":7,"7mo":7, "octavo":8,"8vo":8, "noveno":9,"9no":9,
        "decimo":10,"10mo":10,
    }

    def __init__(self, bd: BaseDatos):
        self._bd = bd

    @staticmethod
    def _fmt_prereq(prereq_str: str, prereq_cr: int) -> str:
        if prereq_cr and prereq_cr > 0:
            return f"{prereq_cr} créditos acumulados"
        return prereq_str if prereq_str else "ninguno"

    @classmethod
    def _detectar_semestre(cls, t: str) -> Optional[int]:
        for k, v in cls._SEMESTRES.items():
            if re.search(rf"\b{k}\b", t):
                return v
        m = re.search(r"\b(10|[1-9])\b", t)
        return int(m.group(1)) if m else None

    @staticmethod
    def _detectar_plan(t: str) -> Optional[str]:
        if "2017" in t or "plan antiguo" in t or "plan anterior" in t or "malla antigua" in t:
            return "PLAN CURRICULAR 2017"
        if "2025" in t or "plan nuevo" in t or "plan vigente" in t or "plan actual" in t or "malla nueva" in t:
            return "PLAN CURRICULAR 2025"
        return None

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)

        # Información general sobre el régimen (no depende de tabla)
        if "credito" in t and ("semestre" in t or "minimo" in t or "regular" in t):
            return ("Régimen académico (Reglamento UNSAAC):\n"
                    "  - 1 crédito = 16 horas teóricas o 32 horas prácticas (Ley 30220 Art. 39).\n"
                    "  - Estudiante regular: mínimo 12 créditos por semestre.\n"
                    "  - Estudiante regular puede matricular hasta 22 créditos (26 si su promedio ≥16).\n"
                    "  - Cada semestre dura 17 semanas (3 períodos lectivos de 5, 6 y 6 semanas).")
        if "electivo" in t:
            return ("Cursos electivos: si una asignatura desaprobada tiene carácter electivo y "
                    "sigue en el currículo, puedes optar entre cursarla de nuevo o llevar otra "
                    "electiva en su reemplazo (a diferencia de las obligatorias).")

        # Por semestre (ahora detecta 1-10 y desambigua entre plan 2017 / 2025)
        sem = self._detectar_semestre(t)
        if sem:
            plan = self._detectar_plan(t)
            if plan:
                f = self._bd.consultar(
                    "SELECT DISTINCT nombre, creditos, prerrequisitos, prereq_creditos FROM cursos "
                    "WHERE semestre=? AND curricula=? ORDER BY nombre", (sem, plan))
                if not f: return f"No hay cursos del semestre {sem} en el {plan}."
                return f"Cursos del semestre {sem} — {plan}:\n" + "\n".join(
                    f"  - {x['nombre']} ({x['creditos']} cr.; prereq: "
                    f"{self._fmt_prereq(x['prerrequisitos'], x['prereq_creditos'])})"
                    for x in f)
            f = self._bd.consultar(
                "SELECT DISTINCT nombre, creditos, prerrequisitos, prereq_creditos, curricula FROM cursos "
                "WHERE semestre=? ORDER BY curricula, nombre", (sem,))
            if not f: return f"No hay cursos del semestre {sem}."
            por_plan = {}
            for x in f:
                por_plan.setdefault(x["curricula"], []).append(x)
            partes = [f"Cursos del semestre {sem} (la Escuela tiene dos planes curriculares en la BD):"]
            for plan_nombre, filas in por_plan.items():
                partes.append(f"\n{plan_nombre}:")
                partes.extend(
                    f"  - {x['nombre']} ({x['creditos']} cr.; prereq: "
                    f"{self._fmt_prereq(x['prerrequisitos'], x['prereq_creditos'])})"
                    for x in filas)
            return "\n".join(partes)

        # Por área
        for area, patrones in self.AREAS.items():
            if area in t or area.split()[0] in t:
                cond = " OR ".join(["nombre LIKE ?"] * len(patrones))
                params = tuple(f"%{p}%" for p in patrones)
                f = self._bd.consultar(
                    f"SELECT DISTINCT nombre, creditos, semestre, curricula FROM cursos "
                    f"WHERE {cond} ORDER BY semestre", params)
                if f:
                    return f"Cursos del área de {area}:\n" + "\n".join(
                        f"  - {x['nombre']} (sem. {x['semestre'] if x['semestre'] is not None else 'electivo'}, "
                        f"{x['creditos']} cr., {x['curricula']})" for x in f)

        # default: catálogo completo, agrupado por plan
        f = self._bd.consultar(
            "SELECT DISTINCT nombre, creditos, semestre, curricula FROM cursos "
            "ORDER BY curricula, semestre, nombre")
        return ("Cursos disponibles (la Escuela maneja dos planes curriculares):\n"
                + "\n".join(
                    f"  - [{x['curricula']} | Sem {x['semestre'] if x['semestre'] is not None else 'electivo'}] "
                    f"{x['nombre']} ({x['creditos']} cr.)" for x in f))


# =====================================================================
class GestorHorarios:
    """NUEVO. Día, hora, docente, aula y sección por curso o por docente,
    a partir de la tabla horarios_cursos (antes no usada por ningún módulo)."""

    DIAS = {"LU": "Lunes", "MA": "Martes", "MI": "Miércoles", "JU": "Jueves", "VI": "Viernes"}
    TIPOS = {"T": "Teoría", "P": "Práctica", "L": "Laboratorio"}
    _ORDEN_DIA = "CASE dia WHEN 'LU' THEN 1 WHEN 'MA' THEN 2 WHEN 'MI' THEN 3 WHEN 'JU' THEN 4 WHEN 'VI' THEN 5 END"

    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def _fmt_curso_fila(self, x) -> str:
        dia = self.DIAS.get(x["dia"], x["dia"])
        tipo = self.TIPOS.get(x["tipo"], x["tipo"])
        return (f"  - {dia} {x['hora_inicio'][:5]}–{x['hora_fin'][:5]} "
                f"[{tipo}, secc. {x['seccion']}] docente: {x['docente']} — aula: {x['aula']}")

    def _fmt_docente_fila(self, x) -> str:
        dia = self.DIAS.get(x["dia"], x["dia"])
        tipo = self.TIPOS.get(x["tipo"], x["tipo"])
        return (f"  - {x['codigo_curso']} — {dia} {x['hora_inicio'][:5]}–{x['hora_fin'][:5]} "
                f"[{tipo}, secc. {x['seccion']}] — aula: {x['aula']}")

    def _resolver_codigo_curso(self, texto: str) -> Optional[str]:
        for token in re.findall(r"[A-Za-z]{2,6}-?\d{2,3}-?\d{0,2}", texto):
            f = self._bd.consultar_uno("SELECT codigo FROM cursos WHERE codigo = ?", (token.upper(),))
            if f: return f["codigo"]
        t = _sin_tildes(texto)
        palabras_t = {p for p in t.split() if len(p) >= 5}
        if not palabras_t: return None
        cursos = self._bd.consultar("SELECT DISTINCT codigo, nombre FROM cursos")
        mejor, mejor_score = None, 0
        for c in cursos:
            palabras_c = {p for p in _sin_tildes(c["nombre"]).split() if len(p) >= 5}
            score = len(palabras_t & palabras_c)
            if score > mejor_score:
                mejor, mejor_score = c["codigo"], score
        return mejor

    def responder(self, texto: str) -> str:
        codigo = self._resolver_codigo_curso(texto)
        if codigo:
            curso = self._bd.consultar_uno("SELECT nombre FROM cursos WHERE codigo=?", (codigo,))
            filas = self._bd.consultar(
                "SELECT seccion, docente, dia, hora_inicio, hora_fin, tipo, aula FROM horarios_cursos "
                f"WHERE codigo_curso=? ORDER BY seccion, {self._ORDEN_DIA}, hora_inicio", (codigo,))
            nombre = curso["nombre"] if curso else codigo
            if not filas:
                return f"No hay horario registrado para {nombre} ({codigo})."
            return f"Horario de {nombre} ({codigo}):\n" + "\n".join(self._fmt_curso_fila(x) for x in filas)

        # Intento por nombre de docente (tolerante a tildes)
        t = _sin_tildes(texto)
        palabras = [p for p in t.split() if len(p) >= 4]
        if palabras:
            docentes = self._bd.consultar("SELECT DISTINCT docente FROM horarios_cursos")
            for d in docentes:
                if any(p in _sin_tildes(d["docente"]) for p in palabras):
                    filas = self._bd.consultar(
                        "SELECT codigo_curso, seccion, dia, hora_inicio, hora_fin, tipo, aula "
                        f"FROM horarios_cursos WHERE docente=? ORDER BY {self._ORDEN_DIA}, hora_inicio",
                        (d["docente"],))
                    return f"Horario de {d['docente']}:\n" + "\n".join(
                        self._fmt_docente_fila(x) for x in filas)

        return ("No identifiqué el curso ni el docente. Indícame el código del curso "
                "(p. ej. 'IF458'), su nombre completo, o el nombre del docente.")


# =====================================================================
class GestorTramites:
    """Trámites académicos (ASK puro). Reescrito: la tabla tramites pasó de 12 a 42
    filas con descripciones nuevas (catálogo TUPA); el mapeo anterior ya no encontraba
    casi ninguna coincidencia real."""

    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def _frag(self, fragmento: str):
        return self._bd.consultar_uno(
            "SELECT * FROM tramites WHERE descripcion LIKE ?", (f"%{fragmento}%",))

    def _formatear(self, fila) -> str:
        partes = [f"**{fila['descripcion']}**", fila['detalle']]
        if fila['requisitos']:  partes.append(f"Requisitos: {fila['requisitos']}")
        if fila['plazo']:       partes.append(f"Plazo: {fila['plazo']}")
        if fila['costo']:       partes.append(f"Costo: {fila['costo']}")
        if fila['enlace']:      partes.append(f"Más info: {fila['enlace']}")
        return "\n".join(partes)

    # Cada entrada: (claves_any, claves_all, fragmento_descripcion)
    # - claves_any: basta que UNA esté presente (sinónimos del mismo concepto)
    # - claves_all: deben estar TODAS presentes (combinación de dos conceptos)
    MAPA = [
        (["matricula condicionada","bajo rendimiento"], [], "condicionada por bajo rendimiento"),
        (["curso dirigido","cursos dirigidos"], [], "cursos dirigidos"),
        (["matricula ingresante","matricula de ingresante","soy ingresante"], [], "matrícula de ingresante"),
        (["traslado externo"], [], "Traslado externo"),
        (["titulado","graduado","segunda profesion","segunda carrera"], [], "titulados o graduados"),
        (["subsana"], [], "Subsanación de asignaturas"),
        (["buena conducta"], [], "buena conducta"),
        (["centro de computo","ccomputo"], [], "base de datos del Centro de Cómputo"),
        (["rectifica"], [], "Rectificación de nombre"),
        (["certificado de notas","certificado de estudios"], [], "Certificado de estudios"),
        (["egresado","creditos acumulados"], [], "créditos acumulados"),
        (["tercio","quinto superior","decimo superior","orden de merito"], [], "quinto y décimo superior"),
        (["no deudor","no ser deudor"], [], "no ser deudor"),
        (["constancia de ingreso"], [], "Constancia de ingreso"),
        (["constancia de estudios"], [], "Constancia de estudios"),
        (["ficha de seguimiento"], [], "Ficha de seguimiento"),
        ([], ["duplicado","carne universitario"], "Duplicado de carné universitario"),
        (["carne universitario","carnet universitario"], [], "Carné universitario"),
        ([], ["duplicado","biblioteca"], "Duplicado de carné de biblioteca"),
        (["carne de biblioteca","carnet de biblioteca"], [], "Carné de biblioteca"),
        ([], ["duplicado","matricula"], "constancia de matrícula"),
        ([], ["duplicado","notas"], "constancia de notas"),
        (["silabo"], [], "sílabos"),
        (["carta de presentacion"], [], "Carta de presentación"),
        (["convalida"], [], "Convalidación de asignaturas"),
        (["homologa"], [], "Homologación de asignaturas"),
        (["reserva"], [], "Reserva de matrícula"),
        (["reinicio"], [], "Reinicio de estudios"),
        (["bachiller"], [], "Bachiller"),
        (["sustentacion de tesis","titulo por tesis"], [], "Sustentación de Tesis"),
        (["suficiencia profesional"], [], "Suficiencia Profesional"),
        (["servicios a nivel profesional","experiencia profesional"], [], "Servicios a Nivel Profesional"),
        (["modificacion del plan de tesis","modificar mi tesis","modificar plan de tesis"], [], "Modificación del plan de tesis"),
        (["aprobacion de dictamen","dictamen de tesis"], [], "dictamen"),
        (["dictaminador"], [], "dictaminadores de tesis"),
        (["nombramiento de asesor","inscribir mi tema","tema de tesis"], [], "nombramiento de asesor"),
        (["fecha de sustentacion","hora y lugar","lugar de sustentacion"], [], "hora y lugar de sustentación"),
        (["rotulado","diploma"], [], "Rotulado de diploma"),
        (["juramentacion","colacion","promesa de honor"], [], "juramentación"),
        (["matricula"], [], "alumnos regulares"),  # catch-all de matrícula, va al final
    ]

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)

        # Traslado interno: hay dos variantes (misma Facultad / otra Facultad)
        if "traslado interno" in t:
            if "misma facultad" in t or "misma escuela" in t:
                f = self._frag("Traslado interno (misma")
                if f: return self._formatear(f)
            if "otra facultad" in t:
                f = self._frag("Traslado interno (otra")
                if f: return self._formatear(f)
            filas = self._bd.consultar(
                "SELECT * FROM tramites WHERE descripcion LIKE 'Traslado interno%' ORDER BY id")
            return ("Traslado interno — existen dos modalidades:\n\n" +
                    "\n\n".join(self._formatear(x) for x in filas))

        for claves_any, claves_all, frag in self.MAPA:
            any_ok = any(k in t for k in claves_any) if claves_any else True
            all_ok = all(k in t for k in claves_all) if claves_all else True
            if any_ok and all_ok:
                f = self._frag(frag)
                if f: return self._formatear(f)

        # default: catálogo agrupado (refleja las 3 secciones del TUPA en la BD)
        f = self._bd.consultar("SELECT id, descripcion FROM tramites ORDER BY id")
        academicos = [x for x in f if x["id"] <= 15]
        documentos = [x for x in f if 16 <= x["id"] <= 31]
        titulacion = [x for x in f if x["id"] >= 32]
        return ("Trámites disponibles en la Escuela (catálogo TUPA):\n\n"
                "Trámites académicos:\n" + "\n".join(f"  - {x['descripcion']}" for x in academicos) +
                "\n\nConstancias, certificados y carnés:\n" + "\n".join(f"  - {x['descripcion']}" for x in documentos) +
                "\n\nGrados, títulos y tesis:\n" + "\n".join(f"  - {x['descripcion']}" for x in titulacion) +
                "\n\nPregunta por uno específico para ver requisitos, plazo y costo.")

# =====================================================================
class GestorTesis:
    """Líneas y temas de investigación (ASK puro)."""
    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        if "repositorio" in t or "publicad" in t:
            return ("Repositorio Institucional Digital UNSAAC (acceso abierto):\n"
                    "  https://repositorio.unsaac.edu.pe/handle/20.500.12918/85\n"
                    "Es obligatorio depositar la versión final de la tesis en formato digital.")
        if "semiller" in t or "grupo de investigacion" in t:
            return ("Información de semilleros y grupos de investigación: VRIN UNSAAC\n"
                    "  https://vrin.unsaac.edu.pe/investigacion/\n"
                    "La lista actual de semilleros vinculados a la EPIIS debe verificarse "
                    "directamente con la Dirección de la Escuela.")
        f = self._bd.consultar("SELECT tema FROM temas_tesis ORDER BY tema")
        return ("Áreas/temas de tesis disponibles en la Escuela:\n" +
                "\n".join(f"  - {x['tema']}" for x in f) +
                "\n\nLíneas activas: IA y Machine Learning, Ciberseguridad, Sistemas "
                "Distribuidos y Cloud, Big Data, Desarrollo de Software, IoT.")


# =====================================================================
class GestorTitulacion:
    """Modalidades de titulación, costos TUPA, plazos (ASK puro)."""
    PALABRAS = {
        "tesis": "Sustentación de Tesis",
        "sustenta": "Sustentación de Tesis",
        "suficiencia": "Trabajo de Suficiencia Profesional",
        "servicios profesionales": "Servicios a Nivel Profesional",
        "experiencia": "Servicios a Nivel Profesional",
        "bachiller": "Bachiller (Calificación de Expediente)",
        "dictamen": "Aprobación de Dictamen de Tesis",
        "modificacion": "Modificación del Plan de Tesis",
    }
    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def _formatear(self, f) -> str:
        return (f"**{f['modalidad']}**\n"
                f"{f['descripcion']}\n"
                f"  - Costo TUPA: S/ {f['costo_soles']:.2f}\n"
                f"  - Plazo de atención: {f['plazo_dias_habiles']} días hábiles\n"
                f"  - Requisitos: {f['requisitos']}")

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        if "modalidad" in t and ("titul" in t or "form" in t):
            f = self._bd.consultar("SELECT modalidad, descripcion FROM titulacion_modalidades "
                                   "WHERE id BETWEEN 1 AND 4 ORDER BY id")
            return ("Modalidades de titulación en la UNSAAC:\n" +
                    "\n".join(f"  - {x['modalidad']}: {x['descripcion'][:120]}..." for x in f))
        for clave, modalidad in self.PALABRAS.items():
            if clave in t:
                f = self._bd.consultar_uno(
                    "SELECT * FROM titulacion_modalidades WHERE modalidad = ?", (modalidad,))
                if f: return self._formatear(f)
        if "jurado" in t or "sustenta" in t:
            return ("Sustentación de tesis:\n"
                    "  - Jurado: presidido por el Decano (puede delegar). Hasta 4 docentes: "
                    "2 dictaminadores + 2 replicantes.\n"
                    "  - Exposición: máx. 45 min (tesis individual), 90 min (colectiva).\n"
                    "  - Cada jurado pregunta hasta 10 minutos.\n"
                    "  - Calificación: 14-15 aprobado, 15-17 con distinción, 18-20 con excelencia.\n"
                    "  - Si te desaprueban: nueva fecha tras 30 días; segunda desaprobación "
                    "espera mínimo 6 meses.")
        f = self._bd.consultar("SELECT modalidad, costo_soles, plazo_dias_habiles "
                               "FROM titulacion_modalidades ORDER BY id")
        return ("Trámites de grados y títulos (TUPA 2025):\n" +
                "\n".join(f"  - {x['modalidad']}: S/ {x['costo_soles']:.2f} "
                          f"({x['plazo_dias_habiles']} días hábiles)" for x in f))


# =====================================================================
class ModuloIngresantes:
    """Estadísticas y modalidades de admisión (ASK puro con agregaciones)."""
    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        if "deportista" in t:
            return ("Modalidad Deportistas Calificados:\n"
                    "  Vacantes específicas en el proceso de admisión para deportistas "
                    "calificados. Información oficial:\n"
                    "  https://admision.unsaac.edu.pe/modalidad-deportistas-calificados/")
        if "nota" in t and ("alta" in t or "mayor" in t or "maxima" in t):
            f = self._bd.consultar_uno(
                "SELECT MAX(mayor_nota) AS n FROM ingresantes")
            return f"La nota más alta de ingreso registrada es {f['n']}."
        tot = self._bd.consultar_uno("SELECT SUM(cantidad) AS t, anio FROM ingresantes")
        det = self._bd.consultar(
            "SELECT modalidad, oportunidad, cantidad, mayor_nota FROM ingresantes ORDER BY modalidad")
        return (f"Ingresantes {tot['anio']} — total: {tot['t']} estudiantes.\n" +
                "\n".join(f"  - {x['modalidad']} ({x['oportunidad']}): {x['cantidad']} "
                          f"ingresantes, nota máx. {x['mayor_nota']}" for x in det))


# =====================================================================
class GestorTutorias:
    """ASK puro contra subtemas (tabla tutorias_info)."""
    SUBTEMAS = {
        "proceso": ["proceso","como funciona","como es la tutoria","acompañamiento","obligator"],
        "tipos": ["tipo","individual","grupal","modalidad"],
        "horarios": ["frecuencia","cada cuanto","cuando","horari","reuniones","momentos"],
        "tutor_asignado": ["asignan","mi tutor","quien es mi","cuantos estudiantes","cambio",
                            "cambiar tutor","desde cuando","comite"],
        "problemas": ["falto","no asisto","reprogr","faltas","derivacion","bienestar","derivar"],
    }
    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        for subtema, claves in self.SUBTEMAS.items():
            if any(k in t for k in claves):
                f = self._bd.consultar_uno(
                    "SELECT contenido FROM tutorias_info WHERE subtema=?", (subtema,))
                if f: return f["contenido"]
        f = self._bd.consultar(
            "SELECT subtema, contenido FROM tutorias_info ORDER BY id")
        return "Sistema de tutorías (Reglamento UNSAAC):\n" + "\n\n".join(
            f"[{x['subtema']}] {x['contenido']}" for x in f)


# =====================================================================
class GestorServicios:
    """Servicios universitarios (ASK puro contra servicios_universitarios).
    Ajuste: 'Atención Psicológica' vive en categoria='salud' (no 'bienestar') en la BD,
    así que 'psicolog' ahora dispara salud primero. También se agregó 'deportista'."""
    CATEGORIAS = {
        "salud":     ["salud","medic","centro de salud","odontolog","primeros auxilios","psicolog"],
        "comedor":   ["comedor","comida","almuerzo","alimentacion","menu"],
        "becas":     ["beca","pronabec","beca 18","unisc","cooperacion","permanencia"],
        "deportes":  ["deporte","deportista","futbol","voleibol","natacion","atletismo","deportiv"],
        "vivienda":  ["vivienda","alojamiento"],
        "movilidad": ["movilidad","rpu","intercambio","red peruana"],
        "convenios": ["convenio","cooperacion","octi"],
        "recursos":  ["laboratorio","software","biblioteca virtual","plataforma","correo institucional",
                      "wifi","internet","repositorio digital","sistema academico"],
        "bienestar": ["bienestar","apoyo socioeconomic","defensoria","acoso"],
    }

    def __init__(self, bd: BaseDatos):
        self._bd = bd

    def _fmt(self, f) -> str:
        partes = [f"**{f['nombre']}**", f['descripcion']]
        if f['ubicacion']: partes.append(f"Ubicación: {f['ubicacion']}")
        if f['enlace']:    partes.append(f"Más info: {f['enlace']}")
        return "\n".join(partes)

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        for categoria, claves in self.CATEGORIAS.items():
            if any(k in t for k in claves):
                f = self._bd.consultar(
                    "SELECT * FROM servicios_universitarios WHERE categoria=?", (categoria,))
                if not f: continue
                if len(f) == 1: return self._fmt(f[0])
                return (f"Servicios de {categoria}:\n\n" +
                        "\n\n".join(self._fmt(x) for x in f))
        f = self._bd.consultar(
            "SELECT DISTINCT categoria FROM servicios_universitarios ORDER BY categoria")
        return ("Servicios universitarios disponibles. Pregúntame por una categoría:\n  - " +
                "\n  - ".join(x["categoria"] for x in f))

### 5.1 `GestorElegibilidad` ampliado — el único consumidor del motor

Cuatro conjuntos de reglas:

1. **`REGLAS_PRACTICAS`** (existente) — combina `créditos ≥ 170` y `seguro vigente`.
2. **`REGLAS_BACHILLERATO`** (reformulado según TUPA 2025) — combina 5 hechos
   independientes: egresado, idioma extranjero, computación básica, trabajo de
   investigación sustentado y constancia de prácticas. Ninguno solo basta.
3. **`REGLAS_TITULACION`** (NUEVO) — deriva qué modalidad de titulación le corresponde
   al estudiante combinando bachiller, fecha de ingreso, créditos aprobados y experiencia
   profesional.
4. **`REGLAS_REGULARIDAD`** (NUEVO) — clasifica al estudiante como regular/no regular y
   deriva su tope de créditos según promedio ponderado.

Cada conjunto genera una traza explicativa (`motor.explicar()`) que el bot devuelve al
usuario — eso es el módulo de explicación del sistema experto.

**En simple:** este es el único "empleado" que no se conforma con buscar un dato suelto en la base de datos, sino que razona combinando varios datos a la vez para sacar una conclusión. Por ejemplo, para saber si alguien puede tramitar el bachillerato no basta un solo dato: hay que revisar 5 condiciones distintas al mismo tiempo (ser egresado, tener idioma, tener computación básica, etc.). Por eso es el único módulo que usa el Motor de Inferencia explicado arriba.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 9: CLASE GestorElegibilidad (ÚNICO CONSUMIDOR DEL MOTOR)
# ─────────────────────────────────────────────────────────────────────
# Este es el ÚNICO módulo que usa el MotorInferencia. Combina múltiples
# hechos independientes (que nunca están juntos en una sola fila de la BD)
# para derivar conclusiones nuevas. Opera 4 líneas de razonamiento:
#
#   1. REGLAS_PRACTICAS (3 reglas):
#      Combina créditos ≥ 170 + seguro vigente → puede iniciar prácticas
#
#   2. REGLAS_BACHILLERATO (5 reglas, según TUPA 2025):
#      Combina 5 hechos: egresado + idioma extranjero + computación básica
#      + trabajo de investigación sustentado + constancia de prácticas
#      → puede tramitar bachillerato
#
#   3. REGLAS_TITULACION (6 reglas):
#      Combina bachiller + año de ingreso + créditos + experiencia
#      → deriva qué modalidad(es) de titulación le corresponden
#
#   4. REGLAS_REGULARIDAD (5 reglas):
#      Combina créditos del semestre + promedio ponderado
#      → clasifica como regular/no regular y deriva tope de créditos
#
# Cada evaluador (_evaluar_X) extrae hechos del texto del usuario
# mediante patrones léxicos, los inyecta al motor con decir(), ejecuta
# inferir() y devuelve el veredicto junto con la traza explicativa.
# ═══════════════════════════════════════════════════════════════════════
class GestorElegibilidad:
    """Único módulo que usa el MotorInferencia. Combina hechos independientes
    para derivar conclusiones que no existen en ninguna fila de la BD."""

    # ---------- 1. Prácticas (ya existía) ----------
    REGLAS_PRACTICAS = [
        ("R1", ["creditos_suficientes"], "cumple_creditos_practicas"),
        ("R2", ["tiene_seguro"], "cumple_seguro_practicas"),
        ("R3", ["cumple_creditos_practicas", "cumple_seguro_practicas"],
         "puede_iniciar_practicas"),
    ]

    # ---------- 2. Bachillerato (REFORMULADO según TUPA) ----------
    REGLAS_BACHILLERATO = [
        ("R1", ["plan_completo"], "es_egresado"),
        ("R2", ["tiene_idioma_extranjero"], "cumple_idioma"),
        ("R3", ["tiene_computacion_basica"], "cumple_computacion"),
        ("R4", ["tiene_trabajo_investigacion_sustentado"], "cumple_investigacion"),
        ("R5", ["es_egresado", "cumple_idioma", "cumple_computacion",
                "cumple_investigacion", "tiene_constancia_practicas"],
               "puede_tramitar_bachillerato"),
    ]

    # ---------- 3. Modalidad de titulación (NUEVO) ----------
    REGLAS_TITULACION = [
        ("R6", ["creditos_pct_70"], "puede_inscribir_trabajo_investigacion"),
        ("R7", ["creditos_pct_80"], "puede_inscribir_plan_tesis"),
        ("R8", ["puede_inscribir_plan_tesis", "tiene_director_tesis"],
               "plan_tesis_admisible"),
        ("R9", ["tiene_bachiller"], "puede_trabajo_suficiencia"),
        ("R10",["tiene_bachiller", "plan_tesis_admisible"],
               "puede_titularse_por_tesis"),
        ("R11",["ingreso_antes_2014", "tiene_bachiller", "exp_3_anios"],
               "puede_servicios_profesionales"),
    ]

    # ---------- 4. Regularidad académica (NUEVO) ----------
    REGLAS_REGULARIDAD = [
        ("R12",["creditos_sem_ge_12"], "estudiante_regular"),
        ("R13",["creditos_sem_lt_12"], "estudiante_no_regular"),
        ("R14",["estudiante_regular", "promedio_ge_16"], "tope_creditos_26"),
        ("R15",["estudiante_regular", "promedio_lt_16"], "tope_creditos_22"),
        ("R16",["estudiante_no_regular"], "tope_creditos_menor_12"),
    ]

    # ============ Extracción de hechos del texto del usuario ============

    @staticmethod
    def _creditos_mencionados(t: str) -> Optional[int]:
        m = re.search(r"(\d{2,3})\s*cr[eé]dito", t)
        return int(m.group(1)) if m else None

    @staticmethod
    def _porcentaje_mencionado(t: str) -> Optional[int]:
        m = re.search(r"(\d{1,3})\s*%", t)
        if m: return int(m.group(1))
        m = re.search(r"(\d{1,3})\s*por\s*ciento", t)
        return int(m.group(1)) if m else None

    @staticmethod
    def _experiencia_anios(t: str) -> Optional[int]:
        m = re.search(r"(\d+)\s*a[nñ]os?\s+(de\s+)?(experiencia|trabajando|trabajo)", t)
        return int(m.group(1)) if m else None

    @staticmethod
    def _anio_ingreso(t: str) -> Optional[int]:
        m = re.search(r"ingres[eé]?\s+(?:en|el)?\s*(?:el\s+)?(\d{4})", t)
        return int(m.group(1)) if m else None

    @staticmethod
    def _promedio_mencionado(t: str) -> Optional[float]:
        m = re.search(r"promedio\D*(\d{1,2}(?:[.,]\d+)?)", t)
        return float(m.group(1).replace(",", ".")) if m else None

    # ============ Evaluadores por tipo de consulta ============

    def _evaluar_practicas(self, t: str) -> str:
        motor = MotorInferencia(self.REGLAS_PRACTICAS)
        recibido = False
        cr = self._creditos_mencionados(t)
        if cr is not None:
            motor.decir("creditos_suficientes", cr >= 170); recibido = True
        elif any(k in t for k in ["viii semestre","octavo semestre","8vo semestre",
                                   "ix semestre","9no semestre","x semestre"]):
            motor.decir("creditos_suficientes", True); recibido = True
        if "sin seguro" in t or "no tengo seguro" in t:
            motor.decir("tiene_seguro", False); recibido = True
        elif "seguro" in t or "soat" in t:
            motor.decir("tiene_seguro", True); recibido = True

        motor.inferir()
        if motor.preguntar("puede_iniciar_practicas"):
            v = "Sí: con los datos que me das, SÍ cumples los requisitos para iniciar prácticas."
        elif recibido:
            v = ("Con los datos que me das aún no cumples todos los requisitos "
                 "(170 créditos o VIII semestre aprobado, y seguro vigente).")
        else:
            v = ("Para evaluar tu elegibilidad dime cuántos créditos tienes aprobados y si "
                 "tienes seguro vigente; por ejemplo: 'tengo 175 créditos y seguro vigente, "
                 "¿puedo iniciar prácticas?'")
        return v + "\n\nRazonamiento del motor de inferencia:\n" + motor.explicar()

    def _evaluar_bachillerato(self, t: str) -> str:
        motor = MotorInferencia(self.REGLAS_BACHILLERATO)
        if "plan" in t and ("complet" in t or "termin" in t):
            motor.decir("plan_completo", True)
        if "egresad" in t:
            motor.decir("plan_completo", True)
        if "idioma" in t or "ingles" in t or "extranjero" in t:
            motor.decir("tiene_idioma_extranjero", True)
        if "computacion" in t or "ofimatica" in t:
            motor.decir("tiene_computacion_basica", True)
        if "trabajo de investigacion" in t or "investigacion sustentad" in t:
            motor.decir("tiene_trabajo_investigacion_sustentado", True)
        if "practica" in t or "constancia de practica" in t or "pp" in t.split():
            motor.decir("tiene_constancia_practicas", True)
        if "todos los certificados" in t or "todos los requisitos" in t:
            motor.decir("tiene_idioma_extranjero", True)
            motor.decir("tiene_computacion_basica", True)
            motor.decir("tiene_constancia_practicas", True)

        motor.inferir()
        if motor.preguntar("puede_tramitar_bachillerato"):
            v = "Sí: cumples los cinco requisitos del TUPA para tramitar el bachillerato."
        else:
            v = ("Para el bachillerato necesitas cinco hechos independientes: plan de estudios "
                 "completo (egresado), certificado de idioma extranjero, certificado de "
                 "computación básica, trabajo de investigación sustentado y constancia de "
                 "prácticas. Indícame cuáles ya tienes.")
        return v + "\n\nRazonamiento del motor de inferencia:\n" + motor.explicar()

    def _evaluar_titulacion(self, t: str) -> str:
        motor = MotorInferencia(self.REGLAS_TITULACION)
        if "bachiller" in t and any(k in t for k in ["tengo","soy","ya soy","ya tengo"]):
            motor.decir("tiene_bachiller", True)
        pct = self._porcentaje_mencionado(t)
        if pct is not None:
            motor.decir("creditos_pct_70", pct >= 70)
            motor.decir("creditos_pct_80", pct >= 80)
        anio = self._anio_ingreso(t)
        if anio is not None and anio < 2014:
            motor.decir("ingreso_antes_2014", True)
        if "antes de 2014" in t or "antes del 2014" in t or "antes de junio de 2014" in t:
            motor.decir("ingreso_antes_2014", True)
        exp = self._experiencia_anios(t)
        if exp is not None and exp >= 3:
            motor.decir("exp_3_anios", True)
        if "director" in t and "tesis" in t:
            motor.decir("tiene_director_tesis", True)
        if "plan de tesis avalado" in t or "plan de tesis aprobado" in t:
            motor.decir("puede_inscribir_plan_tesis", True)
            motor.decir("tiene_director_tesis", True)

        motor.inferir()
        habilitadas = []
        if motor.preguntar("puede_titularse_por_tesis"):
            habilitadas.append("Sustentación de Tesis")
        if motor.preguntar("puede_trabajo_suficiencia"):
            habilitadas.append("Trabajo de Suficiencia Profesional")
        if motor.preguntar("puede_servicios_profesionales"):
            habilitadas.append("Servicios a Nivel Profesional (por experiencia)")
        if habilitadas:
            v = "Modalidades de titulación habilitadas para ti:\n  - " + "\n  - ".join(habilitadas)
        else:
            v = ("Para evaluar tu modalidad de titulación dime si ya tienes bachiller, en qué "
                 "año ingresaste, qué porcentaje de créditos aprobaste y, si aplica, cuántos "
                 "años de experiencia profesional tienes.")
        return v + "\n\nRazonamiento del motor de inferencia:\n" + motor.explicar()

    def _evaluar_regularidad(self, t: str) -> str:
        motor = MotorInferencia(self.REGLAS_REGULARIDAD)
        cr = self._creditos_mencionados(t)
        # interpretación: "matriculo X créditos" = créditos del semestre
        if cr is not None:
            motor.decir("creditos_sem_ge_12", cr >= 12)
            motor.decir("creditos_sem_lt_12", cr < 12)
        prom = self._promedio_mencionado(t)
        if prom is not None:
            motor.decir("promedio_ge_16", prom >= 16)
            motor.decir("promedio_lt_16", prom < 16)

        motor.inferir()
        partes = []
        if motor.preguntar("estudiante_regular"):
            partes.append("Eres estudiante REGULAR (≥ 12 créditos por semestre).")
        if motor.preguntar("estudiante_no_regular"):
            partes.append("Eres estudiante NO REGULAR (< 12 créditos). Recuerda: no se "
                          "permiten más de 3 semestres consecutivos en esta condición.")
        if motor.preguntar("tope_creditos_26"):
            partes.append("Tu tope de créditos del semestre es 26 (promedio ponderado ≥ 16).")
        elif motor.preguntar("tope_creditos_22"):
            partes.append("Tu tope de créditos del semestre es 22.")
        elif motor.preguntar("tope_creditos_menor_12"):
            partes.append("Como no regular, llevas menos de 12 créditos por semestre.")

        if not partes:
            v = ("Para determinar tu regularidad dime cuántos créditos estás matriculando "
                 "este semestre y, si corresponde, tu promedio ponderado.")
        else:
            v = "\n".join(partes)
        return v + "\n\nRazonamiento del motor de inferencia:\n" + motor.explicar()

    # ============ Dispatcher ============

    def responder(self, texto: str) -> str:
        t = _sin_tildes(texto)
        # Orden: titulación primero (más específico), luego bachiller, luego regularidad,
        # luego prácticas (default)
        if any(k in t for k in ["titul","tesis","suficiencia","servicios profesional",
                                  "modalidad de titul","experiencia profesional"]):
            return self._evaluar_titulacion(t)
        if "bachiller" in t:
            return self._evaluar_bachillerato(t)
        if any(k in t for k in ["regular","tope","matricul","carga","promedio"]):
            return self._evaluar_regularidad(t)
        return self._evaluar_practicas(t)


print("GestorElegibilidad ampliado definido.")


GestorElegibilidad ampliado definido.


## 6. Clase `EscuelaBot` — orquestador

**Descripción técnica:** `EscuelaBot` es la **fachada (Facade pattern)** del sistema: coordina los demás componentes (BD, clasificador y módulos de dominio) y expone una única interfaz pública, `responder`.

En el constructor (`__init__`) realiza tres tareas, en orden:

1. **Construye la base de datos** y crea los 12 módulos de dominio, pasándole a cada uno la misma instancia de `BaseDatos` (inyección de dependencias), de modo que ningún módulo necesita conocer directamente el motor de base de datos.
2. **Entrena el clasificador** (`ModeloClasificador`, TF-IDF + Random Forest) con el corpus completo de preguntas etiquetadas.
3. **Construye la tabla de despacho:** un diccionario que asocia cada intención con el método del módulo que debe responderla, evitando una larga cadena de condicionales y permitiendo agregar nuevas intenciones solo añadiendo una entrada al diccionario.

El método `responder(pregunta)` primero clasifica la pregunta para obtener la intención y su nivel de confianza. Si la intención es un saludo o despedida, devuelve una respuesta fija. Si la confianza es muy baja, pide que se reformule la pregunta en vez de arriesgarse a responder mal. En cualquier otro caso, busca en la tabla de despacho el módulo correspondiente y le delega la respuesta.

Adicionalmente, `iniciar_chat()` ofrece un modo de conversación continua por consola, leyendo preguntas y mostrando respuestas hasta que el usuario decide terminar.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 10: CLASE EscuelaBot (ORQUESTADOR PRINCIPAL)
# ─────────────────────────────────────────────────────────────────────
# Clase principal que integra todos los componentes del chatbot:
#   - Construye la BD en memoria y crea la capa BaseDatos
#   - Instancia los 11 módulos de conocimiento (POO)
#   - Entrena el ModeloClasificador con el corpus
#   - Define la TABLA DE DESPACHO: diccionario {intención → método}
#     que conecta cada intención clasificada con el módulo responsable
#
# Flujo del pipeline responder(pregunta):
#   Pregunta → clasificador.predecir() → (intención, confianza)
#     → si es "saludo"/"despedida" → respuesta fija
#     → si confianza < umbral     → pide reformular
#     → si es válida              → despacho[intención](pregunta)
#       → el módulo consulta la BD o el motor → respuesta al usuario
#
# También define:
#   - _responder_contacto()  → datos de la directora, teléfono, web
#   - _responder_derivar()   → honestidad académica: "este dato no se
#                               puede confirmar, consulta la Escuela"
#   - iniciar_chat()         → bucle interactivo (input/print)
# ═══════════════════════════════════════════════════════════════════════
class EscuelaBot:
    """Chatbot de la EPIIS — pipeline completo."""

    def __init__(self, corpus: dict):
        self._conn = construir_base_datos()
        self._bd = BaseDatos(self._conn)

        # Módulos POO
        self.info = InformacionGeneral(self._bd)
        self.docentes = DirectorioDocentes(self._bd)
        self.cursos = RepositorioCursos(self._bd)
        self.tramites = GestorTramites(self._bd)
        self.tesis = GestorTesis(self._bd)
        self.titulacion = GestorTitulacion(self._bd)
        self.ingresantes = ModuloIngresantes(self._bd)
        self.tutorias = GestorTutorias(self._bd)
        self.servicios = GestorServicios(self._bd)
        self.elegibilidad = GestorElegibilidad()
        self.practicas = self.elegibilidad
        self.elegibilidad = GestorElegibilidad()
        self.horarios = GestorHorarios(self._bd)

        # Clasificador
        self.clasificador = ModeloClasificador(n_estimators=300)
        self.clasificador.entrenar(corpus)

        # Tabla de despacho
        self._despacho = {
            "informacion_general": self.info.responder,
            "docentes": self.docentes.responder,
            "cursos": self.cursos.responder,
            "tramites": self.tramites.responder,
            "tutorias": self.tutorias.responder,
            "practicas": self.practicas.responder,
            "temas_tesis": self.tesis.responder,
            "titulacion": self.titulacion.responder,
            "ingresantes": self.ingresantes.responder,
            "servicios": self.servicios.responder,
            "contacto": self._responder_contacto,
            "elegibilidad": self.elegibilidad.responder,
            "derivar_a_escuela": self._responder_derivar,
            "horarios": self.horarios.responder,
        }

    def _responder_contacto(self, texto: str) -> str:
        tel = self._bd.consultar_uno(
            "SELECT contenido FROM informacion_general WHERE titulo='Teléfono'")
        directora = self._bd.consultar_uno(
            "SELECT contenido FROM informacion_general WHERE titulo='Directora'")
        depto = self._bd.consultar_uno(
            "SELECT contenido FROM informacion_general WHERE titulo='Departamento Académico'")
        return ("Datos de contacto de la EPIIS:\n"
                f"  - Directora: {directora['contenido'] if directora else 'N/D'}\n"
                f"  - Departamento Académico: {depto['contenido'] if depto else 'N/D'}\n"
                f"  - Teléfono: {tel['contenido'] if tel else 'N/D'}\n"
                "  - Sitio: in.unsaac.edu.pe — Departamento: daii.unsaac.edu.pe\n"
                "  - Atención: lunes a viernes, 8:00–17:00.")

    @staticmethod
    def _responder_derivar(texto: str) -> str:
        return ("Este dato no se encuentra confirmado en las fuentes públicas (el "
                "documento institucional lo marca como información que debe validarse "
                "con la Escuela). Te recomiendo consultar directamente:\n"
                "  - Dirección de Escuela (EPIIS) — daii.unsaac.edu.pe\n"
                "  - Teléfono: 084-232398, anexo 1402\n"
                "  - Centro de Cómputo (catálogo de cursos y horarios): "
                "ccomputo.unsaac.edu.pe\n"
                "Evito darte un dato inventado.")

    @staticmethod
    def _saludo() -> str:
        return ("¡Hola! Soy el asistente virtual de la Escuela Profesional de Ingeniería "
                "Informática y de Sistemas (UNSAAC). Puedo ayudarte con información "
                "general, docentes, cursos, horarios, trámites, tutorías, prácticas, tesis, "
                "titulación, ingresantes, servicios universitarios o evaluar tu "
                "elegibilidad (prácticas, bachiller, modalidad de titulación, "
                "regularidad). ¿En qué te ayudo?")

    @staticmethod
    def _despedida() -> str:
        return "¡Gracias por usar el asistente de la EPIIS! Que tengas un buen día. 👋"

    def responder(self, pregunta: str) -> str:
        intencion, confianza = self.clasificador.predecir(pregunta)
        if intencion == "saludo": return self._saludo()
        if intencion == "despedida": return self._despedida()
        if confianza < self.clasificador.umbral:
            return ("No estoy seguro de haber entendido tu consulta. ¿Podrías "
                    "reformularla? Puedo ayudarte con información general, docentes, "
                    "cursos, horarios, trámites, tutorías, prácticas, tesis, titulación, "
                    "ingresantes, servicios o evaluar tu elegibilidad.")
        handler = self._despacho.get(intencion)
        return handler(pregunta) if handler else self._saludo()

    def iniciar_chat(self) -> None:
        print(self._saludo())
        print("(escribe 'salir' para terminar)\n")
        while True:
            try: pregunta = input("Tú: ").strip()
            except (EOFError, KeyboardInterrupt):
                print("\n" + self._despedida()); break
            if not pregunta: continue
            if pregunta.lower() in {"salir","exit","quit"}:
                print("Bot:", self._despedida()); break
            print("Bot:", self.responder(pregunta), "\n")

print("Clase EscuelaBot definida.")

Clase EscuelaBot definida.


## 7. Instanciación y entrenamiento

Crea la instancia `bot = EscuelaBot(CORPUS)`, lo cual dispara en cadena todo el proceso de inicialización definido en el constructor de `EscuelaBot`: se construye la base de datos en SQLite, se crean los 12 módulos de dominio, y se entrena el clasificador (TF-IDF + Random Forest) con el corpus completo de preguntas.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 11: INSTANCIACIÓN Y ENTRENAMIENTO
# ─────────────────────────────────────────────────────────────────────
# Crea la instancia del chatbot: esto ejecuta en cadena la construcción
# de la BD, la instanciación de los 11 módulos y el entrenamiento del
# clasificador Random Forest con todo el corpus. Una vez terminado,
# el objeto 'bot' está listo para recibir preguntas.
# ═══════════════════════════════════════════════════════════════════════
bot = EscuelaBot(CORPUS)
print("\nChatbot listo.")


Repositorio: 10 tablas creadas, 31 sentencias INSERT.
Exactitud en validación: 0.832
Modelo entrenado con 1931 ejemplos, 15 intenciones.

Chatbot listo.


## 8. Casos de prueba

Cubrimos las 15 intenciones, y para `elegibilidad` ejercitamos las cuatro líneas de
reglas: prácticas, bachillerato, titulación y regularidad académica.

**Descripción técnica:** esta celda ejecuta una batería de llamadas a `bot.responder(pregunta)` para validar, de forma manual, que el pipeline completo funciona correctamente de extremo a extremo: clasificación de intención, enrutamiento por la tabla de despacho, y generación de la respuesta final (vía consulta SQL o vía motor de inferencia, según el módulo).

Se cubren las 15 intenciones del corpus, asegurando que cada una active al módulo correcto. Para la intención `elegibilidad` se incluyen casos puntuales de cada una de las cuatro líneas de razonamiento del `GestorElegibilidad` (prácticas, bachillerato, titulación y regularidad académica), de modo que se verifique no solo que el modelo clasifica bien la intención, sino que el motor de inferencia deriva la conclusión correcta a partir de los hechos extraídos de cada pregunta.

No se trata de pruebas automatizadas con aserciones (no hay `assert` ni framework de testing): es una verificación funcional por inspección visual de las respuestas impresas.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 12:DE CASOS DE PRUEBA
# ─────────────────────────────────────────────────────────────────────
# Evalúa el sistema con  preguntas representativas, una por cada
# intención + las 4 líneas de inferencia del GestorElegibilidad.
# Para cada pregunta:
#   1. Clasifica con el modelo → obtiene intención y confianza
#   2. Compara con la intención esperada (✓ o ✗)
#   3. Ejecuta bot.responder() para mostrar la respuesta completa
# ═══════════════════════════════════════════════════════════════════════
casos = [
    # — intenciones simples (ASK puro) —
    ("Hola, buenos días", "saludo"),
    ("¿Cuál es la misión de la escuela?", "informacion_general"),
    ("¿Quién es la directora de la escuela?", "informacion_general"),
    ("¿Cuál es el perfil del egresado?", "informacion_general"),
    ("¿La escuela está acreditada?", "informacion_general"),
    ("¿Qué docentes tienen dedicación exclusiva?", "docentes"),
    ("¿Qué cursos hay en el semestre 7?", "cursos"),
    ("¿Cuántos créditos hay que llevar por semestre?", "cursos"),
    ("¿Cómo se hace el retiro de un curso?", "tramites"),
    ("¿Cómo solicito una constancia de estudios?", "tramites"),
    ("¿Cómo funciona el cambio de tutor?", "tutorias"),
    ("¿Cada cuánto se hacen las tutorías?", "tutorias"),
    ("¿Qué requisitos necesito para las prácticas preprofesionales?", "practicas"),
    ("¿Cuáles son los temas de tesis disponibles?", "temas_tesis"),
    ("¿Cuáles son las modalidades de titulación?", "titulacion"),
    ("¿Cuánto cuesta titularse por tesis?", "titulacion"),
    ("¿Cuántos ingresantes hubo este año?", "ingresantes"),
    ("¿Hay centro universitario de salud?", "servicios"),
    ("Becas de posgrado en Brasil", "servicios"),
    ("¿La UNSAAC participa en RPU?", "servicios"),
    ("Datos de contacto de la escuela", "contacto"),
    ("¿Cuál es el horario de atención del profesor Carrasco?", "derivar_a_escuela"),
    ("Muchas gracias, eso es todo", "despedida"),
    # — elegibilidad: cuatro líneas de reglas —
    ("tengo 175 créditos y seguro vigente, ¿puedo iniciar prácticas?", "elegibilidad"),
    ("tengo plan de estudios completo, idioma extranjero, computación, "
     "trabajo de investigación sustentado y constancia de prácticas, "
     "¿puedo tramitar el bachiller?", "elegibilidad"),
    ("ingresé en 2013, tengo bachiller y 4 años de experiencia, "
     "¿qué modalidad de titulación me corresponde?", "elegibilidad"),
    ("matriculé 18 créditos y mi promedio es 17, "
     "¿hasta cuántos créditos puedo llevar?", "elegibilidad"),
]

ok = 0
for pregunta, esperado in casos:
    intencion, confianza = bot.clasificador.predecir(pregunta)
    estado = "✓" if intencion == esperado else "✗"
    if intencion == esperado: ok += 1
    print("=" * 78)
    print(f"{estado} Usuario   : {pregunta}")
    print(f"  Esperado  : {esperado}")
    print(f"  Intención : {intencion}  (confianza {confianza:.2f})")
    print(f"  Bot       :")
    print(textwrap.indent(bot.responder(pregunta), "              "))
print("=" * 78)
print(f"\nRESUMEN: {ok}/{len(casos)} casos clasificados correctamente.")



casos_tramites = [
    ("quiero hacer reserva de mi matrícula", "Reserva de matrícula"),
    ("soy ingresante y quiero saber del proceso de matrícula", "matrícula de ingresante"),
    ("tengo bajo rendimiento, ¿cómo hago mi matrícula?", "condicionada por bajo rendimiento"),
    ("perdí mi carné universitario, necesito un duplicado de matrícula", "Duplicado de carné universitario"),
    ("quiero el duplicado de mi carné de biblioteca y también matricularme", "Duplicado de carné de biblioteca"),
    ("soy egresado y quiero matricularme en un curso dirigido", "cursos dirigidos"),
    ("necesito convalidar un curso, ¿eso afecta mi matrícula?", "Convalidación de asignaturas"),
    ("voy a tramitar mi bachiller, ¿necesito estar matriculado?", "Bachiller"),
    ("quiero hacer mi reinicio de estudios y matricularme de nuevo", "Reinicio de estudios"),
    ("necesito la ficha de seguimiento para mi matrícula condicionada", "condicionada por bajo rendimiento"),
    ("quiero saber sobre traslado externo y la matrícula correspondiente", "Traslado externo"),
    ("tengo que rectificar mi nombre antes de matricularme", "Rectificación de nombre"),
    ("necesito mi certificado de estudios y matrícula del semestre", "Certificado de estudios"),
    ("quiero matricularme en cursos dirigidos antes de egresar", "cursos dirigidos"),
    ("¿cómo me matriculo este semestre?", "alumnos regulares"),
    ("perdí mi constancia de matrícula, ¿cómo saco otra?", "constancia de matrícula"),
    ("quiero hacer un traslado interno a otra Facultad", "Traslado interno"),
]

ok_intencion = 0
ok_contenido = 0

for pregunta, fragmento_esperado in casos_tramites:
    # 1) Clasificador (Random Forest): ¿identifica bien que es intent "tramites"?
    intencion, confianza = bot.clasificador.predecir(pregunta)
    intencion_ok = (intencion == "tramites")
    if intencion_ok: ok_intencion += 1

    # 2) Contenido: ¿GestorTramites elige el trámite específico correcto?
    respuesta = bot.tramites.responder(pregunta)
    contenido_ok = fragmento_esperado.lower() in respuesta.lower()
    if contenido_ok: ok_contenido += 1

    estado_int = "✓" if intencion_ok else "✗"
    estado_cont = "✓" if contenido_ok else "✗"
    primera_linea = respuesta.split("\n")[0]

    print("=" * 78)
    print(f"Pregunta     : {pregunta}")
    print(f"{estado_int} Intención  : {intencion}  (esperado 'tramites', confianza {confianza:.2f})")
    print(f"{estado_cont} Contenido  : esperado~'{fragmento_esperado}' | obtenido: {primera_linea[:60]}")

print("=" * 78)
print(f"\nRESUMEN INTENCIÓN : {ok_intencion}/{len(casos_tramites)} clasificados como 'tramites'")
print(f"RESUMEN CONTENIDO : {ok_contenido}/{len(casos_tramites)} con el trámite correcto")

✓ Usuario   : Hola, buenos días
  Esperado  : saludo
  Intención : saludo  (confianza 0.97)
  Bot       :
              ¡Hola! Soy el asistente virtual de la Escuela Profesional de Ingeniería Informática y de Sistemas (UNSAAC). Puedo ayudarte con información general, docentes, cursos, horarios, trámites, tutorías, prácticas, tesis, titulación, ingresantes, servicios universitarios o evaluar tu elegibilidad (prácticas, bachiller, modalidad de titulación, regularidad). ¿En qué te ayudo?
✓ Usuario   : ¿Cuál es la misión de la escuela?
  Esperado  : informacion_general
  Intención : informacion_general  (confianza 0.99)
  Bot       :
              Misión:
              La Escuela Profesional de Ingeniería Informática y de Sistemas realiza y fomenta la investigación mediante cursos, conferencias y coordinación permanente con docentes y estudiantes, promoviendo el desarrollo y conclusión de tesis y el aporte tecnológico a la región.
✓ Usuario   : ¿Quién es la directora de la escuela?
  Esper

## 9. Chat interactivo



In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BLOQUE 13: CHAT INTERACTIVO
# ─────────────────────────────────────────────────────────────────────
# Inicia el bucle de conversación con el usuario. El estudiante escribe
# una pregunta, el bot la clasifica, consulta el módulo correspondiente
# y devuelve la respuesta. Escribir 'salir' termina la sesión.
# ═══════════════════════════════════════════════════════════════════════
bot.iniciar_chat()

¡Hola! Soy el asistente virtual de la Escuela Profesional de Ingeniería Informática y de Sistemas (UNSAAC). Puedo ayudarte con información general, docentes, cursos, horarios, trámites, tutorías, prácticas, tesis, titulación, ingresantes, servicios universitarios o evaluar tu elegibilidad (prácticas, bachiller, modalidad de titulación, regularidad). ¿En qué te ayudo?
(escribe 'salir' para terminar)

Tú: holaaaaaaaaaa
Bot: ¡Hola! Soy el asistente virtual de la Escuela Profesional de Ingeniería Informática y de Sistemas (UNSAAC). Puedo ayudarte con información general, docentes, cursos, horarios, trámites, tutorías, prácticas, tesis, titulación, ingresantes, servicios universitarios o evaluar tu elegibilidad (prácticas, bachiller, modalidad de titulación, regularidad). ¿En qué te ayudo? 

Tú: Cuál es la misión de la Escuela
Bot: Misión:
La Escuela Profesional de Ingeniería Informática y de Sistemas realiza y fomenta la investigación mediante cursos, conferencias y coordinación permanente